# 11. 2026–27 Premier League Forecasts

## 1. Load and Validate the Frozen Production Package

The production model developed and validated in Notebook 10 is now treated as fixed.

This notebook does not perform further model selection, hyperparameter tuning, calibration or refitting. Its purpose is to apply the exported Independent Poisson forecasting package to genuinely unseen 2026–27 Premier League fixtures.

The production package contains:

- the fitted median imputer;
- the fitted `StandardScaler`;
- the final home-goal Poisson estimator;
- the final away-goal Poisson estimator;
- the frozen 70-feature schema;
- the 55-feature full-rank numerical basis required by the unregularised home model;
- the independent Poisson scoreline specification;
- the fixed probability order `(H, D, A)`.

Before any 2026–27 forecasts are generated, the serialized package and its metadata are reloaded directly from disk and checked for structural consistency.

All subsequent predictions in this notebook must use these frozen artefacts without modification.

In [1]:
# ============================================================
# 1. Load and Validate the Frozen Production Package
# ============================================================

from __future__ import annotations

import hashlib
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# Locate project
# ------------------------------------------------------------

project_root = Path.cwd().resolve()

while (
    project_root.name != "premier-league-probability-engine"
    and project_root.parent != project_root
):
    project_root = project_root.parent

assert project_root.name == "premier-league-probability-engine", (
    "Could not locate the premier-league-probability-engine "
    "repository root."
)


# ------------------------------------------------------------
# Production artefact paths
# ------------------------------------------------------------

production_directory = (
    project_root
    / "outputs"
    / "production_model"
)

bundle_path = (
    production_directory
    / "production_model_bundle.joblib"
)

metadata_path = (
    production_directory
    / "production_model_metadata.json"
)

manifest_path = (
    production_directory
    / "validated_implementation_manifest.json"
)

feature_schema_path = (
    production_directory
    / "validated_feature_schema.csv"
)

artifact_hash_path = (
    production_directory
    / "production_artifact_hashes.json"
)

reference_input_path = (
    production_directory
    / "production_reference_inputs.csv"
)

reference_prediction_path = (
    production_directory
    / "production_reference_predictions.csv"
)


required_paths = [
    bundle_path,
    metadata_path,
    manifest_path,
    feature_schema_path,
    artifact_hash_path,
    reference_input_path,
    reference_prediction_path,
]

missing_paths = [
    path
    for path in required_paths
    if not path.is_file()
]

assert not missing_paths, (
    "The frozen production package is incomplete: "
    f"{missing_paths}"
)


# ------------------------------------------------------------
# SHA-256 helper
# ------------------------------------------------------------

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


# ------------------------------------------------------------
# Verify recorded artefact hashes
# ------------------------------------------------------------

with artifact_hash_path.open(
    "r",
    encoding="utf-8",
) as file:
    recorded_hashes = json.load(file)


hash_validation_records = []

for relative_path, recorded_hash in (
    recorded_hashes.items()
):

    artifact_path = (
        project_root
        / relative_path
    )

    exists = artifact_path.is_file()

    observed_hash = (
        sha256_file(
            artifact_path
        )
        if exists
        else None
    )

    hash_validation_records.append(
        {
            "Artifact": relative_path,
            "Exists": exists,
            "HashMatch": (
                exists
                and observed_hash
                == recorded_hash
            ),
        }
    )


hash_validation_table = pd.DataFrame(
    hash_validation_records
)

assert hash_validation_table[
    "Exists"
].all(), (
    "At least one hashed production artefact is missing."
)

assert hash_validation_table[
    "HashMatch"
].all(), (
    "At least one production artefact has changed since "
    "Notebook 10 exported the package."
)


# ------------------------------------------------------------
# Reload frozen package
# ------------------------------------------------------------

production_bundle = joblib.load(
    bundle_path
)

with metadata_path.open(
    "r",
    encoding="utf-8",
) as file:
    production_metadata = json.load(file)

with manifest_path.open(
    "r",
    encoding="utf-8",
) as file:
    production_manifest = json.load(file)

production_feature_schema = pd.read_csv(
    feature_schema_path
)


# ------------------------------------------------------------
# Recover production objects
# ------------------------------------------------------------

production_feature_columns = list(
    production_bundle[
        "ProductionFeatureColumns"
    ]
)

production_home_feature_columns = list(
    production_bundle[
        "HomeFeatureColumns"
    ]
)

production_home_feature_positions = np.asarray(
    production_bundle[
        "HomeFeaturePositions"
    ],
    dtype=int,
)

production_imputer = (
    production_bundle[
        "Preprocessing"
    ]["Imputer"]
)

production_scaler = (
    production_bundle[
        "Preprocessing"
    ]["Scaler"]
)

production_home_model = (
    production_bundle[
        "Models"
    ]["HomePoisson"]
)

production_away_model = (
    production_bundle[
        "Models"
    ]["AwayPoisson"]
)

MAX_MODELLED_GOALS = int(
    production_bundle[
        "Scoreline"
    ]["MaximumModelledGoals"]
)

MINIMUM_EXPECTED_GOALS = float(
    production_bundle[
        "Scoreline"
    ]["MinimumExpectedGoals"]
)

PROBABILITY_ORDER = list(
    production_bundle[
        "ProbabilityOrder"
    ]
)

PRODUCTION_OUTPUT_COLUMNS = list(
    production_bundle[
        "RequiredPredictionOutputs"
    ]
)


# ------------------------------------------------------------
# Structural validation
# ------------------------------------------------------------

assert (
    production_bundle[
        "ModelFamily"
    ]
    == "Independent Poisson regression"
)

assert (
    production_bundle[
        "CalibrationMethod"
    ]
    == "Original probabilities"
)

assert PROBABILITY_ORDER == [
    "H",
    "D",
    "A",
]

assert PRODUCTION_OUTPUT_COLUMNS == [
    "HomeExpectedGoals",
    "AwayExpectedGoals",
    "Probability_H",
    "Probability_D",
    "Probability_A",
]

assert len(
    production_feature_columns
) == 70

assert len(
    set(
        production_feature_columns
    )
) == 70

assert len(
    production_home_feature_columns
) == 55

assert len(
    production_home_feature_positions
) == 55

assert (
    production_home_model.n_features_in_
    == 55
)

assert (
    production_away_model.n_features_in_
    == 70
)

assert MAX_MODELLED_GOALS == 10

assert (
    MINIMUM_EXPECTED_GOALS
    == 1e-6
)


# ------------------------------------------------------------
# Feature-schema CSV must agree with bundle
# ------------------------------------------------------------

feature_schema_order = (
    production_feature_schema
    .sort_values(
        "FeaturePosition",
        kind="mergesort",
    )[
        "Feature"
    ]
    .astype(str)
    .tolist()
)

assert (
    feature_schema_order
    == production_feature_columns
), (
    "The saved feature-schema CSV does not match "
    "the serialized production bundle."
)


# ------------------------------------------------------------
# Training metadata validation
# ------------------------------------------------------------

training_metadata = (
    production_bundle[
        "Training"
    ]
)

assert (
    training_metadata[
        "FixtureCount"
    ]
    == 4180
)

assert (
    training_metadata[
        "SeasonCount"
    ]
    == 11
)

assert (
    training_metadata[
        "FirstSeason"
    ]
    == "2015-16"
)

assert (
    training_metadata[
        "LastSeason"
    ]
    == "2025-26"
)


# ------------------------------------------------------------
# Compact validation table
# ------------------------------------------------------------

production_package_validation = pd.DataFrame(
    [
        {
            "Validation": (
                "Recorded hashes verified"
            ),
            "Value": int(
                hash_validation_table[
                    "HashMatch"
                ].sum()
            ),
            "Expected": len(
                hash_validation_table
            ),
            "Status": "PASS",
        },
        {
            "Validation": (
                "Frozen feature count"
            ),
            "Value": len(
                production_feature_columns
            ),
            "Expected": 70,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Home numerical predictors"
            ),
            "Value": (
                production_home_model
                .n_features_in_
            ),
            "Expected": 55,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Away numerical predictors"
            ),
            "Value": (
                production_away_model
                .n_features_in_
            ),
            "Expected": 70,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Historical training fixtures"
            ),
            "Value": (
                training_metadata[
                    "FixtureCount"
                ]
            ),
            "Expected": 4180,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Historical seasons"
            ),
            "Value": (
                training_metadata[
                    "SeasonCount"
                ]
            ),
            "Expected": 11,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Probability order"
            ),
            "Value": (
                PROBABILITY_ORDER
            ),
            "Expected": [
                "H",
                "D",
                "A",
            ],
            "Status": "PASS",
        },
        {
            "Validation": (
                "Scoreline maximum goals"
            ),
            "Value": (
                MAX_MODELLED_GOALS
            ),
            "Expected": 10,
            "Status": "PASS",
        },
    ]
)


production_package_profile = pd.DataFrame(
    [
        {
            "Metric": "Artifact version",
            "Value": (
                production_bundle[
                    "ArtifactVersion"
                ]
            ),
        },
        {
            "Metric": "Model family",
            "Value": (
                production_bundle[
                    "ModelFamily"
                ]
            ),
        },
        {
            "Metric": "Calibration",
            "Value": (
                production_bundle[
                    "CalibrationMethod"
                ]
            ),
        },
        {
            "Metric": "Training fixtures",
            "Value": (
                training_metadata[
                    "FixtureCount"
                ]
            ),
        },
        {
            "Metric": "Training seasons",
            "Value": (
                training_metadata[
                    "SeasonCount"
                ]
            ),
        },
        {
            "Metric": "Training cutoff",
            "Value": (
                training_metadata[
                    "TrainingCutoff"
                ]
            ),
        },
        {
            "Metric": "Frozen predictors",
            "Value": len(
                production_feature_columns
            ),
        },
        {
            "Metric": "Home predictors",
            "Value": len(
                production_home_feature_columns
            ),
        },
        {
            "Metric": "Away predictors",
            "Value": (
                production_away_model
                .n_features_in_
            ),
        },
        {
            "Metric": "Home solver",
            "Value": (
                production_home_model
                .solver
            ),
        },
        {
            "Metric": "Away solver",
            "Value": (
                production_away_model
                .solver
            ),
        },
    ]
)


production_package_ready = True


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print(
    "Frozen production package profile:"
)
display(
    production_package_profile
)

print(
    "\nProduction package validation:"
)
display(
    production_package_validation
)

print(
    "\nArtifact hash validation:"
)
display(
    hash_validation_table
)

print(
    "\nProduction package status: VALID"
)

print(
    "Ready for 2026-27 fixtures:",
    production_package_ready,
)

Frozen production package profile:


,Metric,Value
0,Artifact version,production-refit-2026-27-v1
1,Model family,Independent Poisson regression
2,Calibration,Original probabilities
3,Training fixtures,4180
4,Training seasons,11
5,Training cutoff,2026-05-24T00:00:00
6,Frozen predictors,70
7,Home predictors,55
8,Away predictors,70
9,Home solver,newton-cholesky



Production package validation:


,Validation,Value,Expected,Status
0,Recorded hashes verified,6,6,PASS
1,Frozen feature count,70,70,PASS
2,Home numerical predictors,55,55,PASS
3,Away numerical predictors,70,70,PASS
4,Historical training fixtures,4180,4180,PASS
5,Historical seasons,11,11,PASS
6,Probability order,"[H, D, A]","[H, D, A]",PASS
7,Scoreline maximum goals,10,10,PASS



Artifact hash validation:


,Artifact,Exists,HashMatch
0,outputs/production_model/production_model_bund...,True,True
1,outputs/production_model/production_model_meta...,True,True
2,outputs/production_model/validated_implementat...,True,True
3,outputs/production_model/validated_feature_sch...,True,True
4,outputs/production_model/production_reference_...,True,True
5,outputs/production_model/production_reference_...,True,True



Production package status: VALID
Ready for 2026-27 fixtures: True


### Results and Interpretation

The frozen production package was loaded successfully from the artefacts exported by Notebook 10.

All package-integrity and structural validation checks passed. The recorded SHA-256 hashes match the current production files, confirming that the fitted artefacts have not changed since export.

The reloaded package retains the complete production specification: 70 frozen predictors, a 55-column full-rank numerical basis for the home-goal model, all 70 predictors for the away-goal model, the fitted preprocessing transformations, the independent Poisson scoreline specification and the fixed probability order `(H, D, A)`.

The historical training metadata also remains intact, covering 4,180 completed Premier League fixtures across eleven seasons through 2025–26.

The production package is therefore valid and frozen. No further fitting, feature selection, calibration or hyperparameter tuning will be performed in this notebook.

## 2. Load and Validate the 2026–27 Premier League Fixtures

The frozen production model requires a complete and internally consistent set of genuinely unseen fixtures before forecasting can begin.

The official Premier League 2026–27 fixture schedule is therefore downloaded and converted into a canonical modelling table containing fixture date, provisional kickoff time, home team and away team.

The source schedule contains 380 fixtures between 20 clubs. Validation requires each club to appear exactly 38 times, with 19 home fixtures and 19 away fixtures, and every pair of clubs to meet exactly twice with the home venue reversed.

Official club names are retained for presentation while an additional modelling-name representation is created to match the naming convention used by the historical feature-engineering data.

Fixtures remain subject to later scheduling amendments. The retrieved schedule is therefore stored locally together with its retrieval timestamp and source information so that the exact forecast input can be reproduced subsequently.

No model probabilities or engineered predictors are created in this section.

In [4]:
# ============================================================
# 2. Load and Validate the 2026-27 Premier League Fixtures
# ============================================================

from __future__ import annotations

import html
import json
import re
from datetime import datetime, timezone
from html.parser import HTMLParser
from pathlib import Path
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

SEASON = "2026-27"

OFFICIAL_FIXTURE_URL = (
    "https://www.premierleague.com/en/news/"
    "4675097/all-380-fixtures-for-202627-premier-league-season"
)

EXPECTED_FIXTURES = 380
EXPECTED_TEAMS = 20
EXPECTED_FIXTURES_PER_TEAM = 38
EXPECTED_HOME_FIXTURES = 19
EXPECTED_AWAY_FIXTURES = 19


# ------------------------------------------------------------
# Locate project
# ------------------------------------------------------------

project_root = Path.cwd().resolve()

while (
    project_root.name != "premier-league-probability-engine"
    and project_root.parent != project_root
):
    project_root = project_root.parent

assert project_root.name == "premier-league-probability-engine"


fixture_output_path = (
    project_root
    / "data"
    / "raw"
    / "premier_league_fixtures_2026_27_official.csv"
)

fixture_metadata_path = (
    project_root
    / "data"
    / "raw"
    / "premier_league_fixtures_2026_27_metadata.json"
)

historical_results_path = (
    project_root
    / "data"
    / "raw"
    / "premier_league_goal_targets_2015_16_to_2025_26.csv"
)


# ------------------------------------------------------------
# Download official Premier League fixture article
# ------------------------------------------------------------

request = Request(
    OFFICIAL_FIXTURE_URL,
    headers={
        "User-Agent": (
            "Mozilla/5.0 "
            "Premier-League-Probability-Engine/1.0"
        )
    },
)

try:
    with urlopen(
        request,
        timeout=30,
    ) as response:
        source_html = response.read().decode(
            "utf-8",
            errors="replace",
        )

except Exception as error:
    raise RuntimeError(
        "Could not download the official 2026-27 "
        "Premier League fixture page."
    ) from error


assert source_html.strip(), (
    "The official fixture page returned no content."
)


# ------------------------------------------------------------
# Extract visible text using only the Python standard library
# ------------------------------------------------------------

class VisibleTextParser(HTMLParser):

    block_tags = {
        "article",
        "br",
        "div",
        "h1",
        "h2",
        "h3",
        "h4",
        "li",
        "p",
        "section",
    }

    ignored_tags = {
        "script",
        "style",
        "noscript",
        "svg",
    }

    def __init__(self):
        super().__init__()
        self.parts = []
        self.ignore_depth = 0

    def handle_starttag(
        self,
        tag,
        attrs,
    ):
        if tag in self.ignored_tags:
            self.ignore_depth += 1

        elif (
            self.ignore_depth == 0
            and tag in self.block_tags
        ):
            self.parts.append("\n")

    def handle_endtag(
        self,
        tag,
    ):
        if tag in self.ignored_tags:
            self.ignore_depth = max(
                0,
                self.ignore_depth - 1,
            )

        elif (
            self.ignore_depth == 0
            and tag in self.block_tags
        ):
            self.parts.append("\n")

    def handle_data(
        self,
        data,
    ):
        if self.ignore_depth == 0:
            self.parts.append(data)


parser = VisibleTextParser()
parser.feed(source_html)

page_text = html.unescape(
    "".join(
        parser.parts
    )
)

source_lines = [
    re.sub(
        r"\s+",
        " ",
        line,
    ).strip()
    for line in page_text.splitlines()
]

source_lines = [
    line
    for line in source_lines
    if line
]


# ------------------------------------------------------------
# Date-heading parser
# ------------------------------------------------------------

weekday_pattern = (
    r"(?:Monday|Tuesday|Wednesday|Thursday|Friday|"
    r"Saturday|Sunday)"
)

month_pattern = (
    r"(?:January|February|March|April|May|June|"
    r"July|August|September|October|November|December)"
)

date_heading_pattern = re.compile(
    rf"^{weekday_pattern}\s+"
    rf"(?P<day>\d{{1,2}})\s+"
    rf"(?P<month>{month_pattern})"
    rf"(?:\s+(?P<year>\d{{4}}))?$"
)


# ------------------------------------------------------------
# Parse fixture lines
# ------------------------------------------------------------

fixture_records = []

current_date = None
current_year = 2026
previous_month_number = None
source_order = 0


for line in source_lines:

    date_match = date_heading_pattern.match(
        line
    )

    if date_match is not None:

        day = int(
            date_match.group(
                "day"
            )
        )

        month_name = (
            date_match.group(
                "month"
            )
        )

        explicit_year = (
            date_match.group(
                "year"
            )
        )

        month_number = int(
            pd.Timestamp(
                f"2000-{month_name}-01"
            ).month
        )

        if explicit_year is not None:

            current_year = int(
                explicit_year
            )

        elif (
            previous_month_number is not None
            and month_number
            < previous_month_number
        ):
            current_year += 1

        current_date = pd.Timestamp(
            year=current_year,
            month=month_number,
            day=day,
        )

        previous_month_number = (
            month_number
        )

        continue


    if (
        current_date is None
        or " v " not in line
        or line.startswith("*")
    ):
        continue


    fixture_line = line.strip()


    # Remove TV annotations and trailing footnote stars.
    fixture_line = re.sub(
        r"\s+\([^)]*\)\*{0,4}\s*$",
        "",
        fixture_line,
    )

    fixture_line = re.sub(
        r"\*+$",
        "",
        fixture_line,
    ).strip()


    # Extract explicit kickoff time where supplied.
    time_match = re.match(
        r"^(?P<time>\d{1,2}:\d{2})\s+"
        r"(?P<fixture>.+)$",
        fixture_line,
    )

    if time_match is not None:

        kickoff_time = (
            time_match.group(
                "time"
            )
        )

        fixture_text = (
            time_match.group(
                "fixture"
            )
        )

        time_source = (
            "Explicit in official schedule"
        )

    else:

        fixture_text = fixture_line

        # The official article states that unspecified
        # weekend/Bank Holiday games default to 15:00 and
        # midweek games to 20:00.
        if current_date.weekday() in {
            1,
            2,
            3,
        }:
            kickoff_time = "20:00"

        else:
            kickoff_time = "15:00"

        time_source = (
            "Default stated by official schedule"
        )


    fixture_parts = fixture_text.split(
        " v ",
        maxsplit=1,
    )

    if len(fixture_parts) != 2:
        continue


    home_team = (
        fixture_parts[0].strip()
    )

    away_team = (
        fixture_parts[1].strip()
    )


    # Ignore any non-fixture article text that happened
    # to contain " v ".
    if not home_team or not away_team:
        continue


    source_order += 1

    fixture_records.append(
        {
            "Season": SEASON,
            "Date": current_date,
            "Time": kickoff_time,
            "OfficialHomeTeam": (
                home_team
            ),
            "OfficialAwayTeam": (
                away_team
            ),
            "TimeSource": (
                time_source
            ),
            "SourceOrder": (
                source_order
            ),
        }
    )


fixtures = pd.DataFrame(
    fixture_records
)


assert not fixtures.empty, (
    "No fixtures were parsed from the official page."
)


# ------------------------------------------------------------
# The article should produce exactly 380 fixtures
# ------------------------------------------------------------

if len(fixtures) != EXPECTED_FIXTURES:

    display(
        fixtures.head(20)
    )

    display(
        fixtures.tail(20)
    )

    raise RuntimeError(
        "Official fixture parser did not recover exactly "
        f"380 matches; recovered {len(fixtures)}."
    )


# ------------------------------------------------------------
# Convert official team names to historical model conventions
# ------------------------------------------------------------

official_to_model_team = {
    "AFC Bournemouth": "Bournemouth",
    "Brighton & Hove Albion": "Brighton",
    "Coventry City": "Coventry",
    "Leeds United": "Leeds",
    "Hull City": "Hull",
    "Ipswich Town": "Ipswich",
    "Manchester City": "Man City",
    "Manchester United": "Man United",
    "Newcastle United": "Newcastle",
    "Nottingham Forest": "Nott'm Forest",
    "Tottenham Hotspur": "Tottenham",
}


def model_team_name(
    official_name,
):
    return official_to_model_team.get(
        official_name,
        official_name,
    )


fixtures["HomeTeam"] = (
    fixtures[
        "OfficialHomeTeam"
    ].map(
        model_team_name
    )
)

fixtures["AwayTeam"] = (
    fixtures[
        "OfficialAwayTeam"
    ].map(
        model_team_name
    )
)


# ------------------------------------------------------------
# Construct kickoff timestamp
# ------------------------------------------------------------

fixtures["Kickoff"] = pd.to_datetime(
    fixtures[
        "Date"
    ].dt.strftime(
        "%Y-%m-%d"
    )
    + " "
    + fixtures["Time"],
    errors="raise",
)

fixtures = (
    fixtures
    .sort_values(
        by=[
            "Kickoff",
            "SourceOrder",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

fixtures.insert(
    0,
    "FixtureNumber",
    np.arange(
        1,
        len(fixtures) + 1,
    ),
)


# ------------------------------------------------------------
# Core fixture validation
# ------------------------------------------------------------

official_teams = sorted(
    set(
        fixtures[
            "OfficialHomeTeam"
        ]
    )
    | set(
        fixtures[
            "OfficialAwayTeam"
        ]
    )
)

model_teams = sorted(
    set(
        fixtures[
            "HomeTeam"
        ]
    )
    | set(
        fixtures[
            "AwayTeam"
        ]
    )
)


assert len(
    official_teams
) == EXPECTED_TEAMS

assert len(
    model_teams
) == EXPECTED_TEAMS


assert not (
    fixtures[
        "OfficialHomeTeam"
    ]
    == fixtures[
        "OfficialAwayTeam"
    ]
).any(), (
    "At least one fixture has the same home and away team."
)


duplicate_fixture_count = int(
    fixtures.duplicated(
        subset=[
            "OfficialHomeTeam",
            "OfficialAwayTeam",
        ]
    ).sum()
)

assert duplicate_fixture_count == 0, (
    "At least one ordered home/away pairing appears twice."
)


# ------------------------------------------------------------
# Team-level 38 / 19 / 19 audit
# ------------------------------------------------------------

team_fixture_records = []

for team in official_teams:

    home_count = int(
        (
            fixtures[
                "OfficialHomeTeam"
            ]
            == team
        ).sum()
    )

    away_count = int(
        (
            fixtures[
                "OfficialAwayTeam"
            ]
            == team
        ).sum()
    )

    team_fixture_records.append(
        {
            "Team": team,
            "HomeFixtures": (
                home_count
            ),
            "AwayFixtures": (
                away_count
            ),
            "TotalFixtures": (
                home_count
                + away_count
            ),
        }
    )


team_fixture_audit = pd.DataFrame(
    team_fixture_records
)


assert (
    team_fixture_audit[
        "HomeFixtures"
    ]
    == EXPECTED_HOME_FIXTURES
).all()

assert (
    team_fixture_audit[
        "AwayFixtures"
    ]
    == EXPECTED_AWAY_FIXTURES
).all()

assert (
    team_fixture_audit[
        "TotalFixtures"
    ]
    == EXPECTED_FIXTURES_PER_TEAM
).all()


# ------------------------------------------------------------
# Validate every pair meets once at each venue
# ------------------------------------------------------------

ordered_pairs = set(
    zip(
        fixtures[
            "OfficialHomeTeam"
        ],
        fixtures[
            "OfficialAwayTeam"
        ],
    )
)

missing_reverse_pairings = []

for (
    home_team,
    away_team,
) in ordered_pairs:

    if (
        away_team,
        home_team,
    ) not in ordered_pairs:

        missing_reverse_pairings.append(
            (
                home_team,
                away_team,
            )
        )


assert not missing_reverse_pairings, (
    "Some team pairings do not have the required "
    f"reverse fixture: {missing_reverse_pairings}"
)


expected_ordered_pair_count = (
    EXPECTED_TEAMS
    * (
        EXPECTED_TEAMS - 1
    )
)

assert len(
    ordered_pairs
) == expected_ordered_pair_count


# ------------------------------------------------------------
# Season-boundary validation
# ------------------------------------------------------------

first_fixture_date = (
    fixtures["Date"].min()
)

last_fixture_date = (
    fixtures["Date"].max()
)


assert first_fixture_date == pd.Timestamp(
    "2026-08-21"
)

assert last_fixture_date == pd.Timestamp(
    "2027-05-30"
)


# ------------------------------------------------------------
# Compare league membership with 2025-26 training season
# ------------------------------------------------------------

teams_seen_in_training = set()

previous_season_teams = set()


if historical_results_path.is_file():

    historical_results = pd.read_csv(
        historical_results_path,
        low_memory=False,
    )

    teams_seen_in_training = (
        set(
            historical_results[
                "HomeTeam"
            ].dropna()
        )
        | set(
            historical_results[
                "AwayTeam"
            ].dropna()
        )
    )

    previous_season_rows = (
        historical_results.loc[
            historical_results[
                "Season"
            ].astype(str)
            == "2025-26"
        ]
    )

    previous_season_teams = (
        set(
            previous_season_rows[
                "HomeTeam"
            ].dropna()
        )
        | set(
            previous_season_rows[
                "AwayTeam"
            ].dropna()
        )
    )


new_teams_vs_2025_26 = sorted(
    set(model_teams)
    - previous_season_teams
)

departed_teams_vs_2025_26 = sorted(
    previous_season_teams
    - set(model_teams)
)

teams_never_seen_in_training = sorted(
    set(model_teams)
    - teams_seen_in_training
)


# ------------------------------------------------------------
# Save canonical fixture list
# ------------------------------------------------------------

fixtures_to_save = fixtures[
    [
        "FixtureNumber",
        "Season",
        "Date",
        "Time",
        "Kickoff",
        "OfficialHomeTeam",
        "OfficialAwayTeam",
        "HomeTeam",
        "AwayTeam",
        "TimeSource",
        "SourceOrder",
    ]
].copy()


fixtures_to_save[
    "Date"
] = fixtures_to_save[
    "Date"
].dt.strftime(
    "%Y-%m-%d"
)

fixtures_to_save[
    "Kickoff"
] = fixtures_to_save[
    "Kickoff"
].dt.strftime(
    "%Y-%m-%d %H:%M:%S"
)


fixture_output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

fixtures_to_save.to_csv(
    fixture_output_path,
    index=False,
)


retrieved_utc = datetime.now(
    timezone.utc
).isoformat()


fixture_metadata = {
    "Season": SEASON,
    "Source": "Premier League official fixture article",
    "SourceURL": OFFICIAL_FIXTURE_URL,
    "RetrievedUTC": retrieved_utc,
    "FixtureCount": int(
        len(fixtures)
    ),
    "TeamCount": int(
        len(official_teams)
    ),
    "FirstFixtureDate": (
        first_fixture_date
        .date()
        .isoformat()
    ),
    "LastFixtureDate": (
        last_fixture_date
        .date()
        .isoformat()
    ),
    "NewTeamsVs2025_26": (
        new_teams_vs_2025_26
    ),
    "DepartedTeamsVs2025_26": (
        departed_teams_vs_2025_26
    ),
    "TeamsNeverSeenInTraining": (
        teams_never_seen_in_training
    ),
    "ScheduleStatus": (
        "Official schedule; fixtures remain subject to change"
    ),
}


fixture_metadata_path.write_text(
    json.dumps(
        fixture_metadata,
        indent=4,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)


# ------------------------------------------------------------
# Validation report
# ------------------------------------------------------------

fixture_validation_table = pd.DataFrame(
    [
        {
            "Validation": "Fixture count",
            "Value": len(
                fixtures
            ),
            "Expected": 380,
            "Status": "PASS",
        },
        {
            "Validation": "Unique teams",
            "Value": len(
                official_teams
            ),
            "Expected": 20,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Ordered fixture pairs"
            ),
            "Value": len(
                ordered_pairs
            ),
            "Expected": 380,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Duplicate ordered pairs"
            ),
            "Value": (
                duplicate_fixture_count
            ),
            "Expected": 0,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Teams with 38 fixtures"
            ),
            "Value": int(
                (
                    team_fixture_audit[
                        "TotalFixtures"
                    ]
                    == 38
                ).sum()
            ),
            "Expected": 20,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Teams with 19 home fixtures"
            ),
            "Value": int(
                (
                    team_fixture_audit[
                        "HomeFixtures"
                    ]
                    == 19
                ).sum()
            ),
            "Expected": 20,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Teams with 19 away fixtures"
            ),
            "Value": int(
                (
                    team_fixture_audit[
                        "AwayFixtures"
                    ]
                    == 19
                ).sum()
            ),
            "Expected": 20,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Missing reverse pairings"
            ),
            "Value": len(
                missing_reverse_pairings
            ),
            "Expected": 0,
            "Status": "PASS",
        },
    ]
)


fixture_profile = pd.DataFrame(
    [
        {
            "Metric": "Season",
            "Value": SEASON,
        },
        {
            "Metric": "Fixtures",
            "Value": len(
                fixtures
            ),
        },
        {
            "Metric": "Teams",
            "Value": len(
                official_teams
            ),
        },
        {
            "Metric": "First fixture",
            "Value": (
                first_fixture_date
                .date()
                .isoformat()
            ),
        },
        {
            "Metric": "Last fixture",
            "Value": (
                last_fixture_date
                .date()
                .isoformat()
            ),
        },
        {
            "Metric": (
                "New teams vs 2025-26"
            ),
            "Value": (
                new_teams_vs_2025_26
            ),
        },
        {
            "Metric": (
                "Departed teams vs 2025-26"
            ),
            "Value": (
                departed_teams_vs_2025_26
            ),
        },
        {
            "Metric": (
                "Never seen in training"
            ),
            "Value": (
                teams_never_seen_in_training
            ),
        },
    ]
)


fixture_list_ready = True


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print(
    "2026-27 fixture profile:"
)
display(
    fixture_profile
)

print(
    "\nFixture-list validation:"
)
display(
    fixture_validation_table
)

print(
    "\nTeam fixture audit:"
)
display(
    team_fixture_audit
)

print(
    "\nOpening 2026-27 fixtures:"
)
display(
    fixtures[
        [
            "FixtureNumber",
            "Kickoff",
            "OfficialHomeTeam",
            "OfficialAwayTeam",
            "HomeTeam",
            "AwayTeam",
        ]
    ].head(20)
)

print(
    "\nCanonical fixture file:",
    fixture_output_path
    .relative_to(
        project_root
    )
    .as_posix(),
)

print(
    "\n2026-27 fixture-list status: VALID"
)

print(
    "Ready for forecast feature construction:",
    fixture_list_ready,
)

2026-27 fixture profile:


,Metric,Value
0,Season,2026-27
1,Fixtures,380
2,Teams,20
3,First fixture,2026-08-21
4,Last fixture,2027-05-30
5,New teams vs 2025-26,"[Coventry, Hull, Ipswich]"
6,Departed teams vs 2025-26,"[Burnley, West Ham, Wolves]"
7,Never seen in training,[Coventry]



Fixture-list validation:


,Validation,Value,Expected,Status
0,Fixture count,380,380,PASS
1,Unique teams,20,20,PASS
2,Ordered fixture pairs,380,380,PASS
3,Duplicate ordered pairs,0,0,PASS
4,Teams with 38 fixtures,20,20,PASS
5,Teams with 19 home fixtures,20,20,PASS
6,Teams with 19 away fixtures,20,20,PASS
7,Missing reverse pairings,0,0,PASS



Team fixture audit:


,Team,HomeFixtures,AwayFixtures,TotalFixtures
0,AFC Bournemouth,19,19,38
1,Arsenal,19,19,38
2,Aston Villa,19,19,38
3,Brentford,19,19,38
4,Brighton & Hove Albion,19,19,38
5,Chelsea,19,19,38
6,Coventry City,19,19,38
7,Crystal Palace,19,19,38
8,Everton,19,19,38
9,Fulham,19,19,38



Opening 2026-27 fixtures:


,FixtureNumber,Kickoff,OfficialHomeTeam,OfficialAwayTeam,HomeTeam,AwayTeam
0,1,2026-08-21 20:00:00,Arsenal,Coventry City,Arsenal,Coventry
1,2,2026-08-22 12:30:00,Hull City,Manchester United,Hull,Man United
2,3,2026-08-22 15:00:00,Everton,Crystal Palace,Everton,Crystal Palace
3,4,2026-08-22 15:00:00,Ipswich Town,Sunderland,Ipswich,Sunderland
4,5,2026-08-22 15:00:00,Nottingham Forest,Leeds United,Nott'm Forest,Leeds
5,6,2026-08-22 17:30:00,Brentford,Tottenham Hotspur,Brentford,Tottenham
6,7,2026-08-23 14:00:00,Brighton & Hove Albion,Aston Villa,Brighton,Aston Villa
7,8,2026-08-23 14:00:00,Manchester City,AFC Bournemouth,Man City,Bournemouth
8,9,2026-08-23 16:30:00,Newcastle United,Liverpool,Newcastle,Liverpool
9,10,2026-08-24 20:00:00,Fulham,Chelsea,Fulham,Chelsea



Canonical fixture file: data/raw/premier_league_fixtures_2026_27_official.csv

2026-27 fixture-list status: VALID
Ready for forecast feature construction: True


### Results and Interpretation

The official 2026–27 Premier League fixture list has been loaded and validated successfully.

The schedule contains exactly 380 fixtures involving 20 clubs. Every team appears in 38 matches, consisting of 19 home and 19 away fixtures, and every ordered home/away pairing occurs exactly once with a corresponding reverse fixture later in the season.

The fixture window and league membership also passed the season-level checks, while the official club names have been mapped to the historical naming conventions required by the production feature pipeline.

The canonical fixture schedule has therefore been established for forecasting.

An important forecasting distinction now applies. Before the season begins, the complete pre-match state is known only for each club's first league fixture. Later fixtures depend on information that does not yet exist, including updated Elo ratings, rolling form and league-table position. Producing all 380 probabilities immediately by filling those future states with assumed values would therefore change the meaning of the validated pre-match model.

The next section consequently constructs exact production features for the opening round and identifies the remaining fixtures as requiring sequential state updates as the 2026–27 season progresses.

## 3. Construct the Opening-Round Pre-Match Feature State

The frozen production model was designed for fixture-level pre-match forecasting rather than unconditional prediction of an entire season before any matches have been played.

Several predictors evolve after every completed fixture:

- Elo ratings;
- five-match general form;
- five-match venue form;
- cumulative league points and goal difference;
- league position.

Those values are known exactly for the opening round because no 2026–27 result has yet occurred. For later fixtures, however, they depend on the outcomes of preceding matches and cannot be constructed without introducing assumptions or simulated future states.

This section therefore identifies the first league fixture of every club and constructs the exact 70-column production state for those ten matches.

The opening-round feature conventions follow the established historical pipeline:

- Elo ratings are carried forward from the completed historical result sequence;
- five-match general and venue form are structurally missing at the beginning of a new season;
- first-match rest and short-rest variables are structurally missing;
- seven- and fourteen-day same-season congestion counts begin at zero;
- each club's match number begins at one;
- season progress begins at zero;
- league points and goal difference begin at zero;
- league position and position-category indicators are undefined before any league table exists.

These structural missing values are retained deliberately and will be handled by the fitted production median imputer.

The remaining 370 fixtures are retained in the schedule but are marked as requiring sequential state updates before genuine pre-match probabilities can be produced.

In [5]:
# ============================================================
# 3. Construct the Opening-Round Pre-Match Feature State
# ============================================================

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

TARGET_SEASON = "2026-27"

INITIAL_ELO = 1500.0
ELO_K_FACTOR = 20.0
ELO_HOME_ADVANTAGE = 50.0
ELO_SCALE = 400.0

EXPECTED_OPENING_FIXTURES = 10
EXPECTED_OPENING_TEAMS = 20
EXPECTED_FEATURE_COUNT = 70


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

assert "production_package_ready" in globals()
assert production_package_ready is True

assert "fixture_list_ready" in globals()
assert fixture_list_ready is True

assert len(production_feature_columns) == 70
assert len(fixtures) == 380


# ------------------------------------------------------------
# Locate historical result history
# ------------------------------------------------------------

project_root = Path.cwd().resolve()

while (
    project_root.name != "premier-league-probability-engine"
    and project_root.parent != project_root
):
    project_root = project_root.parent

assert project_root.name == "premier-league-probability-engine"


historical_results_path = (
    project_root
    / "data"
    / "raw"
    / "premier_league_goal_targets_2015_16_to_2025_26.csv"
)

opening_feature_output_path = (
    project_root
    / "outputs"
    / "forecasts"
    / "2026_27"
    / "opening_round_features.csv"
)

forecastability_output_path = (
    project_root
    / "outputs"
    / "forecasts"
    / "2026_27"
    / "fixture_forecastability.csv"
)

opening_feature_output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

assert historical_results_path.is_file()


# ============================================================
# A. IDENTIFY THE FIRST FIXTURE OF EVERY CLUB
# ============================================================

fixture_schedule = fixtures.copy()

fixture_schedule = (
    fixture_schedule
    .sort_values(
        by=[
            "Kickoff",
            "SourceOrder",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# First scheduled fixture for each club.
first_fixture_number_by_team = {}

for row in fixture_schedule.itertuples():

    if row.HomeTeam not in first_fixture_number_by_team:
        first_fixture_number_by_team[
            row.HomeTeam
        ] = row.FixtureNumber

    if row.AwayTeam not in first_fixture_number_by_team:
        first_fixture_number_by_team[
            row.AwayTeam
        ] = row.FixtureNumber


assert len(
    first_fixture_number_by_team
) == EXPECTED_OPENING_TEAMS


opening_fixture_numbers = sorted(
    set(
        first_fixture_number_by_team.values()
    )
)

assert len(
    opening_fixture_numbers
) == EXPECTED_OPENING_FIXTURES, (
    "The first scheduled fixture of every club should resolve "
    "to exactly ten opening-round matches."
)


opening_fixtures = (
    fixture_schedule.loc[
        fixture_schedule[
            "FixtureNumber"
        ].isin(
            opening_fixture_numbers
        )
    ]
    .sort_values(
        by=[
            "Kickoff",
            "SourceOrder",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


opening_teams = (
    set(
        opening_fixtures[
            "HomeTeam"
        ]
    )
    |
    set(
        opening_fixtures[
            "AwayTeam"
        ]
    )
)

assert len(opening_fixtures) == 10
assert len(opening_teams) == 20


team_opening_appearance_counts = pd.concat(
    [
        opening_fixtures[
            "HomeTeam"
        ],
        opening_fixtures[
            "AwayTeam"
        ],
    ],
    ignore_index=True,
).value_counts()

assert (
    team_opening_appearance_counts
    == 1
).all(), (
    "Every club must appear exactly once in the opening fixture set."
)


# ============================================================
# B. FORECASTABILITY AUDIT FOR ALL 380 FIXTURES
# ============================================================

fixture_forecastability = fixture_schedule[
    [
        "FixtureNumber",
        "Season",
        "Kickoff",
        "OfficialHomeTeam",
        "OfficialAwayTeam",
        "HomeTeam",
        "AwayTeam",
    ]
].copy()


fixture_forecastability[
    "ExactPreMatchStateAvailableNow"
] = (
    fixture_forecastability[
        "FixtureNumber"
    ].isin(
        opening_fixture_numbers
    )
)


fixture_forecastability[
    "ForecastState"
] = np.where(
    fixture_forecastability[
        "ExactPreMatchStateAvailableNow"
    ],
    "READY",
    "REQUIRES_SEQUENTIAL_UPDATE",
)


fixture_forecastability[
    "Reason"
] = np.where(
    fixture_forecastability[
        "ExactPreMatchStateAvailableNow"
    ],
    (
        "First league fixture for both clubs; "
        "all required pre-match state is available."
    ),
    (
        "At least one required feature depends on earlier "
        "2026-27 results, including Elo, rolling form or "
        "league-table state."
    ),
)


ready_fixture_count = int(
    fixture_forecastability[
        "ExactPreMatchStateAvailableNow"
    ].sum()
)

sequential_fixture_count = (
    len(fixture_forecastability)
    - ready_fixture_count
)

assert ready_fixture_count == 10
assert sequential_fixture_count == 370


# ============================================================
# C. REPLAY HISTORICAL ELO THROUGH THE END OF 2025-26
# ============================================================

historical_results = pd.read_csv(
    historical_results_path,
    low_memory=False,
).copy()


required_historical_columns = [
    "Season",
    "Date",
    "HomeTeam",
    "AwayTeam",
    "FTHG",
    "FTAG",
    "FTR",
]

missing_historical_columns = [
    column
    for column in required_historical_columns
    if column not in historical_results.columns
]

assert not missing_historical_columns, (
    "Historical result data is missing columns: "
    f"{missing_historical_columns}"
)


historical_results[
    "Date"
] = pd.to_datetime(
    historical_results[
        "Date"
    ],
    errors="raise",
).dt.normalize()


historical_results[
    "_HistoricalOrder"
] = np.arange(
    len(historical_results)
)


historical_results = (
    historical_results
    .sort_values(
        by=[
            "Date",
            "_HistoricalOrder",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


assert len(historical_results) == 4180
assert historical_results["Season"].iloc[-1] == "2025-26"


def expected_home_elo_score(
    home_rating: float,
    away_rating: float,
) -> float:

    return 1.0 / (
        1.0
        + 10.0 ** (
            (
                away_rating
                - (
                    home_rating
                    + ELO_HOME_ADVANTAGE
                )
            )
            / ELO_SCALE
        )
    )


final_historical_elo = {}


for row in historical_results.itertuples(
    index=False
):

    home_rating = final_historical_elo.get(
        row.HomeTeam,
        INITIAL_ELO,
    )

    away_rating = final_historical_elo.get(
        row.AwayTeam,
        INITIAL_ELO,
    )


    expected_home_score = (
        expected_home_elo_score(
            home_rating,
            away_rating,
        )
    )


    if row.FTR == "H":
        actual_home_score = 1.0

    elif row.FTR == "D":
        actual_home_score = 0.5

    elif row.FTR == "A":
        actual_home_score = 0.0

    else:
        raise ValueError(
            f"Unexpected historical result label: {row.FTR!r}"
        )


    rating_change = (
        ELO_K_FACTOR
        * (
            actual_home_score
            - expected_home_score
        )
    )


    final_historical_elo[
        row.HomeTeam
    ] = (
        home_rating
        + rating_change
    )

    final_historical_elo[
        row.AwayTeam
    ] = (
        away_rating
        - rating_change
    )


# ------------------------------------------------------------
# Elo values for the 20 2026-27 teams
# ------------------------------------------------------------

opening_team_elo = pd.DataFrame(
    {
        "Team": sorted(
            opening_teams
        ),
    }
)

opening_team_elo[
    "PreSeasonElo"
] = (
    opening_team_elo[
        "Team"
    ].map(
        final_historical_elo
    )
)


opening_team_elo[
    "SeenPreviously"
] = (
    opening_team_elo[
        "PreSeasonElo"
    ].notna()
)


opening_team_elo[
    "PreSeasonElo"
] = (
    opening_team_elo[
        "PreSeasonElo"
    ].fillna(
        INITIAL_ELO
    )
)


# Teams never seen in the Premier League training history
# receive the same initial Elo convention used by the
# historical Elo process.
never_seen_opening_teams = (
    opening_team_elo.loc[
        ~opening_team_elo[
            "SeenPreviously"
        ],
        "Team",
    ]
    .tolist()
)


# ============================================================
# D. CONFIRM THE EXACT 70-COLUMN FEATURE CONTRACT
# ============================================================

expected_feature_columns = {
    # Elo
    "HomeEloBefore",
    "AwayEloBefore",
    "EloDifference",

    # General form
    "HomeRollingPoints5",
    "AwayRollingPoints5",
    "HomeRollingGoalsFor5",
    "AwayRollingGoalsFor5",
    "HomeRollingGoalsAgainst5",
    "AwayRollingGoalsAgainst5",
    "HomeRollingGoalDifference5",
    "AwayRollingGoalDifference5",
    "HomeRollingWinRate5",
    "AwayRollingWinRate5",

    # General form differences
    "PointsFormDifference5",
    "GoalsForFormDifference5",
    "GoalsAgainstFormDifference5",
    "GoalDifferenceFormDifference5",
    "WinRateFormDifference5",

    # Venue form
    "HomeVenueRollingPoints5",
    "AwayVenueRollingPoints5",
    "HomeVenueRollingGoalsFor5",
    "AwayVenueRollingGoalsFor5",
    "HomeVenueRollingGoalsAgainst5",
    "AwayVenueRollingGoalsAgainst5",
    "HomeVenueRollingGoalDifference5",
    "AwayVenueRollingGoalDifference5",
    "HomeVenueRollingWinRate5",
    "AwayVenueRollingWinRate5",

    # Venue form differences
    "VenuePointsFormDifference5",
    "VenueGoalsForFormDifference5",
    "VenueGoalsAgainstFormDifference5",
    "VenueGoalDifferenceFormDifference5",
    "VenueWinRateFormDifference5",

    # Rest / congestion
    "HomeRestDays",
    "AwayRestDays",
    "HomeShortRest3",
    "AwayShortRest3",
    "HomeMatchesLast7Days",
    "AwayMatchesLast7Days",
    "HomeMatchesLast14Days",
    "AwayMatchesLast14Days",
    "RestDaysDifference",
    "ShortRestDifference3",
    "MatchesLast7DaysDifference",
    "MatchesLast14DaysDifference",

    # Match number / season progress
    "HomeMatchWeek",
    "AwayMatchWeek",
    "HomeSeasonProgress",
    "AwaySeasonProgress",
    "AverageSeasonProgress",
    "SeasonProgressDifference",
    "MatchesPlayedDifference",
    "SecondHalfSeason",

    # League table
    "HomePointsBefore",
    "AwayPointsBefore",
    "HomeLeaguePosition",
    "AwayLeaguePosition",
    "HomeGoalDifferenceBefore",
    "AwayGoalDifferenceBefore",
    "PointsDifference",
    "PositionDifference",
    "GoalDifferenceDifference",

    # Position categories
    "HomeTop4Before",
    "AwayTop4Before",
    "HomeTop6Before",
    "AwayTop6Before",
    "HomeTopHalfBefore",
    "AwayTopHalfBefore",
    "HomeBottom3Before",
    "AwayBottom3Before",
}


assert len(
    expected_feature_columns
) == EXPECTED_FEATURE_COUNT


actual_feature_columns = set(
    production_feature_columns
)

missing_expected_features = sorted(
    expected_feature_columns
    - actual_feature_columns
)

unexpected_frozen_features = sorted(
    actual_feature_columns
    - expected_feature_columns
)


assert not missing_expected_features, (
    "Frozen schema is missing expected feature names: "
    f"{missing_expected_features}"
)

assert not unexpected_frozen_features, (
    "Frozen schema contains features not accounted for by "
    f"the opening-state constructor: {unexpected_frozen_features}"
)


# ============================================================
# E. CREATE THE OPENING-ROUND FEATURE MATRIX
# ============================================================

opening_round_features = pd.DataFrame(
    np.nan,
    index=opening_fixtures.index,
    columns=production_feature_columns,
    dtype=float,
)


def set_opening_feature(
    feature_name: str,
    values,
) -> None:

    assert feature_name in opening_round_features.columns, (
        f"Unknown frozen feature: {feature_name}"
    )

    opening_round_features[
        feature_name
    ] = values


# ------------------------------------------------------------
# Elo
# ------------------------------------------------------------

home_elo_values = (
    opening_fixtures[
        "HomeTeam"
    ]
    .map(
        final_historical_elo
    )
    .fillna(
        INITIAL_ELO
    )
    .astype(float)
)

away_elo_values = (
    opening_fixtures[
        "AwayTeam"
    ]
    .map(
        final_historical_elo
    )
    .fillna(
        INITIAL_ELO
    )
    .astype(float)
)


set_opening_feature(
    "HomeEloBefore",
    home_elo_values,
)

set_opening_feature(
    "AwayEloBefore",
    away_elo_values,
)

set_opening_feature(
    "EloDifference",
    (
        home_elo_values
        - away_elo_values
    ),
)


# ------------------------------------------------------------
# General and venue rolling form
#
# These remain NaN deliberately. The historical feature
# pipeline resets rolling windows at each new season.
# ------------------------------------------------------------

rolling_features = [
    "HomeRollingPoints5",
    "AwayRollingPoints5",
    "HomeRollingGoalsFor5",
    "AwayRollingGoalsFor5",
    "HomeRollingGoalsAgainst5",
    "AwayRollingGoalsAgainst5",
    "HomeRollingGoalDifference5",
    "AwayRollingGoalDifference5",
    "HomeRollingWinRate5",
    "AwayRollingWinRate5",
    "PointsFormDifference5",
    "GoalsForFormDifference5",
    "GoalsAgainstFormDifference5",
    "GoalDifferenceFormDifference5",
    "WinRateFormDifference5",
    "HomeVenueRollingPoints5",
    "AwayVenueRollingPoints5",
    "HomeVenueRollingGoalsFor5",
    "AwayVenueRollingGoalsFor5",
    "HomeVenueRollingGoalsAgainst5",
    "AwayVenueRollingGoalsAgainst5",
    "HomeVenueRollingGoalDifference5",
    "AwayVenueRollingGoalDifference5",
    "HomeVenueRollingWinRate5",
    "AwayVenueRollingWinRate5",
    "VenuePointsFormDifference5",
    "VenueGoalsForFormDifference5",
    "VenueGoalsAgainstFormDifference5",
    "VenueGoalDifferenceFormDifference5",
    "VenueWinRateFormDifference5",
]

for feature in rolling_features:
    set_opening_feature(
        feature,
        np.nan,
    )


# ------------------------------------------------------------
# Rest and congestion
#
# First league fixture => no same-season previous match.
# ------------------------------------------------------------

for feature in [
    "HomeRestDays",
    "AwayRestDays",
    "HomeShortRest3",
    "AwayShortRest3",
    "RestDaysDifference",
    "ShortRestDifference3",
]:
    set_opening_feature(
        feature,
        np.nan,
    )


for feature in [
    "HomeMatchesLast7Days",
    "AwayMatchesLast7Days",
    "HomeMatchesLast14Days",
    "AwayMatchesLast14Days",
    "MatchesLast7DaysDifference",
    "MatchesLast14DaysDifference",
]:
    set_opening_feature(
        feature,
        0.0,
    )


# ------------------------------------------------------------
# Match number and season progress
# ------------------------------------------------------------

set_opening_feature(
    "HomeMatchWeek",
    1.0,
)

set_opening_feature(
    "AwayMatchWeek",
    1.0,
)

set_opening_feature(
    "HomeSeasonProgress",
    0.0,
)

set_opening_feature(
    "AwaySeasonProgress",
    0.0,
)

set_opening_feature(
    "AverageSeasonProgress",
    0.0,
)

set_opening_feature(
    "SeasonProgressDifference",
    0.0,
)

set_opening_feature(
    "MatchesPlayedDifference",
    0.0,
)

set_opening_feature(
    "SecondHalfSeason",
    0.0,
)


# ------------------------------------------------------------
# Opening league-table state
# ------------------------------------------------------------

set_opening_feature(
    "HomePointsBefore",
    0.0,
)

set_opening_feature(
    "AwayPointsBefore",
    0.0,
)

set_opening_feature(
    "HomeGoalDifferenceBefore",
    0.0,
)

set_opening_feature(
    "AwayGoalDifferenceBefore",
    0.0,
)

set_opening_feature(
    "PointsDifference",
    0.0,
)

set_opening_feature(
    "GoalDifferenceDifference",
    0.0,
)


# No league ranking exists before any 2026-27 match.
for feature in [
    "HomeLeaguePosition",
    "AwayLeaguePosition",
    "PositionDifference",
    "HomeTop4Before",
    "AwayTop4Before",
    "HomeTop6Before",
    "AwayTop6Before",
    "HomeTopHalfBefore",
    "AwayTopHalfBefore",
    "HomeBottom3Before",
    "AwayBottom3Before",
]:
    set_opening_feature(
        feature,
        np.nan,
    )


# ============================================================
# F. VALIDATE THE OPENING FEATURE MATRIX
# ============================================================

assert opening_round_features.shape == (
    EXPECTED_OPENING_FIXTURES,
    EXPECTED_FEATURE_COUNT,
)


assert (
    opening_round_features.columns.tolist()
    == production_feature_columns
), (
    "Opening feature order does not match the frozen model schema."
)


opening_numeric_array = (
    opening_round_features
    .to_numpy(
        dtype=float
    )
)

assert not np.isinf(
    opening_numeric_array
).any(), (
    "Opening feature matrix contains infinite values."
)


# ------------------------------------------------------------
# Validate expected structural missingness
# ------------------------------------------------------------

opening_missing_counts = (
    opening_round_features
    .isna()
    .sum()
)

features_missing_for_all_openers = (
    opening_missing_counts.loc[
        opening_missing_counts
        == EXPECTED_OPENING_FIXTURES
    ]
    .index
    .tolist()
)


assert set(
    rolling_features
).issubset(
    set(
        features_missing_for_all_openers
    )
)


# ------------------------------------------------------------
# Feature identities
# ------------------------------------------------------------

assert np.allclose(
    opening_round_features[
        "EloDifference"
    ],
    (
        opening_round_features[
            "HomeEloBefore"
        ]
        - opening_round_features[
            "AwayEloBefore"
        ]
    ),
)

assert (
    opening_round_features[
        "PointsDifference"
    ]
    == 0
).all()

assert (
    opening_round_features[
        "GoalDifferenceDifference"
    ]
    == 0
).all()

assert (
    opening_round_features[
        "HomeMatchWeek"
    ]
    == 1
).all()

assert (
    opening_round_features[
        "AwayMatchWeek"
    ]
    == 1
).all()


# ============================================================
# G. COMBINE FIXTURE METADATA + FEATURES AND SAVE
# ============================================================

opening_round_feature_table = pd.concat(
    [
        opening_fixtures[
            [
                "FixtureNumber",
                "Season",
                "Kickoff",
                "OfficialHomeTeam",
                "OfficialAwayTeam",
                "HomeTeam",
                "AwayTeam",
            ]
        ]
        .reset_index(drop=True),

        opening_round_features
        .reset_index(drop=True),
    ],
    axis=1,
)


opening_round_feature_table.to_csv(
    opening_feature_output_path,
    index=False,
)

fixture_forecastability.to_csv(
    forecastability_output_path,
    index=False,
)


# ============================================================
# H. REPORT
# ============================================================

opening_feature_validation = pd.DataFrame(
    [
        {
            "Validation": "Opening fixtures",
            "Value": len(
                opening_round_feature_table
            ),
            "Expected": 10,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Teams represented exactly once"
            ),
            "Value": len(
                opening_teams
            ),
            "Expected": 20,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Frozen predictor count"
            ),
            "Value": (
                opening_round_features.shape[1]
            ),
            "Expected": 70,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Infinite predictor values"
            ),
            "Value": int(
                np.isinf(
                    opening_numeric_array
                ).sum()
            ),
            "Expected": 0,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Fixtures forecastable now"
            ),
            "Value": ready_fixture_count,
            "Expected": 10,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Fixtures requiring sequential update"
            ),
            "Value": (
                sequential_fixture_count
            ),
            "Expected": 370,
            "Status": "PASS",
        },
    ]
)


opening_feature_profile = pd.DataFrame(
    [
        {
            "Metric": (
                "Opening fixtures"
            ),
            "Value": 10,
        },
        {
            "Metric": (
                "Frozen predictors"
            ),
            "Value": 70,
        },
        {
            "Metric": (
                "Features structurally missing "
                "for all opening fixtures"
            ),
            "Value": len(
                features_missing_for_all_openers
            ),
        },
        {
            "Metric": (
                "Teams not previously seen "
                "in training Elo history"
            ),
            "Value": (
                never_seen_opening_teams
            ),
        },
        {
            "Metric": (
                "Exact forecasts available now"
            ),
            "Value": ready_fixture_count,
        },
        {
            "Metric": (
                "Sequential forecasts remaining"
            ),
            "Value": (
                sequential_fixture_count
            ),
        },
    ]
)


opening_elo_sample = (
    opening_round_feature_table[
        [
            "FixtureNumber",
            "Kickoff",
            "HomeTeam",
            "AwayTeam",
            "HomeEloBefore",
            "AwayEloBefore",
            "EloDifference",
        ]
    ]
)


opening_round_features_ready = True


print(
    "Opening-round feature profile:"
)
display(
    opening_feature_profile
)

print(
    "\nOpening-round feature validation:"
)
display(
    opening_feature_validation
)

print(
    "\nOpening-round Elo state:"
)
display(
    opening_elo_sample
)

print(
    "\nOpening feature missingness:"
)
display(
    pd.DataFrame(
        {
            "Feature": (
                opening_missing_counts.index
            ),
            "MissingFixtures": (
                opening_missing_counts.values
            ),
        }
    )
    .loc[
        lambda frame:
        frame["MissingFixtures"] > 0
    ]
    .sort_values(
        by=[
            "MissingFixtures",
            "Feature",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

print(
    "\nOpening feature file:",
    opening_feature_output_path
    .relative_to(project_root)
    .as_posix(),
)

print(
    "Forecastability audit:",
    forecastability_output_path
    .relative_to(project_root)
    .as_posix(),
)

print(
    "\nOpening-round feature status: VALID"
)

print(
    "Ready to generate opening-round probabilities:",
    opening_round_features_ready,
)

Opening-round feature profile:


,Metric,Value
0,Opening fixtures,10
1,Frozen predictors,70
2,Features structurally missing for all opening ...,47
3,Teams not previously seen in training Elo history,[Coventry]
4,Exact forecasts available now,10
5,Sequential forecasts remaining,370



Opening-round feature validation:


,Validation,Value,Expected,Status
0,Opening fixtures,10,10,PASS
1,Teams represented exactly once,20,20,PASS
2,Frozen predictor count,70,70,PASS
3,Infinite predictor values,0,0,PASS
4,Fixtures forecastable now,10,10,PASS
5,Fixtures requiring sequential update,370,370,PASS



Opening-round Elo state:


,FixtureNumber,Kickoff,HomeTeam,AwayTeam,HomeEloBefore,AwayEloBefore,EloDifference
0,1,2026-08-21 20:00:00,Arsenal,Coventry,1772.087827,1500.000000,272.087827
1,2,2026-08-22 12:30:00,Hull,Man United,1429.577787,1656.774984,-227.197197
2,3,2026-08-22 15:00:00,Everton,Crystal Palace,1551.038353,1547.084407,3.953945
3,4,2026-08-22 15:00:00,Ipswich,Sunderland,1409.616738,1512.942699,-103.325961
4,5,2026-08-22 15:00:00,Nott'm Forest,Leeds,1556.899594,1527.697284,29.202310
5,6,2026-08-22 17:30:00,Brentford,Tottenham,1581.873124,1497.149080,84.724044
6,7,2026-08-23 14:00:00,Brighton,Aston Villa,1590.290388,1644.547933,-54.257545
7,8,2026-08-23 14:00:00,Man City,Bournemouth,1751.686278,1622.871894,128.814384
8,9,2026-08-23 16:30:00,Newcastle,Liverpool,1574.763995,1662.759816,-87.995821
9,10,2026-08-24 20:00:00,Fulham,Chelsea,1561.568396,1588.278794,-26.710398



Opening feature missingness:


,Feature,MissingFixtures
0,AwayBottom3Before,10
1,AwayLeaguePosition,10
2,AwayRestDays,10
3,AwayRollingGoalDifference5,10
4,AwayRollingGoalsAgainst5,10
5,AwayRollingGoalsFor5,10
6,AwayRollingPoints5,10
7,AwayRollingWinRate5,10
8,AwayShortRest3,10
9,AwayTop4Before,10



Opening feature file: outputs/forecasts/2026_27/opening_round_features.csv
Forecastability audit: outputs/forecasts/2026_27/fixture_forecastability.csv

Opening-round feature status: VALID
Ready to generate opening-round probabilities: True


### Results and Interpretation

The opening-round pre-match feature state was constructed successfully for all ten fixtures.

All 20 Premier League clubs appear exactly once across the opening fixture set, and each fixture contains the complete frozen 70-column predictor schema required by the production model.

There are 47 predictors that are structurally missing for all opening fixtures. These correspond primarily to rolling form, venue form, first-match rest measures and league-position variables that do not yet exist before the first match of a new season. They are deliberately retained as missing values and will be handled by the median imputer fitted during the production refit.

Pre-season Elo ratings were carried forward from the complete historical sequence through the end of 2025–26. Coventry is the only 2026–27 club not represented previously in the Premier League training history and therefore receives the established initial Elo value of 1500.

Exactly ten fixtures currently have a fully defined genuine pre-match state. The remaining 370 require sequential updates as 2026–27 results become available, preventing future match outcomes from entering the forecasting features prematurely.

The opening-round feature matrix is therefore valid and ready for genuine out-of-sample probability generation.

## 4. Generate the First 2026–27 Production Forecasts

The validated opening-round feature matrix is now passed through the frozen production package.

No estimator is refitted and no feature values are altered using 2026–27 outcomes. The fitted median imputer and standardisation transformation are applied exactly as exported from Notebook 10, followed by the final home- and away-goal Poisson regressions.

For each fixture, the two models estimate expected home and away goals. Independent Poisson scoreline probabilities are then constructed over scores from 0–10, the captured probability mass is renormalised, and the scoreline distribution is aggregated into home-win, draw and away-win probabilities in the fixed order `(H, D, A)`.

These ten matches therefore represent the first genuinely unseen forecasts generated by the completed probability engine.

In [6]:
# ============================================================
# 4. Generate the First 2026-27 Production Forecasts
# ============================================================

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import poisson


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

assert "production_package_ready" in globals()
assert production_package_ready is True

assert "opening_round_features_ready" in globals()
assert opening_round_features_ready is True

assert len(production_feature_columns) == 70
assert len(production_home_feature_positions) == 55

assert production_home_model.n_features_in_ == 55
assert production_away_model.n_features_in_ == 70

assert opening_round_feature_table.shape[0] == 10


# ------------------------------------------------------------
# Output path
# ------------------------------------------------------------

project_root = Path.cwd().resolve()

while (
    project_root.name != "premier-league-probability-engine"
    and project_root.parent != project_root
):
    project_root = project_root.parent

assert project_root.name == "premier-league-probability-engine"


opening_forecast_output_path = (
    project_root
    / "outputs"
    / "forecasts"
    / "2026_27"
    / "opening_round_forecasts.csv"
)

opening_forecast_output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# Production prediction function
# ============================================================

def predict_with_frozen_production_model(
    fixture_features: pd.DataFrame,
) -> pd.DataFrame:

    if not isinstance(
        fixture_features,
        pd.DataFrame,
    ):
        raise TypeError(
            "fixture_features must be a pandas DataFrame."
        )

    if fixture_features.empty:
        raise ValueError(
            "No fixture rows were supplied."
        )


    # --------------------------------------------------------
    # Frozen feature schema
    # --------------------------------------------------------

    missing_features = [
        feature
        for feature in production_feature_columns
        if feature not in fixture_features.columns
    ]

    if missing_features:
        raise ValueError(
            "Missing frozen production features: "
            f"{missing_features}"
        )


    X_raw = (
        fixture_features[
            production_feature_columns
        ]
        .copy()
    )


    non_numeric_features = [
        feature
        for feature in production_feature_columns
        if not pd.api.types.is_numeric_dtype(
            X_raw[feature]
        )
    ]

    if non_numeric_features:
        raise TypeError(
            "Non-numeric production predictors detected: "
            f"{non_numeric_features}"
        )


    X_raw_array = (
        X_raw
        .astype(float)
        .to_numpy()
    )

    if np.isinf(
        X_raw_array
    ).any():
        raise ValueError(
            "Infinite predictor values were detected."
        )


    # --------------------------------------------------------
    # Frozen preprocessing
    # --------------------------------------------------------

    X_imputed = production_imputer.transform(
        X_raw
    )

    X_scaled = production_scaler.transform(
        X_imputed
    )


    assert X_scaled.shape == (
        len(fixture_features),
        70,
    )

    assert np.isfinite(
        X_scaled
    ).all()


    # --------------------------------------------------------
    # Model-specific numerical bases
    # --------------------------------------------------------

    X_home = X_scaled[
        :,
        production_home_feature_positions,
    ]

    X_away = X_scaled


    assert X_home.shape[1] == 55
    assert X_away.shape[1] == 70


    # --------------------------------------------------------
    # Expected goals
    # --------------------------------------------------------

    home_expected_goals = np.clip(
        production_home_model.predict(
            X_home
        ),
        MINIMUM_EXPECTED_GOALS,
        None,
    )

    away_expected_goals = np.clip(
        production_away_model.predict(
            X_away
        ),
        MINIMUM_EXPECTED_GOALS,
        None,
    )


    assert np.isfinite(
        home_expected_goals
    ).all()

    assert np.isfinite(
        away_expected_goals
    ).all()

    assert (
        home_expected_goals > 0
    ).all()

    assert (
        away_expected_goals > 0
    ).all()


    # --------------------------------------------------------
    # Independent Poisson scoreline distributions
    # --------------------------------------------------------

    goal_grid = np.arange(
        MAX_MODELLED_GOALS + 1,
        dtype=int,
    )


    home_goal_probabilities = poisson.pmf(
        goal_grid[None, :],
        home_expected_goals[:, None],
    )

    away_goal_probabilities = poisson.pmf(
        goal_grid[None, :],
        away_expected_goals[:, None],
    )


    scoreline_probabilities = (
        home_goal_probabilities[
            :,
            :,
            None,
        ]
        *
        away_goal_probabilities[
            :,
            None,
            :,
        ]
    )


    # --------------------------------------------------------
    # Renormalise truncated 0-10 scoreline mass
    # --------------------------------------------------------

    captured_mass = (
        scoreline_probabilities.sum(
            axis=(1, 2)
        )
    )

    assert np.isfinite(
        captured_mass
    ).all()

    assert (
        captured_mass > 0
    ).all()


    scoreline_probabilities = (
        scoreline_probabilities
        /
        captured_mass[
            :,
            None,
            None,
        ]
    )


    # --------------------------------------------------------
    # H / D / A probabilities
    #
    # Rows    = home goals
    # Columns = away goals
    # --------------------------------------------------------

    probability_home = np.array(
        [
            np.tril(
                matrix,
                k=-1,
            ).sum()
            for matrix
            in scoreline_probabilities
        ]
    )

    probability_draw = np.array(
        [
            np.trace(
                matrix
            )
            for matrix
            in scoreline_probabilities
        ]
    )

    probability_away = np.array(
        [
            np.triu(
                matrix,
                k=1,
            ).sum()
            for matrix
            in scoreline_probabilities
        ]
    )


    # --------------------------------------------------------
    # Most likely exact scoreline
    # --------------------------------------------------------

    modal_home_goals = []
    modal_away_goals = []
    modal_scoreline_probability = []

    for matrix in scoreline_probabilities:

        maximum_index = np.unravel_index(
            np.argmax(matrix),
            matrix.shape,
        )

        modal_home_goals.append(
            int(
                maximum_index[0]
            )
        )

        modal_away_goals.append(
            int(
                maximum_index[1]
            )
        )

        modal_scoreline_probability.append(
            float(
                matrix[
                    maximum_index
                ]
            )
        )


    # --------------------------------------------------------
    # Return forecast table
    # --------------------------------------------------------

    predictions = pd.DataFrame(
        {
            "HomeExpectedGoals": (
                home_expected_goals
            ),
            "AwayExpectedGoals": (
                away_expected_goals
            ),
            "Probability_H": (
                probability_home
            ),
            "Probability_D": (
                probability_draw
            ),
            "Probability_A": (
                probability_away
            ),
            "ModalHomeGoals": (
                modal_home_goals
            ),
            "ModalAwayGoals": (
                modal_away_goals
            ),
            "ModalScoreProbability": (
                modal_scoreline_probability
            ),
        },
        index=fixture_features.index,
    )


    # --------------------------------------------------------
    # Probability validation
    # --------------------------------------------------------

    probability_values = predictions[
        [
            "Probability_H",
            "Probability_D",
            "Probability_A",
        ]
    ].to_numpy()


    probability_sums = (
        probability_values.sum(
            axis=1
        )
    )


    assert np.allclose(
        probability_sums,
        1.0,
        atol=1e-10,
        rtol=0.0,
    )

    assert (
        probability_values >= 0
    ).all()

    assert (
        probability_values <= 1
    ).all()


    return predictions


# ============================================================
# Generate opening-round forecasts
# ============================================================

opening_predictions = (
    predict_with_frozen_production_model(
        opening_round_feature_table
    )
    .reset_index(drop=True)
)


assert opening_predictions.shape[0] == 10


# ------------------------------------------------------------
# Combine fixture identity and forecasts
# ------------------------------------------------------------

opening_round_forecasts = pd.concat(
    [
        opening_round_feature_table[
            [
                "FixtureNumber",
                "Season",
                "Kickoff",
                "OfficialHomeTeam",
                "OfficialAwayTeam",
                "HomeTeam",
                "AwayTeam",
            ]
        ]
        .reset_index(drop=True),

        opening_predictions,
    ],
    axis=1,
)


# ------------------------------------------------------------
# Most likely H / D / A outcome
# ------------------------------------------------------------

probability_matrix = (
    opening_round_forecasts[
        [
            "Probability_H",
            "Probability_D",
            "Probability_A",
        ]
    ]
    .to_numpy()
)


outcome_codes = np.array(
    [
        "H",
        "D",
        "A",
    ]
)

most_likely_indices = np.argmax(
    probability_matrix,
    axis=1,
)

opening_round_forecasts[
    "MostLikelyOutcome"
] = outcome_codes[
    most_likely_indices
]


opening_round_forecasts[
    "MostLikelyOutcomeProbability"
] = np.max(
    probability_matrix,
    axis=1,
)


def readable_prediction(
    row,
):

    if row[
        "MostLikelyOutcome"
    ] == "H":

        return row[
            "OfficialHomeTeam"
        ]

    if row[
        "MostLikelyOutcome"
    ] == "A":

        return row[
            "OfficialAwayTeam"
        ]

    return "Draw"


opening_round_forecasts[
    "ModelFavourite"
] = opening_round_forecasts.apply(
    readable_prediction,
    axis=1,
)


opening_round_forecasts[
    "ModalScoreline"
] = (
    opening_round_forecasts[
        "ModalHomeGoals"
    ]
    .astype(str)
    + "-"
    + opening_round_forecasts[
        "ModalAwayGoals"
    ]
    .astype(str)
)


# ============================================================
# Validate final forecast table
# ============================================================

forecast_probability_sums = (
    opening_round_forecasts[
        [
            "Probability_H",
            "Probability_D",
            "Probability_A",
        ]
    ]
    .sum(axis=1)
)


maximum_probability_sum_error = float(
    np.max(
        np.abs(
            forecast_probability_sums
            - 1.0
        )
    )
)


minimum_probability = float(
    opening_round_forecasts[
        [
            "Probability_H",
            "Probability_D",
            "Probability_A",
        ]
    ]
    .min()
    .min()
)


maximum_probability = float(
    opening_round_forecasts[
        [
            "Probability_H",
            "Probability_D",
            "Probability_A",
        ]
    ]
    .max()
    .max()
)


assert len(
    opening_round_forecasts
) == 10

assert maximum_probability_sum_error < 1e-10

assert minimum_probability >= 0.0

assert maximum_probability <= 1.0

assert opening_round_forecasts[
    [
        "HomeExpectedGoals",
        "AwayExpectedGoals",
    ]
].gt(0).all().all()


# ------------------------------------------------------------
# Save genuine unseen forecasts
# ------------------------------------------------------------

opening_round_forecasts.to_csv(
    opening_forecast_output_path,
    index=False,
)


# ============================================================
# Reporting tables
# ============================================================

forecast_validation_table = pd.DataFrame(
    [
        {
            "Validation": (
                "Opening forecasts"
            ),
            "Value": len(
                opening_round_forecasts
            ),
            "Expected": 10,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Finite expected goals"
            ),
            "Value": bool(
                np.isfinite(
                    opening_round_forecasts[
                        [
                            "HomeExpectedGoals",
                            "AwayExpectedGoals",
                        ]
                    ].to_numpy()
                ).all()
            ),
            "Expected": True,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Probability sum max error"
            ),
            "Value": (
                maximum_probability_sum_error
            ),
            "Expected": "< 1e-10",
            "Status": "PASS",
        },
        {
            "Validation": (
                "Minimum probability"
            ),
            "Value": (
                minimum_probability
            ),
            "Expected": ">= 0",
            "Status": "PASS",
        },
        {
            "Validation": (
                "Maximum probability"
            ),
            "Value": (
                maximum_probability
            ),
            "Expected": "<= 1",
            "Status": "PASS",
        },
        {
            "Validation": (
                "Probability order"
            ),
            "Value": (
                PROBABILITY_ORDER
            ),
            "Expected": [
                "H",
                "D",
                "A",
            ],
            "Status": "PASS",
        },
    ]
)


forecast_display = (
    opening_round_forecasts[
        [
            "Kickoff",
            "OfficialHomeTeam",
            "OfficialAwayTeam",
            "HomeExpectedGoals",
            "AwayExpectedGoals",
            "Probability_H",
            "Probability_D",
            "Probability_A",
            "ModelFavourite",
            "MostLikelyOutcomeProbability",
            "ModalScoreline",
            "ModalScoreProbability",
        ]
    ]
    .copy()
)


for column in [
    "HomeExpectedGoals",
    "AwayExpectedGoals",
    "Probability_H",
    "Probability_D",
    "Probability_A",
    "MostLikelyOutcomeProbability",
    "ModalScoreProbability",
]:

    forecast_display[
        column
    ] = forecast_display[
        column
    ].round(4)


opening_round_forecasts_ready = True


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print(
    "Opening-round production forecasts:"
)
display(
    forecast_display
)

print(
    "\nForecast validation:"
)
display(
    forecast_validation_table
)

print(
    "\nForecast output file:",
    opening_forecast_output_path
    .relative_to(
        project_root
    )
    .as_posix(),
)

print(
    "\n2026-27 opening-round forecast status: VALID"
)

print(
    "First genuine unseen forecasts generated:",
    opening_round_forecasts_ready,
)

Opening-round production forecasts:


,Kickoff,OfficialHomeTeam,OfficialAwayTeam,HomeExpectedGoals,AwayExpectedGoals,Probability_H,Probability_D,Probability_A,ModelFavourite,MostLikelyOutcomeProbability,ModalScoreline,ModalScoreProbability
0,2026-08-21 20:00:00,Arsenal,Coventry City,2.4699,0.9510,0.7061,0.1682,0.1257,Arsenal,0.7061,2-0,0.0997
1,2026-08-22 12:30:00,Hull City,Manchester United,1.0102,2.0448,0.1794,0.2079,0.6127,Manchester United,0.6127,1-2,0.0995
2,2026-08-22 15:00:00,Everton,Crystal Palace,1.4887,1.4128,0.3932,0.2475,0.3593,Everton,0.3932,1-1,0.1156
3,2026-08-22 15:00:00,Ipswich Town,Sunderland,1.1712,1.6194,0.2760,0.2461,0.4779,Sunderland,0.4779,1-1,0.1164
4,2026-08-22 15:00:00,Nottingham Forest,Leeds United,1.5451,1.3528,0.4196,0.2466,0.3338,Nottingham Forest,0.4196,1-1,0.1153
5,2026-08-22 17:30:00,Brentford,Tottenham Hotspur,1.6909,1.2357,0.4806,0.2395,0.2798,Brentford,0.4806,1-1,0.1120
6,2026-08-23 14:00:00,Brighton & Hove Albion,Aston Villa,1.4183,1.5960,0.3404,0.2413,0.4183,Aston Villa,0.4183,1-1,0.1111
7,2026-08-23 14:00:00,Manchester City,AFC Bournemouth,2.0188,1.2234,0.5578,0.2143,0.2278,Manchester City,0.5578,2-1,0.0974
8,2026-08-23 16:30:00,Newcastle United,Liverpool,1.3424,1.6861,0.3061,0.2379,0.4560,Liverpool,0.4560,1-1,0.1095
9,2026-08-24 20:00:00,Fulham,Chelsea,1.4409,1.5003,0.3640,0.2456,0.3903,Chelsea,0.3903,1-1,0.1141



Forecast validation:


,Validation,Value,Expected,Status
0,Opening forecasts,10,10,PASS
1,Finite expected goals,True,True,PASS
2,Probability sum max error,0.0,< 1e-10,PASS
3,Minimum probability,0.125668,>= 0,PASS
4,Maximum probability,0.706123,<= 1,PASS
5,Probability order,"[H, D, A]","[H, D, A]",PASS



Forecast output file: outputs/forecasts/2026_27/opening_round_forecasts.csv

2026-27 opening-round forecast status: VALID
First genuine unseen forecasts generated: True


### Results and Interpretation

The frozen production model generated valid forecasts for all ten opening-round 2026–27 Premier League fixtures.

All expected-goal estimates are finite and positive, every `(H, D, A)` probability vector sums to one, and all probabilities remain inside the valid interval. The maximum forecast probability is approximately $0.7061$ and the minimum is approximately $0.1257$.

The strongest opening-round model preference is Arsenal at home to Coventry City. Arsenal are assigned approximately a $70.6\%$ home-win probability, with expected goals of $2.47$ versus $0.95$ for Coventry.

Manchester United are also clear favourites away at Hull City, with an estimated $61.3\%$ away-win probability. Manchester City receive a $55.8\%$ home-win probability against Bournemouth.

Several fixtures are substantially less certain. Everton–Crystal Palace is close to balanced, with probabilities of approximately $39.3\%$, $24.8\%$ and $35.9\%$ for home win, draw and away win respectively. Fulham–Chelsea is similarly competitive, with Chelsea only narrowly preferred at approximately $39.0\%$ compared with $36.4\%$ for Fulham.

The remaining model favourites are Sunderland away at Ipswich ($47.8\%$), Nottingham Forest against Leeds ($42.0\%$), Brentford against Tottenham ($48.1\%$), Aston Villa away at Brighton ($41.8\%$), and Liverpool away at Newcastle ($45.6\%$).

These probabilities should be interpreted as model forecasts rather than deterministic predictions. In particular, a team being the most likely individual outcome does not imply that outcome is more likely than all alternatives combined.

The forecasts have been exported to `outputs/forecasts/2026_27/opening_round_forecasts.csv` and constitute the first genuinely unseen predictions produced by the completed probability engine.

## 5. Assess Opening-Round Forecast Confidence

Selecting the largest of the three outcome probabilities does not by itself describe how decisive a forecast is.

A fixture where the leading outcome has probability $0.40$ may be substantially more uncertain than one where the leading outcome has probability $0.70$, even though both produce a single model favourite.

This section therefore supplements the opening-round probabilities with several descriptive uncertainty measures.

For each fixture, the analysis calculates:

- the probability assigned to the most likely outcome;
- the difference between the largest and second-largest outcome probabilities;
- predictive entropy across the `(H, D, A)` distribution;
- expected total goals;
- expected home-minus-away goal difference.

Entropy is defined as

$$
H(p)=-\sum_{i \in \{H,D,A\}}p_i\log(p_i),
$$

with larger values representing more diffuse probability distributions.

These quantities are descriptive diagnostics only. They do not alter the frozen production forecasts or introduce a new decision rule.

In [7]:
# ============================================================
# 5. Assess Opening-Round Forecast Confidence
# ============================================================

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

assert "opening_round_forecasts_ready" in globals()
assert opening_round_forecasts_ready is True

assert len(opening_round_forecasts) == 10


probability_columns = [
    "Probability_H",
    "Probability_D",
    "Probability_A",
]


# ------------------------------------------------------------
# Probability matrix
# ------------------------------------------------------------

probabilities = (
    opening_round_forecasts[
        probability_columns
    ]
    .to_numpy(
        dtype=float
    )
)

assert probabilities.shape == (10, 3)

assert np.isfinite(
    probabilities
).all()

assert np.allclose(
    probabilities.sum(axis=1),
    1.0,
    atol=1e-10,
    rtol=0.0,
)


# ------------------------------------------------------------
# Leading and second-largest probability
# ------------------------------------------------------------

sorted_probabilities = np.sort(
    probabilities,
    axis=1,
)

favourite_probability = (
    sorted_probabilities[:, -1]
)

second_probability = (
    sorted_probabilities[:, -2]
)

favourite_margin = (
    favourite_probability
    - second_probability
)


# ------------------------------------------------------------
# Predictive entropy
# ------------------------------------------------------------

predictive_entropy = -np.sum(
    probabilities
    * np.log(
        probabilities
    ),
    axis=1,
)

maximum_entropy = float(
    np.log(3.0)
)

normalised_entropy = (
    predictive_entropy
    / maximum_entropy
)


assert (
    predictive_entropy >= 0
).all()

assert (
    normalised_entropy >= 0
).all()

assert (
    normalised_entropy <= 1
).all()


# ------------------------------------------------------------
# Expected-goal diagnostics
# ------------------------------------------------------------

expected_total_goals = (
    opening_round_forecasts[
        "HomeExpectedGoals"
    ].to_numpy()
    +
    opening_round_forecasts[
        "AwayExpectedGoals"
    ].to_numpy()
)

expected_goal_difference = (
    opening_round_forecasts[
        "HomeExpectedGoals"
    ].to_numpy()
    -
    opening_round_forecasts[
        "AwayExpectedGoals"
    ].to_numpy()
)


# ------------------------------------------------------------
# Descriptive confidence labels
#
# These are reporting categories only, not modelling rules.
# ------------------------------------------------------------

def confidence_label(
    favourite_probability,
    favourite_margin,
):

    if (
        favourite_probability >= 0.60
        and favourite_margin >= 0.25
    ):
        return "Strong"

    if (
        favourite_probability >= 0.45
        and favourite_margin >= 0.12
    ):
        return "Moderate"

    return "Low"


confidence_labels = [
    confidence_label(
        probability,
        margin,
    )
    for probability, margin
    in zip(
        favourite_probability,
        favourite_margin,
    )
]


# ------------------------------------------------------------
# Build diagnostic table
# ------------------------------------------------------------

opening_forecast_diagnostics = (
    opening_round_forecasts[
        [
            "Kickoff",
            "OfficialHomeTeam",
            "OfficialAwayTeam",
            "ModelFavourite",
            "Probability_H",
            "Probability_D",
            "Probability_A",
            "HomeExpectedGoals",
            "AwayExpectedGoals",
        ]
    ]
    .copy()
)


opening_forecast_diagnostics[
    "FavouriteProbability"
] = favourite_probability

opening_forecast_diagnostics[
    "SecondHighestProbability"
] = second_probability

opening_forecast_diagnostics[
    "FavouriteProbabilityMargin"
] = favourite_margin

opening_forecast_diagnostics[
    "PredictiveEntropy"
] = predictive_entropy

opening_forecast_diagnostics[
    "NormalisedEntropy"
] = normalised_entropy

opening_forecast_diagnostics[
    "ExpectedTotalGoals"
] = expected_total_goals

opening_forecast_diagnostics[
    "ExpectedGoalDifference"
] = expected_goal_difference

opening_forecast_diagnostics[
    "Confidence"
] = confidence_labels


# ------------------------------------------------------------
# Ranking tables
# ------------------------------------------------------------

most_decisive_forecasts = (
    opening_forecast_diagnostics
    .sort_values(
        by=[
            "FavouriteProbabilityMargin",
            "FavouriteProbability",
        ],
        ascending=False,
        kind="mergesort",
    )
    .reset_index(drop=True)
)


most_uncertain_forecasts = (
    opening_forecast_diagnostics
    .sort_values(
        by=[
            "NormalisedEntropy",
            "FavouriteProbabilityMargin",
        ],
        ascending=[
            False,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

confidence_summary = (
    opening_forecast_diagnostics[
        "Confidence"
    ]
    .value_counts()
    .reindex(
        [
            "Strong",
            "Moderate",
            "Low",
        ],
        fill_value=0,
    )
    .rename_axis(
        "Confidence"
    )
    .reset_index(
        name="Fixtures"
    )
)


diagnostic_summary = pd.DataFrame(
    [
        {
            "Metric": (
                "Mean favourite probability"
            ),
            "Value": float(
                favourite_probability.mean()
            ),
        },
        {
            "Metric": (
                "Largest favourite probability"
            ),
            "Value": float(
                favourite_probability.max()
            ),
        },
        {
            "Metric": (
                "Smallest favourite probability"
            ),
            "Value": float(
                favourite_probability.min()
            ),
        },
        {
            "Metric": (
                "Mean favourite margin"
            ),
            "Value": float(
                favourite_margin.mean()
            ),
        },
        {
            "Metric": (
                "Mean normalised entropy"
            ),
            "Value": float(
                normalised_entropy.mean()
            ),
        },
        {
            "Metric": (
                "Mean expected total goals"
            ),
            "Value": float(
                expected_total_goals.mean()
            ),
        },
    ]
)


# ------------------------------------------------------------
# Export
# ------------------------------------------------------------

diagnostic_output_path = (
    project_root
    / "outputs"
    / "forecasts"
    / "2026_27"
    / "opening_round_forecast_diagnostics.csv"
)

opening_forecast_diagnostics.to_csv(
    diagnostic_output_path,
    index=False,
)


# ------------------------------------------------------------
# Display formatting
# ------------------------------------------------------------

display_columns = [
    "OfficialHomeTeam",
    "OfficialAwayTeam",
    "ModelFavourite",
    "FavouriteProbability",
    "SecondHighestProbability",
    "FavouriteProbabilityMargin",
    "NormalisedEntropy",
    "ExpectedTotalGoals",
    "ExpectedGoalDifference",
    "Confidence",
]


decisive_display = (
    most_decisive_forecasts[
        display_columns
    ]
    .copy()
)

uncertain_display = (
    most_uncertain_forecasts[
        display_columns
    ]
    .copy()
)


numeric_display_columns = [
    "FavouriteProbability",
    "SecondHighestProbability",
    "FavouriteProbabilityMargin",
    "NormalisedEntropy",
    "ExpectedTotalGoals",
    "ExpectedGoalDifference",
]


for column in numeric_display_columns:

    decisive_display[
        column
    ] = decisive_display[
        column
    ].round(4)

    uncertain_display[
        column
    ] = uncertain_display[
        column
    ].round(4)


opening_forecast_diagnostics_ready = True


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print(
    "Opening-round confidence summary:"
)
display(
    confidence_summary
)

print(
    "\nForecast diagnostic summary:"
)
display(
    diagnostic_summary
)

print(
    "\nMost decisive forecasts:"
)
display(
    decisive_display
)

print(
    "\nMost uncertain forecasts:"
)
display(
    uncertain_display
)

print(
    "\nDiagnostic output file:",
    diagnostic_output_path
    .relative_to(
        project_root
    )
    .as_posix(),
)

print(
    "\nOpening-round diagnostic status: VALID"
)

print(
    "Ready for market comparison:",
    opening_forecast_diagnostics_ready,
)

Opening-round confidence summary:


,Confidence,Fixtures
0,Strong,2
1,Moderate,4
2,Low,4



Forecast diagnostic summary:


,Metric,Value
0,Mean favourite probability,0.491258
1,Largest favourite probability,0.706123
2,Smallest favourite probability,0.390348
3,Mean favourite margin,0.204919
4,Mean normalised entropy,0.929412
5,Mean expected total goals,3.021868



Most decisive forecasts:


,OfficialHomeTeam,OfficialAwayTeam,ModelFavourite,FavouriteProbability,SecondHighestProbability,FavouriteProbabilityMargin,NormalisedEntropy,ExpectedTotalGoals,ExpectedGoalDifference,Confidence
0,Arsenal,Coventry City,Arsenal,0.7061,0.1682,0.5379,0.7338,3.4209,1.5188,Strong
1,Hull City,Manchester United,Manchester United,0.6127,0.2079,0.4048,0.8510,3.0551,-1.0346,Strong
2,Manchester City,AFC Bournemouth,Manchester City,0.5578,0.2278,0.3300,0.9036,3.2422,0.7954,Moderate
3,Ipswich Town,Sunderland,Sunderland,0.4779,0.2760,0.2019,0.9587,2.7907,-0.4482,Moderate
4,Brentford,Tottenham Hotspur,Brentford,0.4806,0.2798,0.2008,0.9565,2.9265,0.4552,Moderate
5,Newcastle United,Liverpool,Liverpool,0.4560,0.3061,0.1498,0.9667,3.0284,-0.3437,Moderate
6,Nottingham Forest,Leeds United,Nottingham Forest,0.4196,0.3338,0.0858,0.9793,2.8978,0.1923,Low
7,Brighton & Hove Albion,Aston Villa,Aston Villa,0.4183,0.3404,0.0779,0.9780,3.0144,-0.1777,Low
8,Everton,Crystal Palace,Everton,0.3932,0.3593,0.0339,0.9834,2.9016,0.0759,Low
9,Fulham,Chelsea,Chelsea,0.3903,0.3640,0.0263,0.9830,2.9411,-0.0594,Low



Most uncertain forecasts:


,OfficialHomeTeam,OfficialAwayTeam,ModelFavourite,FavouriteProbability,SecondHighestProbability,FavouriteProbabilityMargin,NormalisedEntropy,ExpectedTotalGoals,ExpectedGoalDifference,Confidence
0,Everton,Crystal Palace,Everton,0.3932,0.3593,0.0339,0.9834,2.9016,0.0759,Low
1,Fulham,Chelsea,Chelsea,0.3903,0.3640,0.0263,0.9830,2.9411,-0.0594,Low
2,Nottingham Forest,Leeds United,Nottingham Forest,0.4196,0.3338,0.0858,0.9793,2.8978,0.1923,Low
3,Brighton & Hove Albion,Aston Villa,Aston Villa,0.4183,0.3404,0.0779,0.9780,3.0144,-0.1777,Low
4,Newcastle United,Liverpool,Liverpool,0.4560,0.3061,0.1498,0.9667,3.0284,-0.3437,Moderate
5,Ipswich Town,Sunderland,Sunderland,0.4779,0.2760,0.2019,0.9587,2.7907,-0.4482,Moderate
6,Brentford,Tottenham Hotspur,Brentford,0.4806,0.2798,0.2008,0.9565,2.9265,0.4552,Moderate
7,Manchester City,AFC Bournemouth,Manchester City,0.5578,0.2278,0.3300,0.9036,3.2422,0.7954,Moderate
8,Hull City,Manchester United,Manchester United,0.6127,0.2079,0.4048,0.8510,3.0551,-1.0346,Strong
9,Arsenal,Coventry City,Arsenal,0.7061,0.1682,0.5379,0.7338,3.4209,1.5188,Strong



Diagnostic output file: outputs/forecasts/2026_27/opening_round_forecast_diagnostics.csv

Opening-round diagnostic status: VALID
Ready for market comparison: True


### Results and Interpretation

The opening-round forecasts contain a mixture of decisive and highly uncertain probability distributions.

Two fixtures are classified as `Strong`, four as `Moderate` and four as `Low` confidence under the descriptive reporting thresholds used in this section.

Across the ten fixtures, the mean probability assigned to the model favourite is approximately $49.13\%$. The largest favourite probability is Arsenal's $70.61\%$ against Coventry, while the smallest is Chelsea's $39.03\%$ away at Fulham.

The mean gap between the most likely and second-most likely outcomes is approximately $20.49$ percentage points, although this varies substantially by fixture. Arsenal–Coventry is the most decisive forecast, with a margin of approximately $53.79$ percentage points. Hull–Manchester United follows at approximately $40.48$ percentage points, while Manchester City–Bournemouth has a margin of approximately $33.00$ percentage points.

At the opposite end, Fulham–Chelsea has the smallest favourite margin at only approximately $2.63$ percentage points. Everton–Crystal Palace is similarly close at approximately $3.39$ percentage points and also produces the highest predictive entropy in the opening-round set. These fixtures should therefore be regarded as materially more uncertain than the headline model-favourite label alone suggests.

Mean normalised predictive entropy is approximately $0.9294$, indicating that most opening-round forecasts retain substantial probability across multiple possible outcomes rather than collapsing toward a single result.

The model expects approximately $3.02$ total goals per fixture on average across the opening round.

These diagnostics do not alter the underlying probabilities. They provide additional information about how concentrated or uncertain each forecast is before the predictions are compared with the betting market.

## 6. Compare Opening-Round Forecasts with Current Market Probabilities

The opening-round model probabilities are now compared with an external pre-match 1X2 betting-market snapshot collected on 9 August 2026.

The quoted prices represent currently available best odds rather than closing Pinnacle prices. They are therefore not directly equivalent to the historical closing-market benchmark used earlier in the project.

For each fixture, decimal odds are converted into raw implied probabilities using

$$
q_i=\frac{1}{o_i}.
$$

Because the quoted probabilities contain a bookmaker margin, they are normalised to obtain a simple margin-adjusted market distribution:

$$
p_i^{market}
=
\frac{q_i}
{\sum_j q_j}.
$$

Model-market disagreement is then measured separately for the home, draw and away outcomes as

$$
\Delta_i
=
p_i^{model}
-
p_i^{market}.
$$

Positive values indicate that the production model assigns greater probability to an outcome than the current market does, while negative values indicate that the model assigns less probability.

Total variation distance is also reported as

$$
TV
=
\frac{1}{2}
\sum_i
\left|
p_i^{model}-p_i^{market}
\right|,
$$

providing a single descriptive measure of disagreement for each fixture.

This section measures probability disagreement only. It does not establish a profitable betting edge, particularly because these are early market prices rather than closing odds and the historical analysis previously found no demonstrated persistent market-beating advantage.

In [8]:
# ============================================================
# 6. Compare Opening-Round Forecasts with Current Market
# ============================================================

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

assert "opening_round_forecasts_ready" in globals()
assert opening_round_forecasts_ready is True

assert len(opening_round_forecasts) == 10


# ------------------------------------------------------------
# Fixed market snapshot
#
# Snapshot date: 2026-08-09
# Source: Oddschecker 1X2 best-odds listings.
#
# Decimal odds are stored directly so that this notebook
# reproduces the exact market snapshot analysed here even
# if live prices subsequently move.
# ------------------------------------------------------------

MARKET_SNAPSHOT_DATE = "2026-08-09"

market_snapshot = pd.DataFrame(
    [
        {
            "OfficialHomeTeam": "Arsenal",
            "OfficialAwayTeam": "Coventry City",
            "MarketOdds_H": 1.1666666667,
            "MarketOdds_D": 7.50,
            "MarketOdds_A": 19.00,
            "MarketSource": "Oddschecker best odds",
        },
        {
            "OfficialHomeTeam": "Hull City",
            "OfficialAwayTeam": "Manchester United",
            "MarketOdds_H": 7.40,
            "MarketOdds_D": 4.60,
            "MarketOdds_A": 1.47,
            "MarketSource": "Oddschecker best odds",
        },
        {
            "OfficialHomeTeam": "Everton",
            "OfficialAwayTeam": "Crystal Palace",
            "MarketOdds_H": 2.15,
            "MarketOdds_D": 3.60,
            "MarketOdds_A": 3.40,
            "MarketSource": "Oddschecker best odds",
        },
        {
            "OfficialHomeTeam": "Ipswich Town",
            "OfficialAwayTeam": "Sunderland",
            "MarketOdds_H": 2.75,
            "MarketOdds_D": 3.40,
            "MarketOdds_A": 2.60,
            "MarketSource": "Oddschecker best odds",
        },
        {
            "OfficialHomeTeam": "Nottingham Forest",
            "OfficialAwayTeam": "Leeds United",
            "MarketOdds_H": 2.25,
            "MarketOdds_D": 3.60,
            "MarketOdds_A": 3.20,
            "MarketSource": "Oddschecker best odds",
        },
        {
            "OfficialHomeTeam": "Brentford",
            "OfficialAwayTeam": "Tottenham Hotspur",
            "MarketOdds_H": 2.35,
            "MarketOdds_D": 3.60,
            "MarketOdds_A": 2.90,
            "MarketSource": "Oddschecker best odds",
        },
        {
            "OfficialHomeTeam": "Brighton & Hove Albion",
            "OfficialAwayTeam": "Aston Villa",
            "MarketOdds_H": 2.30,
            "MarketOdds_D": 3.80,
            "MarketOdds_A": 3.00,
            "MarketSource": "Oddschecker best odds",
        },
        {
            "OfficialHomeTeam": "Manchester City",
            "OfficialAwayTeam": "AFC Bournemouth",
            "MarketOdds_H": 1.4444444444,
            "MarketOdds_D": 5.25,
            "MarketOdds_A": 6.50,
            "MarketSource": "Oddschecker best odds",
        },
        {
            "OfficialHomeTeam": "Newcastle United",
            "OfficialAwayTeam": "Liverpool",
            "MarketOdds_H": 3.45,
            "MarketOdds_D": 4.00,
            "MarketOdds_A": 2.10,
            "MarketSource": "Oddschecker best odds",
        },
        {
            "OfficialHomeTeam": "Fulham",
            "OfficialAwayTeam": "Chelsea",
            "MarketOdds_H": 3.20,
            "MarketOdds_D": 3.80,
            "MarketOdds_A": 2.20,
            "MarketSource": "Oddschecker best odds",
        },
    ]
)


# ------------------------------------------------------------
# Validate snapshot
# ------------------------------------------------------------

assert len(market_snapshot) == 10

assert not market_snapshot.duplicated(
    subset=[
        "OfficialHomeTeam",
        "OfficialAwayTeam",
    ]
).any()

odds_columns = [
    "MarketOdds_H",
    "MarketOdds_D",
    "MarketOdds_A",
]

assert (
    market_snapshot[
        odds_columns
    ] > 1.0
).all().all()

assert np.isfinite(
    market_snapshot[
        odds_columns
    ].to_numpy()
).all()


# ------------------------------------------------------------
# Merge with frozen model forecasts
# ------------------------------------------------------------

market_comparison = (
    opening_round_forecasts
    .merge(
        market_snapshot,
        on=[
            "OfficialHomeTeam",
            "OfficialAwayTeam",
        ],
        how="left",
        validate="one_to_one",
    )
)


assert len(market_comparison) == 10

assert not market_comparison[
    odds_columns
].isna().any().any(), (
    "At least one opening fixture did not match the "
    "market snapshot."
)


market_comparison[
    "MarketSnapshotDate"
] = MARKET_SNAPSHOT_DATE


# ============================================================
# A. RAW IMPLIED MARKET PROBABILITIES
# ============================================================

market_comparison[
    "RawMarketProbability_H"
] = (
    1.0
    / market_comparison[
        "MarketOdds_H"
    ]
)

market_comparison[
    "RawMarketProbability_D"
] = (
    1.0
    / market_comparison[
        "MarketOdds_D"
    ]
)

market_comparison[
    "RawMarketProbability_A"
] = (
    1.0
    / market_comparison[
        "MarketOdds_A"
    ]
)


raw_market_probability_columns = [
    "RawMarketProbability_H",
    "RawMarketProbability_D",
    "RawMarketProbability_A",
]


market_comparison[
    "MarketOverround"
] = (
    market_comparison[
        raw_market_probability_columns
    ]
    .sum(axis=1)
    - 1.0
)


assert (
    market_comparison[
        "MarketOverround"
    ] > 0
).all(), (
    "Unexpected non-positive market overround detected."
)


# ============================================================
# B. REMOVE THE MARKET MARGIN
# ============================================================

raw_market_sum = (
    market_comparison[
        raw_market_probability_columns
    ]
    .sum(axis=1)
)


market_comparison[
    "MarketProbability_H"
] = (
    market_comparison[
        "RawMarketProbability_H"
    ]
    / raw_market_sum
)

market_comparison[
    "MarketProbability_D"
] = (
    market_comparison[
        "RawMarketProbability_D"
    ]
    / raw_market_sum
)

market_comparison[
    "MarketProbability_A"
] = (
    market_comparison[
        "RawMarketProbability_A"
    ]
    / raw_market_sum
)


market_probability_columns = [
    "MarketProbability_H",
    "MarketProbability_D",
    "MarketProbability_A",
]


assert np.allclose(
    market_comparison[
        market_probability_columns
    ].sum(axis=1),
    1.0,
    atol=1e-12,
    rtol=0.0,
)


# ============================================================
# C. MODEL-MARKET PROBABILITY DIFFERENCES
# ============================================================

market_comparison[
    "ModelMinusMarket_H"
] = (
    market_comparison[
        "Probability_H"
    ]
    - market_comparison[
        "MarketProbability_H"
    ]
)

market_comparison[
    "ModelMinusMarket_D"
] = (
    market_comparison[
        "Probability_D"
    ]
    - market_comparison[
        "MarketProbability_D"
    ]
)

market_comparison[
    "ModelMinusMarket_A"
] = (
    market_comparison[
        "Probability_A"
    ]
    - market_comparison[
        "MarketProbability_A"
    ]
)


difference_columns = [
    "ModelMinusMarket_H",
    "ModelMinusMarket_D",
    "ModelMinusMarket_A",
]


# ------------------------------------------------------------
# Total variation distance
# ------------------------------------------------------------

market_comparison[
    "TotalVariationDistance"
] = (
    0.5
    * market_comparison[
        difference_columns
    ]
    .abs()
    .sum(axis=1)
)


# ------------------------------------------------------------
# Largest individual disagreement
# ------------------------------------------------------------

difference_array = (
    market_comparison[
        difference_columns
    ]
    .to_numpy()
)

absolute_difference_array = np.abs(
    difference_array
)

largest_difference_index = np.argmax(
    absolute_difference_array,
    axis=1,
)

outcome_labels = np.array(
    [
        "H",
        "D",
        "A",
    ]
)


market_comparison[
    "LargestDisagreementOutcome"
] = outcome_labels[
    largest_difference_index
]


market_comparison[
    "LargestAbsoluteDisagreement"
] = np.max(
    absolute_difference_array,
    axis=1,
)


market_comparison[
    "LargestSignedDisagreement"
] = difference_array[
    np.arange(
        len(market_comparison)
    ),
    largest_difference_index,
]


# ============================================================
# D. MARKET FAVOURITE
# ============================================================

market_probability_array = (
    market_comparison[
        market_probability_columns
    ]
    .to_numpy()
)


market_favourite_index = np.argmax(
    market_probability_array,
    axis=1,
)


market_comparison[
    "MarketFavouriteOutcome"
] = outcome_labels[
    market_favourite_index
]


def readable_market_favourite(
    row,
):

    if row[
        "MarketFavouriteOutcome"
    ] == "H":

        return row[
            "OfficialHomeTeam"
        ]

    if row[
        "MarketFavouriteOutcome"
    ] == "A":

        return row[
            "OfficialAwayTeam"
        ]

    return "Draw"


market_comparison[
    "MarketFavourite"
] = market_comparison.apply(
    readable_market_favourite,
    axis=1,
)


market_comparison[
    "ModelMarketFavouriteAgree"
] = (
    market_comparison[
        "MostLikelyOutcome"
    ]
    ==
    market_comparison[
        "MarketFavouriteOutcome"
    ]
)


# ============================================================
# E. VALIDATION
# ============================================================

maximum_market_probability_sum_error = float(
    np.max(
        np.abs(
            market_comparison[
                market_probability_columns
            ]
            .sum(axis=1)
            - 1.0
        )
    )
)


maximum_difference_identity_error = float(
    np.max(
        np.abs(
            market_comparison[
                difference_columns
            ]
            .sum(axis=1)
        )
    )
)


assert maximum_market_probability_sum_error < 1e-12

# Both model and market probability distributions sum to one,
# therefore their three signed differences must sum to zero.
assert maximum_difference_identity_error < 1e-12

assert (
    market_comparison[
        "TotalVariationDistance"
    ] >= 0
).all()

assert (
    market_comparison[
        "TotalVariationDistance"
    ] <= 1
).all()


# ============================================================
# F. SUMMARY STATISTICS
# ============================================================

market_agreement_count = int(
    market_comparison[
        "ModelMarketFavouriteAgree"
    ].sum()
)


mean_market_overround = float(
    market_comparison[
        "MarketOverround"
    ].mean()
)


mean_total_variation_distance = float(
    market_comparison[
        "TotalVariationDistance"
    ].mean()
)


maximum_total_variation_distance = float(
    market_comparison[
        "TotalVariationDistance"
    ].max()
)


market_comparison_summary = pd.DataFrame(
    [
        {
            "Metric": (
                "Opening fixtures compared"
            ),
            "Value": len(
                market_comparison
            ),
        },
        {
            "Metric": (
                "Model/market favourite agreement"
            ),
            "Value": (
                f"{market_agreement_count}/10"
            ),
        },
        {
            "Metric": (
                "Mean quoted-market overround"
            ),
            "Value": (
                mean_market_overround
            ),
        },
        {
            "Metric": (
                "Mean total variation distance"
            ),
            "Value": (
                mean_total_variation_distance
            ),
        },
        {
            "Metric": (
                "Maximum total variation distance"
            ),
            "Value": (
                maximum_total_variation_distance
            ),
        },
    ]
)


# ============================================================
# G. RANK FIXTURES BY MODEL-MARKET DISAGREEMENT
# ============================================================

largest_market_disagreements = (
    market_comparison
    .sort_values(
        by=[
            "TotalVariationDistance",
            "LargestAbsoluteDisagreement",
        ],
        ascending=False,
        kind="mergesort",
    )
    .reset_index(drop=True)
)


comparison_display_columns = [
    "OfficialHomeTeam",
    "OfficialAwayTeam",
    "ModelFavourite",
    "MarketFavourite",
    "Probability_H",
    "MarketProbability_H",
    "ModelMinusMarket_H",
    "Probability_D",
    "MarketProbability_D",
    "ModelMinusMarket_D",
    "Probability_A",
    "MarketProbability_A",
    "ModelMinusMarket_A",
    "TotalVariationDistance",
]


comparison_display = (
    largest_market_disagreements[
        comparison_display_columns
    ]
    .copy()
)


numeric_comparison_columns = [
    "Probability_H",
    "MarketProbability_H",
    "ModelMinusMarket_H",
    "Probability_D",
    "MarketProbability_D",
    "ModelMinusMarket_D",
    "Probability_A",
    "MarketProbability_A",
    "ModelMinusMarket_A",
    "TotalVariationDistance",
]


for column in numeric_comparison_columns:

    comparison_display[
        column
    ] = comparison_display[
        column
    ].round(4)


# ============================================================
# H. EXPORT FIXED MARKET SNAPSHOT + COMPARISON
# ============================================================

market_snapshot_output_path = (
    project_root
    / "outputs"
    / "forecasts"
    / "2026_27"
    / "opening_round_market_snapshot_2026_08_09.csv"
)

market_comparison_output_path = (
    project_root
    / "outputs"
    / "forecasts"
    / "2026_27"
    / "opening_round_model_market_comparison.csv"
)


market_snapshot.to_csv(
    market_snapshot_output_path,
    index=False,
)

market_comparison.to_csv(
    market_comparison_output_path,
    index=False,
)


# ------------------------------------------------------------
# Validation table
# ------------------------------------------------------------

market_validation_table = pd.DataFrame(
    [
        {
            "Validation": (
                "Fixtures matched to market"
            ),
            "Value": len(
                market_comparison
            ),
            "Expected": 10,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Positive quoted overrounds"
            ),
            "Value": int(
                (
                    market_comparison[
                        "MarketOverround"
                    ] > 0
                ).sum()
            ),
            "Expected": 10,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Market probability sum max error"
            ),
            "Value": (
                maximum_market_probability_sum_error
            ),
            "Expected": "< 1e-12",
            "Status": "PASS",
        },
        {
            "Validation": (
                "Difference identity max error"
            ),
            "Value": (
                maximum_difference_identity_error
            ),
            "Expected": "< 1e-12",
            "Status": "PASS",
        },
        {
            "Validation": (
                "Finite comparison values"
            ),
            "Value": bool(
                np.isfinite(
                    market_comparison[
                        [
                            "MarketProbability_H",
                            "MarketProbability_D",
                            "MarketProbability_A",
                            "ModelMinusMarket_H",
                            "ModelMinusMarket_D",
                            "ModelMinusMarket_A",
                            "TotalVariationDistance",
                        ]
                    ]
                    .to_numpy()
                ).all()
            ),
            "Expected": True,
            "Status": "PASS",
        },
    ]
)


opening_market_comparison_ready = True


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print(
    "Model-market comparison summary:"
)
display(
    market_comparison_summary
)

print(
    "\nMarket-comparison validation:"
)
display(
    market_validation_table
)

print(
    "\nFixtures ranked by model-market disagreement:"
)
display(
    comparison_display
)

print(
    "\nMarket snapshot file:",
    market_snapshot_output_path
    .relative_to(project_root)
    .as_posix(),
)

print(
    "Model-market comparison file:",
    market_comparison_output_path
    .relative_to(project_root)
    .as_posix(),
)

print(
    "\nOpening-round market comparison status: VALID"
)

print(
    "Ready for disagreement analysis:",
    opening_market_comparison_ready,
)

Model-market comparison summary:


,Metric,Value
0,Opening fixtures compared,10
1,Model/market favourite agreement,9/10
2,Mean quoted-market overround,0.03523
3,Mean total variation distance,0.074184
4,Maximum total variation distance,0.115597



Market-comparison validation:


,Validation,Value,Expected,Status
0,Fixtures matched to market,10,10,PASS
1,Positive quoted overrounds,10,10,PASS
2,Market probability sum max error,0.0,< 1e-12,PASS
3,Difference identity max error,0.0,< 1e-12,PASS
4,Finite comparison values,True,True,PASS



Fixtures ranked by model-market disagreement:


,OfficialHomeTeam,OfficialAwayTeam,ModelFavourite,MarketFavourite,Probability_H,MarketProbability_H,ModelMinusMarket_H,Probability_D,MarketProbability_D,ModelMinusMarket_D,Probability_A,MarketProbability_A,ModelMinusMarket_A,TotalVariationDistance
0,Arsenal,Coventry City,Arsenal,Arsenal,0.7061,0.8217,-0.1156,0.1682,0.1278,0.0404,0.1257,0.0505,0.0752,0.1156
1,Manchester City,AFC Bournemouth,Manchester City,Manchester City,0.5578,0.6678,-0.1100,0.2143,0.1837,0.0306,0.2278,0.1484,0.0794,0.1100
2,Ipswich Town,Sunderland,Sunderland,Sunderland,0.2760,0.3489,-0.0729,0.2461,0.2822,-0.0360,0.4779,0.3690,0.1089,0.1089
3,Brighton & Hove Albion,Aston Villa,Aston Villa,Brighton & Hove Albion,0.3404,0.4216,-0.0812,0.2413,0.2552,-0.0139,0.4183,0.3232,0.0951,0.0951
4,Everton,Crystal Palace,Everton,Everton,0.3932,0.4485,-0.0553,0.2475,0.2679,-0.0204,0.3593,0.2836,0.0757,0.0757
5,Brentford,Tottenham Hotspur,Brentford,Brentford,0.4806,0.4060,0.0746,0.2395,0.2650,-0.0255,0.2798,0.3290,-0.0491,0.0746
6,Fulham,Chelsea,Chelsea,Chelsea,0.3640,0.3033,0.0607,0.2456,0.2554,-0.0098,0.3903,0.4412,-0.0509,0.0607
7,Hull City,Manchester United,Manchester United,Manchester United,0.1794,0.1308,0.0486,0.2079,0.2105,-0.0026,0.6127,0.6587,-0.0460,0.0486
8,Nottingham Forest,Leeds United,Nottingham Forest,Nottingham Forest,0.4196,0.4295,-0.0099,0.2466,0.2685,-0.0219,0.3338,0.3020,0.0318,0.0318
9,Newcastle United,Liverpool,Liverpool,Liverpool,0.3061,0.2853,0.0209,0.2379,0.2461,-0.0082,0.4560,0.4687,-0.0127,0.0209



Market snapshot file: outputs/forecasts/2026_27/opening_round_market_snapshot_2026_08_09.csv
Model-market comparison file: outputs/forecasts/2026_27/opening_round_model_market_comparison.csv

Opening-round market comparison status: VALID
Ready for disagreement analysis: True


### Results and Interpretation

The opening-round forecasts were matched successfully to all ten current market snapshots, and the quoted-market margin was removed before comparison.

The mean quoted-market overround is approximately $3.52\%$. After normalisation, both the model and market probability distributions sum to one to numerical precision.

The model and market agree on the most likely outcome in nine of the ten opening fixtures. The only disagreement is Brighton & Hove Albion versus Aston Villa: the market currently favours Brighton, whereas the production model assigns the highest probability to Aston Villa.

Despite the high agreement in headline favourites, there are meaningful differences in the underlying probability distributions. The mean total variation distance is approximately $0.0742$, while the largest is approximately $0.1156$.

The largest disagreement occurs for Arsenal versus Coventry City. The market assigns Arsenal approximately $82.17\%$ after margin removal, compared with $70.61\%$ from the model. The model consequently assigns substantially more probability to both the draw and a Coventry win than the current market does.

A similar pattern appears for Manchester City versus Bournemouth. The market assigns Manchester City approximately $66.78\%$, compared with $55.78\%$ from the model. Again, the production model produces a less concentrated distribution around the strong favourite.

The most notable disagreement in the opposite direction occurs for Ipswich Town versus Sunderland. The production model assigns Sunderland approximately $47.79\%$, compared with approximately $36.90\%$ from the market, a difference of roughly $10.89$ percentage points.

Brighton versus Aston Villa is also particularly notable. The model assigns Aston Villa approximately $41.83\%$, compared with approximately $32.32\%$ from the market, while assigning Brighton approximately $34.04\%$ compared with the market's $42.16\%$.

Other meaningful differences include greater model probability for Brentford against Tottenham and greater away-win probability for Crystal Palace against Everton. Newcastle United versus Liverpool shows the closest overall agreement, with a total variation distance of only approximately $0.0209$.

These discrepancies should be interpreted as model-market disagreement rather than evidence of mispricing. The prices are an early best-odds snapshot rather than closing market probabilities, and the historical analysis did not establish a persistent profitable market edge.

## 7. Rank Individual Model–Market Disagreements

Fixture-level total variation distance identifies matches where the model and market differ, but it does not show which individual outcomes are responsible for that disagreement.

The ten opening fixtures are therefore expanded into thirty separate home-win, draw and away-win outcomes.

For each outcome, the analysis reports:

- the frozen production-model probability;
- the margin-adjusted market probability;
- the model-minus-market probability difference;
- the current quoted decimal odds;
- the model-implied fair decimal odds;
- the theoretical expected return at the quoted price.

Model-implied fair odds are defined as

$$
o_i^{fair}=\frac{1}{p_i^{model}}.
$$

For descriptive purposes, the theoretical expected return from staking one unit at the quoted decimal price is

$$
ER_i=p_i^{model}o_i^{quoted}-1.
$$

A positive value means that the quoted price is larger than the price implied by the model's own probability estimate.

This quantity is not treated as evidence of a profitable betting opportunity. It depends entirely on the accuracy of the model probability, uses an early market snapshot rather than closing prices, and must be interpreted in light of the project's historical finding that no persistent market-beating edge has been demonstrated.

In [9]:
# ============================================================
# 7. Rank Individual Model-Market Disagreements
# ============================================================

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

assert "opening_market_comparison_ready" in globals()
assert opening_market_comparison_ready is True

assert len(market_comparison) == 10


# ------------------------------------------------------------
# Expand 10 fixtures into 30 individual outcomes
# ------------------------------------------------------------

outcome_records = []


for row in market_comparison.itertuples(
    index=False
):

    fixture_outcomes = [
        {
            "OutcomeCode": "H",
            "Outcome": (
                row.OfficialHomeTeam
            ),
            "ModelProbability": (
                row.Probability_H
            ),
            "MarketProbability": (
                row.MarketProbability_H
            ),
            "QuotedDecimalOdds": (
                row.MarketOdds_H
            ),
        },
        {
            "OutcomeCode": "D",
            "Outcome": "Draw",
            "ModelProbability": (
                row.Probability_D
            ),
            "MarketProbability": (
                row.MarketProbability_D
            ),
            "QuotedDecimalOdds": (
                row.MarketOdds_D
            ),
        },
        {
            "OutcomeCode": "A",
            "Outcome": (
                row.OfficialAwayTeam
            ),
            "ModelProbability": (
                row.Probability_A
            ),
            "MarketProbability": (
                row.MarketProbability_A
            ),
            "QuotedDecimalOdds": (
                row.MarketOdds_A
            ),
        },
    ]


    for outcome_record in fixture_outcomes:

        outcome_records.append(
            {
                "OfficialHomeTeam": (
                    row.OfficialHomeTeam
                ),
                "OfficialAwayTeam": (
                    row.OfficialAwayTeam
                ),
                **outcome_record,
            }
        )


outcome_comparison = pd.DataFrame(
    outcome_records
)


assert len(outcome_comparison) == 30


# ============================================================
# A. PROBABILITY DIFFERENCE
# ============================================================

outcome_comparison[
    "ModelMinusMarket"
] = (
    outcome_comparison[
        "ModelProbability"
    ]
    -
    outcome_comparison[
        "MarketProbability"
    ]
)


# ============================================================
# B. MODEL FAIR ODDS
# ============================================================

outcome_comparison[
    "ModelFairOdds"
] = (
    1.0
    /
    outcome_comparison[
        "ModelProbability"
    ]
)


assert np.isfinite(
    outcome_comparison[
        "ModelFairOdds"
    ]
).all()

assert (
    outcome_comparison[
        "ModelFairOdds"
    ] > 1
).all()


# ============================================================
# C. THEORETICAL EXPECTED RETURN AT QUOTED PRICE
# ============================================================

outcome_comparison[
    "ModelExpectedReturn"
] = (
    outcome_comparison[
        "ModelProbability"
    ]
    *
    outcome_comparison[
        "QuotedDecimalOdds"
    ]
    - 1.0
)


# ------------------------------------------------------------
# Price ratio
#
# >1 means current quoted odds exceed model fair odds.
# ------------------------------------------------------------

outcome_comparison[
    "QuotedToFairOddsRatio"
] = (
    outcome_comparison[
        "QuotedDecimalOdds"
    ]
    /
    outcome_comparison[
        "ModelFairOdds"
    ]
)


# ============================================================
# D. DIRECTION OF DISAGREEMENT
# ============================================================

outcome_comparison[
    "DisagreementDirection"
] = np.select(
    [
        (
            outcome_comparison[
                "ModelMinusMarket"
            ] > 0
        ),
        (
            outcome_comparison[
                "ModelMinusMarket"
            ] < 0
        ),
    ],
    [
        "Model higher",
        "Market higher",
    ],
    default="Equal",
)


# ============================================================
# E. VALIDATION
# ============================================================

assert (
    outcome_comparison[
        "ModelProbability"
    ]
    .between(
        0,
        1,
        inclusive="both",
    )
    .all()
)

assert (
    outcome_comparison[
        "MarketProbability"
    ]
    .between(
        0,
        1,
        inclusive="both",
    )
    .all()
)

assert (
    outcome_comparison[
        "QuotedDecimalOdds"
    ] > 1
).all()


# Each fixture's three differences must sum to zero.
difference_sum_by_fixture = (
    outcome_comparison
    .groupby(
        [
            "OfficialHomeTeam",
            "OfficialAwayTeam",
        ],
        sort=False,
    )[
        "ModelMinusMarket"
    ]
    .sum()
)


maximum_fixture_difference_sum_error = float(
    np.max(
        np.abs(
            difference_sum_by_fixture
        )
    )
)


assert (
    maximum_fixture_difference_sum_error
    < 1e-12
)


# ============================================================
# F. RANK POSITIVE AND NEGATIVE DISAGREEMENTS
# ============================================================

model_higher_ranked = (
    outcome_comparison
    .sort_values(
        by=[
            "ModelMinusMarket",
            "ModelExpectedReturn",
        ],
        ascending=False,
        kind="mergesort",
    )
    .reset_index(drop=True)
)


market_higher_ranked = (
    outcome_comparison
    .sort_values(
        by=[
            "ModelMinusMarket",
            "ModelExpectedReturn",
        ],
        ascending=True,
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Top positive theoretical expected returns
# ------------------------------------------------------------

expected_return_ranked = (
    outcome_comparison
    .sort_values(
        by=[
            "ModelExpectedReturn",
            "ModelMinusMarket",
        ],
        ascending=False,
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ============================================================
# G. SUMMARY STATISTICS
# ============================================================

positive_probability_differences = int(
    (
        outcome_comparison[
            "ModelMinusMarket"
        ] > 0
    ).sum()
)

positive_model_expected_returns = int(
    (
        outcome_comparison[
            "ModelExpectedReturn"
        ] > 0
    ).sum()
)


maximum_positive_difference = float(
    outcome_comparison[
        "ModelMinusMarket"
    ].max()
)

maximum_negative_difference = float(
    outcome_comparison[
        "ModelMinusMarket"
    ].min()
)


maximum_theoretical_expected_return = float(
    outcome_comparison[
        "ModelExpectedReturn"
    ].max()
)

minimum_theoretical_expected_return = float(
    outcome_comparison[
        "ModelExpectedReturn"
    ].min()
)


outcome_disagreement_summary = pd.DataFrame(
    [
        {
            "Metric": (
                "Individual outcomes compared"
            ),
            "Value": 30,
        },
        {
            "Metric": (
                "Outcomes where model probability "
                "exceeds market"
            ),
            "Value": (
                positive_probability_differences
            ),
        },
        {
            "Metric": (
                "Positive theoretical expected returns"
            ),
            "Value": (
                positive_model_expected_returns
            ),
        },
        {
            "Metric": (
                "Largest positive probability difference"
            ),
            "Value": (
                maximum_positive_difference
            ),
        },
        {
            "Metric": (
                "Largest negative probability difference"
            ),
            "Value": (
                maximum_negative_difference
            ),
        },
        {
            "Metric": (
                "Largest theoretical expected return"
            ),
            "Value": (
                maximum_theoretical_expected_return
            ),
        },
        {
            "Metric": (
                "Lowest theoretical expected return"
            ),
            "Value": (
                minimum_theoretical_expected_return
            ),
        },
    ]
)


# ============================================================
# H. DISPLAY TABLES
# ============================================================

display_columns = [
    "OfficialHomeTeam",
    "OfficialAwayTeam",
    "Outcome",
    "OutcomeCode",
    "ModelProbability",
    "MarketProbability",
    "ModelMinusMarket",
    "QuotedDecimalOdds",
    "ModelFairOdds",
    "ModelExpectedReturn",
]


model_higher_display = (
    model_higher_ranked[
        display_columns
    ]
    .head(10)
    .copy()
)


market_higher_display = (
    market_higher_ranked[
        display_columns
    ]
    .head(10)
    .copy()
)


expected_return_display = (
    expected_return_ranked[
        display_columns
    ]
    .head(10)
    .copy()
)


for table in [
    model_higher_display,
    market_higher_display,
    expected_return_display,
]:

    for column in [
        "ModelProbability",
        "MarketProbability",
        "ModelMinusMarket",
        "ModelExpectedReturn",
    ]:

        table[
            column
        ] = table[
            column
        ].round(4)


    for column in [
        "QuotedDecimalOdds",
        "ModelFairOdds",
    ]:

        table[
            column
        ] = table[
            column
        ].round(3)


# ============================================================
# I. EXPORT
# ============================================================

outcome_comparison_output_path = (
    project_root
    / "outputs"
    / "forecasts"
    / "2026_27"
    / "opening_round_outcome_disagreements.csv"
)


outcome_comparison.to_csv(
    outcome_comparison_output_path,
    index=False,
)


opening_outcome_disagreement_ready = True


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print(
    "Individual outcome disagreement summary:"
)
display(
    outcome_disagreement_summary
)

print(
    "\nLargest outcomes where MODEL probability is higher:"
)
display(
    model_higher_display
)

print(
    "\nLargest outcomes where MARKET probability is higher:"
)
display(
    market_higher_display
)

print(
    "\nHighest theoretical expected returns at quoted prices:"
)
display(
    expected_return_display
)

print(
    "\nOutcome-disagreement file:",
    outcome_comparison_output_path
    .relative_to(project_root)
    .as_posix(),
)

print(
    "\nIndividual disagreement analysis status: VALID"
)

print(
    "Outcome-level disagreement analysis ready:",
    opening_outcome_disagreement_ready,
)

Individual outcome disagreement summary:


,Metric,Value
0,Individual outcomes compared,30.000000
1,Outcomes where model probability exceeds market,12.000000
2,Positive theoretical expected returns,12.000000
3,Largest positive probability difference,0.108913
4,Largest negative probability difference,-0.115597
5,Largest theoretical expected return,1.387685
6,Lowest theoretical expected return,-0.241071



Largest outcomes where MODEL probability is higher:


,OfficialHomeTeam,OfficialAwayTeam,Outcome,OutcomeCode,ModelProbability,MarketProbability,ModelMinusMarket,QuotedDecimalOdds,ModelFairOdds,ModelExpectedReturn
0,Ipswich Town,Sunderland,Sunderland,A,0.4779,0.3690,0.1089,2.60,2.093,0.2425
1,Brighton & Hove Albion,Aston Villa,Aston Villa,A,0.4183,0.3232,0.0951,3.00,2.391,0.2549
2,Manchester City,AFC Bournemouth,AFC Bournemouth,A,0.2278,0.1484,0.0794,6.50,4.389,0.4808
3,Everton,Crystal Palace,Crystal Palace,A,0.3593,0.2836,0.0757,3.40,2.783,0.2217
4,Arsenal,Coventry City,Coventry City,A,0.1257,0.0505,0.0752,19.00,7.957,1.3877
5,Brentford,Tottenham Hotspur,Brentford,H,0.4806,0.4060,0.0746,2.35,2.081,0.1295
6,Fulham,Chelsea,Fulham,H,0.3640,0.3033,0.0607,3.20,2.747,0.1648
7,Hull City,Manchester United,Hull City,H,0.1794,0.1308,0.0486,7.40,5.573,0.3278
8,Arsenal,Coventry City,Draw,D,0.1682,0.1278,0.0404,7.50,5.945,0.2616
9,Nottingham Forest,Leeds United,Leeds United,A,0.3338,0.3020,0.0318,3.20,2.996,0.0682



Largest outcomes where MARKET probability is higher:


,OfficialHomeTeam,OfficialAwayTeam,Outcome,OutcomeCode,ModelProbability,MarketProbability,ModelMinusMarket,QuotedDecimalOdds,ModelFairOdds,ModelExpectedReturn
0,Arsenal,Coventry City,Arsenal,H,0.7061,0.8217,-0.1156,1.167,1.416,-0.1762
1,Manchester City,AFC Bournemouth,Manchester City,H,0.5578,0.6678,-0.1100,1.444,1.793,-0.1942
2,Brighton & Hove Albion,Aston Villa,Brighton & Hove Albion,H,0.3404,0.4216,-0.0812,2.300,2.938,-0.2171
3,Ipswich Town,Sunderland,Ipswich Town,H,0.2760,0.3489,-0.0729,2.750,3.624,-0.2411
4,Everton,Crystal Palace,Everton,H,0.3932,0.4485,-0.0553,2.150,2.543,-0.1546
5,Fulham,Chelsea,Chelsea,A,0.3903,0.4412,-0.0509,2.200,2.562,-0.1412
6,Brentford,Tottenham Hotspur,Tottenham Hotspur,A,0.2798,0.3290,-0.0491,2.900,3.573,-0.1884
7,Hull City,Manchester United,Manchester United,A,0.6127,0.6587,-0.0460,1.470,1.632,-0.0994
8,Ipswich Town,Sunderland,Draw,D,0.2461,0.2822,-0.0360,3.400,4.063,-0.1632
9,Brentford,Tottenham Hotspur,Draw,D,0.2395,0.2650,-0.0255,3.600,4.175,-0.1377



Highest theoretical expected returns at quoted prices:


,OfficialHomeTeam,OfficialAwayTeam,Outcome,OutcomeCode,ModelProbability,MarketProbability,ModelMinusMarket,QuotedDecimalOdds,ModelFairOdds,ModelExpectedReturn
0,Arsenal,Coventry City,Coventry City,A,0.1257,0.0505,0.0752,19.00,7.957,1.3877
1,Manchester City,AFC Bournemouth,AFC Bournemouth,A,0.2278,0.1484,0.0794,6.50,4.389,0.4808
2,Hull City,Manchester United,Hull City,H,0.1794,0.1308,0.0486,7.40,5.573,0.3278
3,Arsenal,Coventry City,Draw,D,0.1682,0.1278,0.0404,7.50,5.945,0.2616
4,Brighton & Hove Albion,Aston Villa,Aston Villa,A,0.4183,0.3232,0.0951,3.00,2.391,0.2549
5,Ipswich Town,Sunderland,Sunderland,A,0.4779,0.3690,0.1089,2.60,2.093,0.2425
6,Everton,Crystal Palace,Crystal Palace,A,0.3593,0.2836,0.0757,3.40,2.783,0.2217
7,Fulham,Chelsea,Fulham,H,0.3640,0.3033,0.0607,3.20,2.747,0.1648
8,Brentford,Tottenham Hotspur,Brentford,H,0.4806,0.4060,0.0746,2.35,2.081,0.1295
9,Manchester City,AFC Bournemouth,Draw,D,0.2143,0.1837,0.0306,5.25,4.666,0.1253



Outcome-disagreement file: outputs/forecasts/2026_27/opening_round_outcome_disagreements.csv

Individual disagreement analysis status: VALID
Outcome-level disagreement analysis ready: True


### Results and Interpretation

The ten opening fixtures produce thirty individual home, draw and away outcomes. The production model assigns a higher probability than the margin-adjusted market to 12 of those 30 outcomes, and the same 12 outcomes consequently produce a positive theoretical expected return at the quoted prices.

The strongest positive probability disagreement is Sunderland away at Ipswich. The model assigns Sunderland approximately $47.79\%$, compared with approximately $36.90\%$ from the market, a difference of roughly $10.89$ percentage points.

Aston Villa away at Brighton represents the next-largest positive disagreement. The model assigns Villa approximately $41.83\%$, compared with approximately $32.32\%$ from the market, a difference of approximately $9.51$ percentage points.

Other notable outcomes where the model probability exceeds the market include:

- Bournemouth away at Manchester City: approximately $+7.94$ percentage points;
- Crystal Palace away at Everton: approximately $+7.57$ percentage points;
- Coventry away at Arsenal: approximately $+7.52$ percentage points;
- Brentford at home to Tottenham: approximately $+7.46$ percentage points;
- Fulham at home to Chelsea: approximately $+6.07$ percentage points;
- Hull at home to Manchester United: approximately $+4.86$ percentage points.

The largest disagreements in the opposite direction concern strong home favourites. Arsenal's home-win probability is approximately $11.56$ percentage points lower in the model than in the market, while Manchester City's is approximately $11.00$ percentage points lower. The model is also materially less positive than the market about Brighton and Ipswich winning their respective fixtures.

The largest calculated theoretical expected return is approximately $+138.8\%$, corresponding to Coventry at quoted decimal odds of $19.00$. This result requires particular caution. Coventry's model probability is approximately $12.57\%$ compared with a market probability near $5.05\%$, so the high quoted price mechanically magnifies the probability disagreement into a very large expected-return estimate.

This demonstrates an important limitation of ranking outcomes purely by theoretical expected return. Long-priced outcomes are extremely sensitive to relatively small probability-estimation errors, and the project's historical betting analysis previously showed that apparent returns were fragile and disproportionately influenced by longshots.

For this reason, the probability difference itself is more informative for identifying substantive model-market disagreement than the headline theoretical expected-return figure. Sunderland and Aston Villa currently represent the clearest opening-round examples of the production model expressing a materially different view from the market.

These results remain descriptive and do not establish profitable betting opportunities.

## 8. Construct a Conservative Model–Market Disagreement Watchlist

The theoretical expected-return ranking is strongly affected by quoted odds and can therefore elevate low-probability longshots when the model assigns only moderately more probability than the market.

For research presentation, a more conservative watchlist is constructed directly from probability disagreement.

An outcome is classified as a `Material disagreement` when:

$$
p^{model}-p^{market} \geq 0.05.
$$

Within that group, outcomes receiving at least $20\%$ probability from the model are classified as `Primary watchlist` observations. Outcomes with the same probability disagreement but model probability below $20\%$ are labelled `Longshot-sensitive`.

The $5$ percentage-point and $20\%$ thresholds are descriptive reporting choices rather than optimised betting rules. They are not estimated from historical returns and are not used to modify the production model.

The purpose is to distinguish substantial differences in probabilistic opinion from theoretical expected-return figures that may be dominated by high quoted prices.

In [10]:
# ============================================================
# 8. Construct a Conservative Model-Market Watchlist
# ============================================================

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

assert "opening_outcome_disagreement_ready" in globals()
assert opening_outcome_disagreement_ready is True

assert len(outcome_comparison) == 30


# ------------------------------------------------------------
# Descriptive thresholds
# ------------------------------------------------------------

MATERIAL_DISAGREEMENT_THRESHOLD = 0.05
PRIMARY_MODEL_PROBABILITY_THRESHOLD = 0.20


# ------------------------------------------------------------
# Classification
# ------------------------------------------------------------

watchlist = outcome_comparison.copy()


watchlist[
    "MaterialPositiveDisagreement"
] = (
    watchlist[
        "ModelMinusMarket"
    ]
    >= MATERIAL_DISAGREEMENT_THRESHOLD
)


watchlist[
    "WatchlistClassification"
] = np.select(
    [
        (
            watchlist[
                "MaterialPositiveDisagreement"
            ]
            &
            (
                watchlist[
                    "ModelProbability"
                ]
                >= PRIMARY_MODEL_PROBABILITY_THRESHOLD
            )
        ),

        (
            watchlist[
                "MaterialPositiveDisagreement"
            ]
            &
            (
                watchlist[
                    "ModelProbability"
                ]
                < PRIMARY_MODEL_PROBABILITY_THRESHOLD
            )
        ),
    ],
    [
        "Primary watchlist",
        "Longshot-sensitive",
    ],
    default="Below material threshold",
)


# ------------------------------------------------------------
# Break-even probability from the quoted odds
# ------------------------------------------------------------

watchlist[
    "QuotedBreakEvenProbability"
] = (
    1.0
    /
    watchlist[
        "QuotedDecimalOdds"
    ]
)


watchlist[
    "ModelMinusBreakEvenProbability"
] = (
    watchlist[
        "ModelProbability"
    ]
    -
    watchlist[
        "QuotedBreakEvenProbability"
    ]
)


# ------------------------------------------------------------
# Relative probability disagreement
#
# Descriptive only.
# ------------------------------------------------------------

watchlist[
    "RelativeProbabilityDifference"
] = (
    watchlist[
        "ModelProbability"
    ]
    /
    watchlist[
        "MarketProbability"
    ]
    - 1.0
)


# ------------------------------------------------------------
# Primary watchlist
# ------------------------------------------------------------

primary_watchlist = (
    watchlist.loc[
        watchlist[
            "WatchlistClassification"
        ]
        == "Primary watchlist"
    ]
    .sort_values(
        by=[
            "ModelMinusMarket",
            "ModelProbability",
        ],
        ascending=False,
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Longshot-sensitive disagreements
# ------------------------------------------------------------

longshot_sensitive = (
    watchlist.loc[
        watchlist[
            "WatchlistClassification"
        ]
        == "Longshot-sensitive"
    ]
    .sort_values(
        by=[
            "ModelMinusMarket",
            "ModelProbability",
        ],
        ascending=False,
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Largest outcomes where market probability is higher
# ------------------------------------------------------------

market_conviction = (
    watchlist
    .sort_values(
        by="ModelMinusMarket",
        ascending=True,
        kind="mergesort",
    )
    .head(5)
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert (
    primary_watchlist[
        "ModelMinusMarket"
    ]
    >= MATERIAL_DISAGREEMENT_THRESHOLD
).all()

assert (
    primary_watchlist[
        "ModelProbability"
    ]
    >= PRIMARY_MODEL_PROBABILITY_THRESHOLD
).all()


assert (
    longshot_sensitive[
        "ModelMinusMarket"
    ]
    >= MATERIAL_DISAGREEMENT_THRESHOLD
).all()

assert (
    longshot_sensitive[
        "ModelProbability"
    ]
    < PRIMARY_MODEL_PROBABILITY_THRESHOLD
).all()


classified_material_count = (
    len(primary_watchlist)
    +
    len(longshot_sensitive)
)

expected_material_count = int(
    (
        outcome_comparison[
            "ModelMinusMarket"
        ]
        >= MATERIAL_DISAGREEMENT_THRESHOLD
    ).sum()
)

assert (
    classified_material_count
    == expected_material_count
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

watchlist_summary = pd.DataFrame(
    [
        {
            "Metric": (
                "Individual outcomes"
            ),
            "Value": 30,
        },
        {
            "Metric": (
                "Material positive disagreements"
            ),
            "Value": (
                classified_material_count
            ),
        },
        {
            "Metric": (
                "Primary watchlist outcomes"
            ),
            "Value": (
                len(primary_watchlist)
            ),
        },
        {
            "Metric": (
                "Longshot-sensitive outcomes"
            ),
            "Value": (
                len(longshot_sensitive)
            ),
        },
        {
            "Metric": (
                "Probability-difference threshold"
            ),
            "Value": (
                MATERIAL_DISAGREEMENT_THRESHOLD
            ),
        },
        {
            "Metric": (
                "Primary minimum model probability"
            ),
            "Value": (
                PRIMARY_MODEL_PROBABILITY_THRESHOLD
            ),
        },
    ]
)


# ------------------------------------------------------------
# Display tables
# ------------------------------------------------------------

display_columns = [
    "OfficialHomeTeam",
    "OfficialAwayTeam",
    "Outcome",
    "OutcomeCode",
    "ModelProbability",
    "MarketProbability",
    "ModelMinusMarket",
    "QuotedDecimalOdds",
    "ModelFairOdds",
    "ModelExpectedReturn",
    "WatchlistClassification",
]


primary_display = (
    primary_watchlist[
        display_columns
    ]
    .copy()
)


longshot_display = (
    longshot_sensitive[
        display_columns
    ]
    .copy()
)


market_conviction_display = (
    market_conviction[
        display_columns
    ]
    .copy()
)


for table in [
    primary_display,
    longshot_display,
    market_conviction_display,
]:

    for column in [
        "ModelProbability",
        "MarketProbability",
        "ModelMinusMarket",
        "ModelExpectedReturn",
    ]:

        table[
            column
        ] = table[
            column
        ].round(4)


    for column in [
        "QuotedDecimalOdds",
        "ModelFairOdds",
    ]:

        table[
            column
        ] = table[
            column
        ].round(3)


# ------------------------------------------------------------
# Export
# ------------------------------------------------------------

watchlist_output_path = (
    project_root
    / "outputs"
    / "forecasts"
    / "2026_27"
    / "opening_round_disagreement_watchlist.csv"
)


watchlist.to_csv(
    watchlist_output_path,
    index=False,
)


opening_watchlist_ready = True


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print(
    "Model-market watchlist summary:"
)
display(
    watchlist_summary
)

print(
    "\nPrimary probability-disagreement watchlist:"
)
display(
    primary_display
)

print(
    "\nLongshot-sensitive disagreements:"
)
display(
    longshot_display
)

print(
    "\nLargest outcomes where the MARKET is more confident:"
)
display(
    market_conviction_display
)

print(
    "\nWatchlist file:",
    watchlist_output_path
    .relative_to(project_root)
    .as_posix(),
)

print(
    "\nOpening-round watchlist status: VALID"
)

print(
    "Conservative disagreement watchlist ready:",
    opening_watchlist_ready,
)

Model-market watchlist summary:


,Metric,Value
0,Individual outcomes,30.00
1,Material positive disagreements,7.00
2,Primary watchlist outcomes,6.00
3,Longshot-sensitive outcomes,1.00
4,Probability-difference threshold,0.05
5,Primary minimum model probability,0.20



Primary probability-disagreement watchlist:


,OfficialHomeTeam,OfficialAwayTeam,Outcome,OutcomeCode,ModelProbability,MarketProbability,ModelMinusMarket,QuotedDecimalOdds,ModelFairOdds,ModelExpectedReturn,WatchlistClassification
0,Ipswich Town,Sunderland,Sunderland,A,0.4779,0.3690,0.1089,2.60,2.093,0.2425,Primary watchlist
1,Brighton & Hove Albion,Aston Villa,Aston Villa,A,0.4183,0.3232,0.0951,3.00,2.391,0.2549,Primary watchlist
2,Manchester City,AFC Bournemouth,AFC Bournemouth,A,0.2278,0.1484,0.0794,6.50,4.389,0.4808,Primary watchlist
3,Everton,Crystal Palace,Crystal Palace,A,0.3593,0.2836,0.0757,3.40,2.783,0.2217,Primary watchlist
4,Brentford,Tottenham Hotspur,Brentford,H,0.4806,0.4060,0.0746,2.35,2.081,0.1295,Primary watchlist
5,Fulham,Chelsea,Fulham,H,0.3640,0.3033,0.0607,3.20,2.747,0.1648,Primary watchlist



Longshot-sensitive disagreements:


,OfficialHomeTeam,OfficialAwayTeam,Outcome,OutcomeCode,ModelProbability,MarketProbability,ModelMinusMarket,QuotedDecimalOdds,ModelFairOdds,ModelExpectedReturn,WatchlistClassification
0,Arsenal,Coventry City,Coventry City,A,0.1257,0.0505,0.0752,19.0,7.957,1.3877,Longshot-sensitive



Largest outcomes where the MARKET is more confident:


,OfficialHomeTeam,OfficialAwayTeam,Outcome,OutcomeCode,ModelProbability,MarketProbability,ModelMinusMarket,QuotedDecimalOdds,ModelFairOdds,ModelExpectedReturn,WatchlistClassification
0,Arsenal,Coventry City,Arsenal,H,0.7061,0.8217,-0.1156,1.167,1.416,-0.1762,Below material threshold
1,Manchester City,AFC Bournemouth,Manchester City,H,0.5578,0.6678,-0.1100,1.444,1.793,-0.1942,Below material threshold
2,Brighton & Hove Albion,Aston Villa,Brighton & Hove Albion,H,0.3404,0.4216,-0.0812,2.300,2.938,-0.2171,Below material threshold
3,Ipswich Town,Sunderland,Ipswich Town,H,0.2760,0.3489,-0.0729,2.750,3.624,-0.2411,Below material threshold
4,Everton,Crystal Palace,Everton,H,0.3932,0.4485,-0.0553,2.150,2.543,-0.1546,Below material threshold



Watchlist file: outputs/forecasts/2026_27/opening_round_disagreement_watchlist.csv

Opening-round watchlist status: VALID
Conservative disagreement watchlist ready: True


### Results and Interpretation

Seven of the thirty opening-round outcomes satisfy the descriptive threshold of at least a five percentage-point positive model–market probability difference.

Six of these also receive at least 20% probability from the production model and therefore form the primary disagreement watchlist:

- Sunderland away at Ipswich;
- Aston Villa away at Brighton;
- Bournemouth away at Manchester City;
- Crystal Palace away at Everton;
- Brentford at home to Tottenham;
- Fulham at home to Chelsea.

Sunderland represents the largest primary probability disagreement. The production model assigns approximately 47.79% to an away win compared with approximately 36.90% from the margin-adjusted market.

Aston Villa follows with approximately 41.83% model probability compared with approximately 32.32% from the market.

Coventry away at Arsenal also exceeds the five percentage-point disagreement threshold, but its production probability is only approximately 12.57%. It is therefore separated as `Longshot-sensitive` rather than included in the primary watchlist. This prevents the very high quoted price from dominating the presentation through an extreme theoretical expected-return calculation.

The classification is deliberately based primarily on probability disagreement rather than theoretical betting return. The thresholds are descriptive and were not optimised against historical betting performance.

The resulting watchlist therefore highlights where the production model currently expresses its strongest substantive disagreement with the opening market without interpreting those differences as demonstrated profitable betting opportunities.

## 9. Build the Final Opening-Round Forecast Table

The modelling, uncertainty and market-comparison outputs are now consolidated into a single presentation-ready forecast table.

For each opening-round fixture, the final table reports:

- expected home and away goals;
- production-model home, draw and away probabilities;
- the model favourite;
- descriptive forecast confidence;
- margin-adjusted market home, draw and away probabilities;
- the current market favourite;
- outcome-specific model-minus-market probability differences;
- total model-market variation distance;
- any primary or longshot-sensitive disagreement identified by the watchlist analysis.

The underlying probabilities remain unchanged. This section performs no additional fitting, calibration or decision-rule optimisation; it only consolidates the previously validated outputs into a reproducible forecasting artefact suitable for reporting and portfolio presentation.

In [11]:
# ============================================================
# 9. Build the Final Opening-Round Forecast Table
# ============================================================

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

assert "opening_round_forecasts_ready" in globals()
assert opening_round_forecasts_ready is True

assert "opening_forecast_diagnostics_ready" in globals()
assert opening_forecast_diagnostics_ready is True

assert "opening_market_comparison_ready" in globals()
assert opening_market_comparison_ready is True

assert "opening_watchlist_ready" in globals()
assert opening_watchlist_ready is True

assert len(opening_round_forecasts) == 10
assert len(market_comparison) == 10
assert len(outcome_comparison) == 30


# ------------------------------------------------------------
# Locate project
# ------------------------------------------------------------

project_root = Path.cwd().resolve()

while (
    project_root.name != "premier-league-probability-engine"
    and project_root.parent != project_root
):
    project_root = project_root.parent

assert project_root.name == "premier-league-probability-engine"


final_forecast_output_path = (
    project_root
    / "outputs"
    / "forecasts"
    / "2026_27"
    / "opening_round_final_forecast_table.csv"
)

final_forecast_output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# A. START FROM THE MODEL-MARKET COMPARISON TABLE
# ============================================================

final_opening_forecasts = market_comparison[
    [
        "FixtureNumber",
        "Season",
        "Kickoff",
        "OfficialHomeTeam",
        "OfficialAwayTeam",

        "HomeExpectedGoals",
        "AwayExpectedGoals",

        "Probability_H",
        "Probability_D",
        "Probability_A",

        "ModelFavourite",
        "MostLikelyOutcome",
        "MostLikelyOutcomeProbability",
        "ModalScoreline",
        "ModalScoreProbability",

        "MarketOdds_H",
        "MarketOdds_D",
        "MarketOdds_A",

        "MarketProbability_H",
        "MarketProbability_D",
        "MarketProbability_A",

        "MarketFavourite",
        "MarketFavouriteOutcome",
        "ModelMarketFavouriteAgree",

        "ModelMinusMarket_H",
        "ModelMinusMarket_D",
        "ModelMinusMarket_A",

        "TotalVariationDistance",
    ]
].copy()


# ============================================================
# B. ADD FORECAST-CONFIDENCE DIAGNOSTICS
# ============================================================

confidence_columns = (
    opening_forecast_diagnostics[
        [
            "OfficialHomeTeam",
            "OfficialAwayTeam",
            "FavouriteProbabilityMargin",
            "NormalisedEntropy",
            "ExpectedTotalGoals",
            "ExpectedGoalDifference",
            "Confidence",
        ]
    ]
    .copy()
)


final_opening_forecasts = (
    final_opening_forecasts
    .merge(
        confidence_columns,
        on=[
            "OfficialHomeTeam",
            "OfficialAwayTeam",
        ],
        how="left",
        validate="one_to_one",
    )
)


assert not final_opening_forecasts[
    "Confidence"
].isna().any()


# ============================================================
# C. COLLAPSE WATCHLIST OUTCOMES TO FIXTURE LEVEL
# ============================================================

primary_watchlist_rows = watchlist.loc[
    watchlist[
        "WatchlistClassification"
    ] == "Primary watchlist"
].copy()


longshot_watchlist_rows = watchlist.loc[
    watchlist[
        "WatchlistClassification"
    ] == "Longshot-sensitive"
].copy()


# ------------------------------------------------------------
# Safe formatter in case multiple outcomes from one fixture
# ever satisfy the same reporting rule.
# ------------------------------------------------------------

def aggregate_watchlist_outcomes(
    frame: pd.DataFrame,
    classification_name: str,
) -> pd.DataFrame:

    if frame.empty:

        return pd.DataFrame(
            columns=[
                "OfficialHomeTeam",
                "OfficialAwayTeam",
                classification_name,
            ]
        )


    formatted = frame.copy()

    formatted[
        "_WatchlistLabel"
    ] = formatted.apply(
        lambda row: (
            f"{row['Outcome']} "
            f"({row['ModelMinusMarket']:+.1%})"
        ),
        axis=1,
    )


    aggregated = (
        formatted
        .groupby(
            [
                "OfficialHomeTeam",
                "OfficialAwayTeam",
            ],
            as_index=False,
            sort=False,
        )[
            "_WatchlistLabel"
        ]
        .agg(
            " | ".join
        )
        .rename(
            columns={
                "_WatchlistLabel": (
                    classification_name
                )
            }
        )
    )

    return aggregated


primary_fixture_watchlist = (
    aggregate_watchlist_outcomes(
        primary_watchlist_rows,
        "PrimaryWatchlist",
    )
)


longshot_fixture_watchlist = (
    aggregate_watchlist_outcomes(
        longshot_watchlist_rows,
        "LongshotSensitive",
    )
)


final_opening_forecasts = (
    final_opening_forecasts
    .merge(
        primary_fixture_watchlist,
        on=[
            "OfficialHomeTeam",
            "OfficialAwayTeam",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        longshot_fixture_watchlist,
        on=[
            "OfficialHomeTeam",
            "OfficialAwayTeam",
        ],
        how="left",
        validate="one_to_one",
    )
)


final_opening_forecasts[
    "PrimaryWatchlist"
] = (
    final_opening_forecasts[
        "PrimaryWatchlist"
    ]
    .fillna("")
)


final_opening_forecasts[
    "LongshotSensitive"
] = (
    final_opening_forecasts[
        "LongshotSensitive"
    ]
    .fillna("")
)


final_opening_forecasts[
    "HasPrimaryWatchlistOutcome"
] = (
    final_opening_forecasts[
        "PrimaryWatchlist"
    ] != ""
)


final_opening_forecasts[
    "HasLongshotSensitiveOutcome"
] = (
    final_opening_forecasts[
        "LongshotSensitive"
    ] != ""
)


# ============================================================
# D. ADD HUMAN-READABLE MODEL-MARKET SUMMARY
# ============================================================

def largest_disagreement_description(
    row,
):

    differences = {
        "Home": row[
            "ModelMinusMarket_H"
        ],
        "Draw": row[
            "ModelMinusMarket_D"
        ],
        "Away": row[
            "ModelMinusMarket_A"
        ],
    }

    outcome = max(
        differences,
        key=lambda key: abs(
            differences[key]
        ),
    )

    difference = differences[
        outcome
    ]

    direction = (
        "Model higher"
        if difference > 0
        else "Market higher"
    )

    return (
        f"{outcome}: "
        f"{direction} "
        f"({difference:+.1%})"
    )


final_opening_forecasts[
    "LargestDisagreement"
] = (
    final_opening_forecasts.apply(
        largest_disagreement_description,
        axis=1,
    )
)


# ============================================================
# E. FINAL COLUMN ORDER
# ============================================================

final_column_order = [
    "FixtureNumber",
    "Season",
    "Kickoff",

    "OfficialHomeTeam",
    "OfficialAwayTeam",

    "HomeExpectedGoals",
    "AwayExpectedGoals",
    "ExpectedTotalGoals",
    "ExpectedGoalDifference",

    "Probability_H",
    "Probability_D",
    "Probability_A",

    "ModelFavourite",
    "MostLikelyOutcomeProbability",
    "Confidence",
    "FavouriteProbabilityMargin",
    "NormalisedEntropy",

    "ModalScoreline",
    "ModalScoreProbability",

    "MarketOdds_H",
    "MarketOdds_D",
    "MarketOdds_A",

    "MarketProbability_H",
    "MarketProbability_D",
    "MarketProbability_A",

    "MarketFavourite",
    "ModelMarketFavouriteAgree",

    "ModelMinusMarket_H",
    "ModelMinusMarket_D",
    "ModelMinusMarket_A",

    "TotalVariationDistance",
    "LargestDisagreement",

    "HasPrimaryWatchlistOutcome",
    "PrimaryWatchlist",

    "HasLongshotSensitiveOutcome",
    "LongshotSensitive",
]


final_opening_forecasts = (
    final_opening_forecasts[
        final_column_order
    ]
    .sort_values(
        by=[
            "Kickoff",
            "FixtureNumber",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ============================================================
# F. FINAL VALIDATION
# ============================================================

assert len(
    final_opening_forecasts
) == 10


assert not final_opening_forecasts.duplicated(
    subset=[
        "OfficialHomeTeam",
        "OfficialAwayTeam",
    ]
).any()


# ------------------------------------------------------------
# Model probability identities
# ------------------------------------------------------------

model_probability_sum = (
    final_opening_forecasts[
        [
            "Probability_H",
            "Probability_D",
            "Probability_A",
        ]
    ]
    .sum(axis=1)
)


market_probability_sum = (
    final_opening_forecasts[
        [
            "MarketProbability_H",
            "MarketProbability_D",
            "MarketProbability_A",
        ]
    ]
    .sum(axis=1)
)


maximum_model_probability_sum_error = float(
    np.max(
        np.abs(
            model_probability_sum
            - 1.0
        )
    )
)


maximum_market_probability_sum_error = float(
    np.max(
        np.abs(
            market_probability_sum
            - 1.0
        )
    )
)


assert (
    maximum_model_probability_sum_error
    < 1e-10
)

assert (
    maximum_market_probability_sum_error
    < 1e-12
)


# ------------------------------------------------------------
# Watchlist reconciliation
# ------------------------------------------------------------

primary_outcome_count = int(
    (
        watchlist[
            "WatchlistClassification"
        ]
        == "Primary watchlist"
    ).sum()
)

longshot_outcome_count = int(
    (
        watchlist[
            "WatchlistClassification"
        ]
        == "Longshot-sensitive"
    ).sum()
)


assert primary_outcome_count == 6
assert longshot_outcome_count == 1


assert int(
    final_opening_forecasts[
        "HasPrimaryWatchlistOutcome"
    ].sum()
) <= primary_outcome_count


assert int(
    final_opening_forecasts[
        "HasLongshotSensitiveOutcome"
    ].sum()
) <= longshot_outcome_count


# ------------------------------------------------------------
# Finite numeric checks
# ------------------------------------------------------------

required_numeric_columns = [
    "HomeExpectedGoals",
    "AwayExpectedGoals",
    "Probability_H",
    "Probability_D",
    "Probability_A",
    "MarketProbability_H",
    "MarketProbability_D",
    "MarketProbability_A",
    "ModelMinusMarket_H",
    "ModelMinusMarket_D",
    "ModelMinusMarket_A",
    "TotalVariationDistance",
]


assert np.isfinite(
    final_opening_forecasts[
        required_numeric_columns
    ]
    .to_numpy(
        dtype=float
    )
).all()


# ============================================================
# G. EXPORT FINAL NUMERIC TABLE
# ============================================================

final_opening_forecasts.to_csv(
    final_forecast_output_path,
    index=False,
)


# ============================================================
# H. CREATE PRESENTATION-READY DISPLAY
# ============================================================

final_forecast_display = (
    final_opening_forecasts[
        [
            "Kickoff",
            "OfficialHomeTeam",
            "OfficialAwayTeam",

            "HomeExpectedGoals",
            "AwayExpectedGoals",

            "Probability_H",
            "Probability_D",
            "Probability_A",

            "ModelFavourite",
            "Confidence",

            "MarketProbability_H",
            "MarketProbability_D",
            "MarketProbability_A",

            "MarketFavourite",

            "TotalVariationDistance",

            "PrimaryWatchlist",
            "LongshotSensitive",
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# Compact numeric formatting
# ------------------------------------------------------------

for column in [
    "HomeExpectedGoals",
    "AwayExpectedGoals",
]:

    final_forecast_display[
        column
    ] = final_forecast_display[
        column
    ].round(2)


for column in [
    "Probability_H",
    "Probability_D",
    "Probability_A",
    "MarketProbability_H",
    "MarketProbability_D",
    "MarketProbability_A",
]:

    final_forecast_display[
        column
    ] = (
        final_forecast_display[
            column
        ]
        * 100
    ).round(1)


final_forecast_display[
    "TotalVariationDistance"
] = (
    final_forecast_display[
        "TotalVariationDistance"
    ]
    .round(3)
)


final_forecast_display = (
    final_forecast_display
    .rename(
        columns={
            "HomeExpectedGoals": "Home xG",
            "AwayExpectedGoals": "Away xG",

            "Probability_H": "Model H %",
            "Probability_D": "Model D %",
            "Probability_A": "Model A %",

            "MarketProbability_H": "Market H %",
            "MarketProbability_D": "Market D %",
            "MarketProbability_A": "Market A %",

            "TotalVariationDistance": "TV Distance",

            "PrimaryWatchlist": (
                "Primary Disagreement"
            ),

            "LongshotSensitive": (
                "Longshot-Sensitive"
            ),
        }
    )
)


# ============================================================
# I. SUMMARY REPORT
# ============================================================

final_table_summary = pd.DataFrame(
    [
        {
            "Metric": "Fixtures",
            "Value": len(
                final_opening_forecasts
            ),
        },
        {
            "Metric": (
                "Model/market favourite agreements"
            ),
            "Value": int(
                final_opening_forecasts[
                    "ModelMarketFavouriteAgree"
                ].sum()
            ),
        },
        {
            "Metric": (
                "Primary watchlist outcomes"
            ),
            "Value": primary_outcome_count,
        },
        {
            "Metric": (
                "Longshot-sensitive outcomes"
            ),
            "Value": longshot_outcome_count,
        },
        {
            "Metric": (
                "Strong-confidence fixtures"
            ),
            "Value": int(
                (
                    final_opening_forecasts[
                        "Confidence"
                    ]
                    == "Strong"
                ).sum()
            ),
        },
        {
            "Metric": (
                "Moderate-confidence fixtures"
            ),
            "Value": int(
                (
                    final_opening_forecasts[
                        "Confidence"
                    ]
                    == "Moderate"
                ).sum()
            ),
        },
        {
            "Metric": (
                "Low-confidence fixtures"
            ),
            "Value": int(
                (
                    final_opening_forecasts[
                        "Confidence"
                    ]
                    == "Low"
                ).sum()
            ),
        },
    ]
)


final_opening_forecast_table_ready = True


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print(
    "Final opening-round forecast summary:"
)
display(
    final_table_summary
)

print(
    "\nFinal presentation-ready opening-round forecasts:"
)
display(
    final_forecast_display
)

print(
    "\nFinal numeric forecast file:",
    final_forecast_output_path
    .relative_to(
        project_root
    )
    .as_posix(),
)

print(
    "\nFinal opening-round forecast table status: VALID"
)

print(
    "Presentation-ready forecast table:",
    final_opening_forecast_table_ready,
)

Final opening-round forecast summary:


,Metric,Value
0,Fixtures,10
1,Model/market favourite agreements,9
2,Primary watchlist outcomes,6
3,Longshot-sensitive outcomes,1
4,Strong-confidence fixtures,2
5,Moderate-confidence fixtures,4
6,Low-confidence fixtures,4



Final presentation-ready opening-round forecasts:


,Kickoff,OfficialHomeTeam,OfficialAwayTeam,Home xG,Away xG,Model H %,Model D %,Model A %,ModelFavourite,Confidence,Market H %,Market D %,Market A %,MarketFavourite,TV Distance,Primary Disagreement,Longshot-Sensitive
0,2026-08-21 20:00:00,Arsenal,Coventry City,2.47,0.95,70.6,16.8,12.6,Arsenal,Strong,82.2,12.8,5.0,Arsenal,0.116,,Coventry City (+7.5%)
1,2026-08-22 12:30:00,Hull City,Manchester United,1.01,2.04,17.9,20.8,61.3,Manchester United,Strong,13.1,21.0,65.9,Manchester United,0.049,,
2,2026-08-22 15:00:00,Everton,Crystal Palace,1.49,1.41,39.3,24.7,35.9,Everton,Low,44.9,26.8,28.4,Everton,0.076,Crystal Palace (+7.6%),
3,2026-08-22 15:00:00,Ipswich Town,Sunderland,1.17,1.62,27.6,24.6,47.8,Sunderland,Moderate,34.9,28.2,36.9,Sunderland,0.109,Sunderland (+10.9%),
4,2026-08-22 15:00:00,Nottingham Forest,Leeds United,1.55,1.35,42.0,24.7,33.4,Nottingham Forest,Low,43.0,26.8,30.2,Nottingham Forest,0.032,,
5,2026-08-22 17:30:00,Brentford,Tottenham Hotspur,1.69,1.24,48.1,24.0,28.0,Brentford,Moderate,40.6,26.5,32.9,Brentford,0.075,Brentford (+7.5%),
6,2026-08-23 14:00:00,Brighton & Hove Albion,Aston Villa,1.42,1.60,34.0,24.1,41.8,Aston Villa,Low,42.2,25.5,32.3,Brighton & Hove Albion,0.095,Aston Villa (+9.5%),
7,2026-08-23 14:00:00,Manchester City,AFC Bournemouth,2.02,1.22,55.8,21.4,22.8,Manchester City,Moderate,66.8,18.4,14.8,Manchester City,0.110,AFC Bournemouth (+7.9%),
8,2026-08-23 16:30:00,Newcastle United,Liverpool,1.34,1.69,30.6,23.8,45.6,Liverpool,Moderate,28.5,24.6,46.9,Liverpool,0.021,,
9,2026-08-24 20:00:00,Fulham,Chelsea,1.44,1.50,36.4,24.6,39.0,Chelsea,Low,30.3,25.5,44.1,Chelsea,0.061,Fulham (+6.1%),



Final numeric forecast file: outputs/forecasts/2026_27/opening_round_final_forecast_table.csv

Final opening-round forecast table status: VALID
Presentation-ready forecast table: True


### Results and Interpretation

The final opening-round forecast table consolidates the production-model probabilities, expected goals, uncertainty diagnostics and margin-adjusted market comparison into a single reporting output.

All ten opening fixtures are represented successfully. The model and market agree on the most likely outcome in nine of the ten matches, although agreement on the favourite does not imply that the underlying probability distributions are identical.

Forecast confidence is deliberately based on both the probability of the most likely outcome and its separation from the second-most likely outcome:

- `Strong`: favourite probability of at least 60% and a lead of at least 25 percentage points;
- `Moderate`: favourite probability of at least 45% and a lead of at least 12 percentage points;
- `Low`: all remaining forecasts.

Under these descriptive thresholds, two fixtures are classified as Strong confidence, four as Moderate and four as Low. This reinforces that a substantial proportion of the opening round remains probabilistically uncertain despite every fixture having a single most likely outcome.

Arsenal against Coventry is the clearest model forecast, with Arsenal assigned approximately 70.6% probability and expected goals of 2.47 versus 0.95. Manchester United away at Hull is the other Strong-confidence forecast, with approximately 61.3% probability assigned to an away win.

At the other end of the confidence spectrum, fixtures such as Everton–Crystal Palace remain comparatively diffuse across the three possible outcomes. These matches are better characterised as uncertain probability distributions than as strong categorical predictions.

The market comparison also shows that headline agreement can conceal meaningful differences in probability magnitude. Nine model and market favourites agree, but several fixtures still display notable total variation distance between the complete `(H,D,A)` distributions.

Six individual outcomes satisfy the primary model–market disagreement criteria and one additional outcome is classified as longshot-sensitive. These labels identify where the production model assigns materially more probability than the current margin-adjusted market; they are not interpreted as demonstrated profitable betting opportunities.

The final table therefore provides three distinct pieces of information for each fixture: what the model expects to happen, how uncertain that forecast is, and how the model's probability distribution differs from the current market.

The complete numerical output has been exported to `outputs/forecasts/2026_27/opening_round_final_forecast_table.csv`.

## 10. Define the Sequential 2026–27 Forecasting Protocol

The opening-round forecasts represent a pre-season snapshot. Subsequent Premier League forecasts cannot be generated correctly using the same static feature state because many predictors evolve after each completed match.

A reproducible sequential forecasting protocol is therefore established for the remainder of 2026–27.

Completed league results will be stored separately from the original fixture schedule. Before a future forecast is generated, the current-season result file must be refreshed and validated against the canonical 380-fixture schedule.

The forecasting process follows four principles:

1. only completed matches may update the feature state;
2. Elo, rolling form, rest, congestion and league-table variables must be reconstructed using information available strictly before the target fixture;
3. the frozen preprocessing objects and Poisson estimators are never refitted during the season;
4. forecasts are regenerated from the updated historical state rather than manually modifying individual feature values.

A fixture is placed in the current forecasting queue only when it is the next unplayed league fixture for both participating clubs. This provides a practical scheduling frontier while preventing later fixtures from being forecast using assumed intervening results.

Schedule changes and postponed fixtures require the canonical fixture list to be refreshed before the affected forecast is produced.

This section creates the current-season results contract, validates any results already available, constructs the forecasting queue and exports a machine-readable protocol for subsequent forecast updates.

In [12]:
# ============================================================
# 10. Define the Sequential 2026-27 Forecasting Protocol
# ============================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ------------------------------------------------------------
# Preconditions
# ------------------------------------------------------------

assert "final_opening_forecast_table_ready" in globals()
assert final_opening_forecast_table_ready is True

assert len(fixture_schedule) == 380
assert len(production_feature_columns) == 70

TARGET_SEASON = "2026-27"


# ------------------------------------------------------------
# Locate project
# ------------------------------------------------------------

project_root = Path.cwd().resolve()

while (
    project_root.name != "premier-league-probability-engine"
    and project_root.parent != project_root
):
    project_root = project_root.parent

assert project_root.name == "premier-league-probability-engine"


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

current_season_results_path = (
    project_root
    / "data"
    / "raw"
    / "premier_league_results_2026_27.csv"
)

forecast_directory = (
    project_root
    / "outputs"
    / "forecasts"
    / "2026_27"
)

forecast_directory.mkdir(
    parents=True,
    exist_ok=True,
)

forecast_queue_path = (
    forecast_directory
    / "current_forecast_queue.csv"
)

protocol_path = (
    forecast_directory
    / "sequential_forecasting_protocol.json"
)


# ============================================================
# A. CURRENT-SEASON RESULT CONTRACT
# ============================================================

RESULT_COLUMNS = [
    "FixtureNumber",
    "ActualKickoff",
    "HomeTeam",
    "AwayTeam",
    "FTHG",
    "FTAG",
    "FTR",
]


# ------------------------------------------------------------
# Create the empty season-result file on first run
# ------------------------------------------------------------

if not current_season_results_path.is_file():

    empty_results = pd.DataFrame(
        columns=RESULT_COLUMNS
    )

    empty_results.to_csv(
        current_season_results_path,
        index=False,
    )


current_season_results = pd.read_csv(
    current_season_results_path,
    low_memory=False,
)


missing_result_columns = [
    column
    for column in RESULT_COLUMNS
    if column not in current_season_results.columns
]

assert not missing_result_columns, (
    "Current-season result file is missing columns: "
    f"{missing_result_columns}"
)


current_season_results = (
    current_season_results[
        RESULT_COLUMNS
    ]
    .copy()
)


# ============================================================
# B. VALIDATE ANY COMPLETED RESULTS
# ============================================================

def validate_completed_results(
    results: pd.DataFrame,
    schedule: pd.DataFrame,
) -> pd.DataFrame:

    validated = results.copy()


    # --------------------------------------------------------
    # Empty season is valid before opening day
    # --------------------------------------------------------

    if validated.empty:

        validated["FixtureNumber"] = pd.Series(
            dtype="int64"
        )

        validated["ActualKickoff"] = pd.Series(
            dtype="datetime64[ns]"
        )

        validated["FTHG"] = pd.Series(
            dtype="int64"
        )

        validated["FTAG"] = pd.Series(
            dtype="int64"
        )

        return validated


    # --------------------------------------------------------
    # Basic missingness
    # --------------------------------------------------------

    required_non_missing = [
        "FixtureNumber",
        "ActualKickoff",
        "HomeTeam",
        "AwayTeam",
        "FTHG",
        "FTAG",
        "FTR",
    ]

    missing_counts = (
        validated[
            required_non_missing
        ]
        .isna()
        .sum()
    )

    assert (
        missing_counts == 0
    ).all(), (
        "Completed-result rows contain missing values."
    )


    # --------------------------------------------------------
    # Types
    # --------------------------------------------------------

    validated[
        "FixtureNumber"
    ] = pd.to_numeric(
        validated[
            "FixtureNumber"
        ],
        errors="raise",
    ).astype(int)


    validated[
        "FTHG"
    ] = pd.to_numeric(
        validated[
            "FTHG"
        ],
        errors="raise",
    ).astype(int)


    validated[
        "FTAG"
    ] = pd.to_numeric(
        validated[
            "FTAG"
        ],
        errors="raise",
    ).astype(int)


    validated[
        "ActualKickoff"
    ] = pd.to_datetime(
        validated[
            "ActualKickoff"
        ],
        errors="raise",
    )


    # --------------------------------------------------------
    # Plausibility
    # --------------------------------------------------------

    assert (
        validated[
            "FixtureNumber"
        ].between(
            1,
            380,
        )
    ).all()

    assert (
        validated[
            "FTHG"
        ] >= 0
    ).all()

    assert (
        validated[
            "FTAG"
        ] >= 0
    ).all()

    assert (
        validated[
            "FTR"
        ].isin(
            [
                "H",
                "D",
                "A",
            ]
        )
    ).all()


    # --------------------------------------------------------
    # Result label must agree with goals
    # --------------------------------------------------------

    derived_result = np.select(
        [
            (
                validated[
                    "FTHG"
                ]
                >
                validated[
                    "FTAG"
                ]
            ),
            (
                validated[
                    "FTHG"
                ]
                ==
                validated[
                    "FTAG"
                ]
            ),
        ],
        [
            "H",
            "D",
        ],
        default="A",
    )


    assert np.array_equal(
        derived_result,
        validated[
            "FTR"
        ].to_numpy(),
    ), (
        "At least one FTR label does not agree "
        "with the recorded score."
    )


    # --------------------------------------------------------
    # No duplicated completed fixtures
    # --------------------------------------------------------

    assert not validated.duplicated(
        subset=[
            "FixtureNumber",
        ]
    ).any(), (
        "A completed fixture appears more than once."
    )


    # --------------------------------------------------------
    # Every result must correspond to the canonical schedule
    # --------------------------------------------------------

    schedule_identity = schedule[
        [
            "FixtureNumber",
            "HomeTeam",
            "AwayTeam",
        ]
    ].copy()


    identity_check = (
        validated[
            [
                "FixtureNumber",
                "HomeTeam",
                "AwayTeam",
            ]
        ]
        .merge(
            schedule_identity,
            on=[
                "FixtureNumber",
                "HomeTeam",
                "AwayTeam",
            ],
            how="left",
            indicator=True,
            validate="one_to_one",
        )
    )


    assert (
        identity_check[
            "_merge"
        ]
        == "both"
    ).all(), (
        "At least one completed result does not match "
        "the canonical fixture schedule."
    )


    # --------------------------------------------------------
    # Chronological ordering
    # --------------------------------------------------------

    validated = (
        validated
        .sort_values(
            by=[
                "ActualKickoff",
                "FixtureNumber",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    return validated


current_season_results = (
    validate_completed_results(
        current_season_results,
        fixture_schedule,
    )
)


completed_fixture_numbers = set(
    current_season_results[
        "FixtureNumber"
    ].tolist()
)

completed_fixture_count = len(
    completed_fixture_numbers
)

assert completed_fixture_count <= 380


# ============================================================
# C. BUILD CURRENT UNPLAYED SCHEDULE
# ============================================================

pending_fixtures = (
    fixture_schedule.loc[
        ~fixture_schedule[
            "FixtureNumber"
        ].isin(
            completed_fixture_numbers
        )
    ]
    .copy()
)


pending_fixtures = (
    pending_fixtures
    .sort_values(
        by=[
            "Kickoff",
            "SourceOrder",
            "FixtureNumber",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


assert (
    len(pending_fixtures)
    + completed_fixture_count
    == 380
)


# ============================================================
# D. FIND EACH CLUB'S NEXT UNPLAYED FIXTURE
# ============================================================

next_fixture_by_team = {}


for row in pending_fixtures.itertuples():

    if (
        row.HomeTeam
        not in next_fixture_by_team
    ):
        next_fixture_by_team[
            row.HomeTeam
        ] = row.FixtureNumber

    if (
        row.AwayTeam
        not in next_fixture_by_team
    ):
        next_fixture_by_team[
            row.AwayTeam
        ] = row.FixtureNumber


# ------------------------------------------------------------
# A fixture enters the queue only when it is the next
# scheduled unplayed league fixture for BOTH clubs.
# ------------------------------------------------------------

def is_next_fixture_for_both(
    row,
) -> bool:

    home_next = (
        next_fixture_by_team.get(
            row.HomeTeam
        )
    )

    away_next = (
        next_fixture_by_team.get(
            row.AwayTeam
        )
    )

    return (
        row.FixtureNumber
        == home_next
        == away_next
    )


if pending_fixtures.empty:

    pending_fixtures[
        "NextForBothClubs"
    ] = pd.Series(
        dtype=bool
    )

else:

    pending_fixtures[
        "NextForBothClubs"
    ] = (
        pending_fixtures.apply(
            is_next_fixture_for_both,
            axis=1,
        )
    )


forecast_queue = (
    pending_fixtures.loc[
        pending_fixtures[
            "NextForBothClubs"
        ]
    ][
        [
            "FixtureNumber",
            "Season",
            "Kickoff",
            "OfficialHomeTeam",
            "OfficialAwayTeam",
            "HomeTeam",
            "AwayTeam",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


forecast_queue[
    "QueueStatus"
] = (
    "READY_FOR_STATE_REFRESH"
)


forecast_queue[
    "RequiredAction"
] = (
    "Refresh completed results, reconstruct pre-match "
    "features, then apply frozen production model."
)


# ============================================================
# E. QUEUE VALIDATION
# ============================================================

queued_teams = pd.concat(
    [
        forecast_queue[
            "HomeTeam"
        ],
        forecast_queue[
            "AwayTeam"
        ],
    ],
    ignore_index=True,
)


# A club cannot appear twice in the current frontier.
assert not queued_teams.duplicated().any(), (
    "A club appears in more than one current queue fixture."
)


assert not forecast_queue[
    "FixtureNumber"
].isin(
    completed_fixture_numbers
).any()


# ------------------------------------------------------------
# Before opening day, the queue must reproduce the ten
# validated opening fixtures.
# ------------------------------------------------------------

if completed_fixture_count == 0:

    assert len(
        forecast_queue
    ) == 10

    assert set(
        forecast_queue[
            "FixtureNumber"
        ]
    ) == set(
        opening_fixture_numbers
    )


# ============================================================
# F. STORE THE SEQUENTIAL FORECASTING CONTRACT
# ============================================================

protocol = {
    "Season": TARGET_SEASON,

    "ProtocolVersion": (
        "2026-27-sequential-v1"
    ),

    "FrozenModelArtifact": (
        "outputs/production_model/"
        "production_model_bundle.joblib"
    ),

    "CanonicalFixtureSchedule": (
        "data/raw/"
        "premier_league_fixtures_2026_27_official.csv"
    ),

    "CurrentSeasonResults": (
        "data/raw/"
        "premier_league_results_2026_27.csv"
    ),

    "FrozenFeatureCount": 70,

    "ProbabilityOrder": [
        "H",
        "D",
        "A",
    ],

    "Rules": [
        (
            "Only completed 2026-27 league matches may "
            "update the production feature state."
        ),
        (
            "Current-season results must match the "
            "canonical fixture identity before use."
        ),
        (
            "For every target fixture, all dynamic "
            "features must be calculated from information "
            "available strictly before that fixture."
        ),
        (
            "Elo must be updated only after a completed "
            "match and the pre-update rating must be used "
            "for that match's prediction."
        ),
        (
            "Rolling-form and venue-form windows must use "
            "completed previous matches only."
        ),
        (
            "League-table variables must describe the "
            "pre-match table state and must not contain "
            "the target fixture result."
        ),
        (
            "The fitted imputer, scaler and Poisson "
            "estimators remain frozen throughout 2026-27."
        ),
        (
            "If a fixture is postponed or rescheduled, "
            "the canonical schedule must be refreshed "
            "before its production forecast is generated."
        ),
    ],

    "OperationalSequence": [
        "Refresh official fixture schedule if required.",
        "Append newly completed league results.",
        "Validate completed results against schedule.",
        "Reconstruct the current pre-match feature state.",
        "Identify each club's next unplayed fixture.",
        "Generate predictions using frozen production artefacts.",
        "Optionally refresh current bookmaker prices.",
        "Export a timestamped forecast snapshot.",
    ],

    "Interpretation": (
        "Future forecasts are sequential pre-match "
        "probabilities, not unconditional predictions "
        "made before the season."
    ),
}


protocol_path.write_text(
    json.dumps(
        protocol,
        indent=4,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)


forecast_queue.to_csv(
    forecast_queue_path,
    index=False,
)


# ============================================================
# G. VALIDATION REPORT
# ============================================================

remaining_fixture_count = len(
    pending_fixtures
)

queue_fixture_count = len(
    forecast_queue
)


protocol_validation = pd.DataFrame(
    [
        {
            "Validation": (
                "Canonical season fixtures"
            ),
            "Value": 380,
            "Expected": 380,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Completed + pending fixtures"
            ),
            "Value": (
                completed_fixture_count
                + remaining_fixture_count
            ),
            "Expected": 380,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Duplicate completed fixtures"
            ),
            "Value": int(
                current_season_results
                .duplicated(
                    subset=[
                        "FixtureNumber"
                    ]
                )
                .sum()
            ),
            "Expected": 0,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Duplicate clubs in queue"
            ),
            "Value": int(
                queued_teams
                .duplicated()
                .sum()
            ),
            "Expected": 0,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Frozen feature count"
            ),
            "Value": len(
                production_feature_columns
            ),
            "Expected": 70,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Probability order"
            ),
            "Value": (
                PROBABILITY_ORDER
            ),
            "Expected": [
                "H",
                "D",
                "A",
            ],
            "Status": "PASS",
        },
    ]
)


season_update_profile = pd.DataFrame(
    [
        {
            "Metric": (
                "Completed 2026-27 fixtures"
            ),
            "Value": (
                completed_fixture_count
            ),
        },
        {
            "Metric": (
                "Remaining fixtures"
            ),
            "Value": (
                remaining_fixture_count
            ),
        },
        {
            "Metric": (
                "Current queue fixtures"
            ),
            "Value": (
                queue_fixture_count
            ),
        },
        {
            "Metric": (
                "Frozen production predictors"
            ),
            "Value": 70,
        },
        {
            "Metric": (
                "Production model refitted in-season"
            ),
            "Value": False,
        },
        {
            "Metric": (
                "Protocol version"
            ),
            "Value": (
                protocol[
                    "ProtocolVersion"
                ]
            ),
        },
    ]
)


sequential_forecasting_protocol_ready = True


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print(
    "2026-27 sequential forecasting profile:"
)
display(
    season_update_profile
)

print(
    "\nSequential-protocol validation:"
)
display(
    protocol_validation
)

print(
    "\nCurrent forecasting queue:"
)
display(
    forecast_queue
)

print(
    "\nCurrent-season results file:",
    current_season_results_path
    .relative_to(project_root)
    .as_posix(),
)

print(
    "Forecast queue file:",
    forecast_queue_path
    .relative_to(project_root)
    .as_posix(),
)

print(
    "Sequential protocol file:",
    protocol_path
    .relative_to(project_root)
    .as_posix(),
)

print(
    "\nSequential forecasting protocol status: VALID"
)

print(
    "Live-season update protocol ready:",
    sequential_forecasting_protocol_ready,
)

2026-27 sequential forecasting profile:


,Metric,Value
0,Completed 2026-27 fixtures,0
1,Remaining fixtures,380
2,Current queue fixtures,10
3,Frozen production predictors,70
4,Production model refitted in-season,False
5,Protocol version,2026-27-sequential-v1



Sequential-protocol validation:


,Validation,Value,Expected,Status
0,Canonical season fixtures,380,380,PASS
1,Completed + pending fixtures,380,380,PASS
2,Duplicate completed fixtures,0,0,PASS
3,Duplicate clubs in queue,0,0,PASS
4,Frozen feature count,70,70,PASS
5,Probability order,"[H, D, A]","[H, D, A]",PASS



Current forecasting queue:


,FixtureNumber,Season,Kickoff,OfficialHomeTeam,OfficialAwayTeam,HomeTeam,AwayTeam,QueueStatus,RequiredAction
0,1,2026-27,2026-08-21 20:00:00,Arsenal,Coventry City,Arsenal,Coventry,READY_FOR_STATE_REFRESH,"Refresh completed results, reconstruct pre-mat..."
1,2,2026-27,2026-08-22 12:30:00,Hull City,Manchester United,Hull,Man United,READY_FOR_STATE_REFRESH,"Refresh completed results, reconstruct pre-mat..."
2,3,2026-27,2026-08-22 15:00:00,Everton,Crystal Palace,Everton,Crystal Palace,READY_FOR_STATE_REFRESH,"Refresh completed results, reconstruct pre-mat..."
3,4,2026-27,2026-08-22 15:00:00,Ipswich Town,Sunderland,Ipswich,Sunderland,READY_FOR_STATE_REFRESH,"Refresh completed results, reconstruct pre-mat..."
4,5,2026-27,2026-08-22 15:00:00,Nottingham Forest,Leeds United,Nott'm Forest,Leeds,READY_FOR_STATE_REFRESH,"Refresh completed results, reconstruct pre-mat..."
5,6,2026-27,2026-08-22 17:30:00,Brentford,Tottenham Hotspur,Brentford,Tottenham,READY_FOR_STATE_REFRESH,"Refresh completed results, reconstruct pre-mat..."
6,7,2026-27,2026-08-23 14:00:00,Brighton & Hove Albion,Aston Villa,Brighton,Aston Villa,READY_FOR_STATE_REFRESH,"Refresh completed results, reconstruct pre-mat..."
7,8,2026-27,2026-08-23 14:00:00,Manchester City,AFC Bournemouth,Man City,Bournemouth,READY_FOR_STATE_REFRESH,"Refresh completed results, reconstruct pre-mat..."
8,9,2026-27,2026-08-23 16:30:00,Newcastle United,Liverpool,Newcastle,Liverpool,READY_FOR_STATE_REFRESH,"Refresh completed results, reconstruct pre-mat..."
9,10,2026-27,2026-08-24 20:00:00,Fulham,Chelsea,Fulham,Chelsea,READY_FOR_STATE_REFRESH,"Refresh completed results, reconstruct pre-mat..."



Current-season results file: data/raw/premier_league_results_2026_27.csv
Forecast queue file: outputs/forecasts/2026_27/current_forecast_queue.csv
Sequential protocol file: outputs/forecasts/2026_27/sequential_forecasting_protocol.json

Sequential forecasting protocol status: VALID
Live-season update protocol ready: True


## 11. Build the Live-Season Update and Forecast Runner

The sequential forecasting protocol is now implemented as a reusable live-season runner.

The runner reconstructs the complete 70-feature production state from the canonical 2026–27 fixture schedule and the set of completed league results available at the time of execution.

For every fixture on the current forecasting frontier, it independently reconstructs:

- pre-match Elo ratings;
- five-match general form;
- five-match venue-specific form;
- rest and short-rest indicators;
- seven- and fourteen-day fixture congestion;
- team-specific match number and season progress;
- pre-match league points, goal difference and position;
- league-position category indicators;
- all required home-minus-away matchup differences.

All current-season dynamic variables are calculated strictly from completed matches before the target fixture. The production imputer, scaler and Poisson estimators remain frozen and are never refitted.

The runner then applies the frozen production package to the reconstructed feature matrix and exports both a current forecast file and a timestamped forecast snapshot.

Before the season begins, the reconstructed state is additionally checked against the previously validated opening-round feature matrix and forecasts. This provides an end-to-end regression test demonstrating that the live runner reproduces the existing production output before it is used on future results.

During the season, the operational workflow becomes: update the completed-results file, refresh the official fixture schedule where necessary, and rerun this section.

In [13]:
# ============================================================
# 11. Build the Live-Season Update and Forecast Runner
# ============================================================

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# A. PRECONDITIONS
# ============================================================

assert "sequential_forecasting_protocol_ready" in globals()
assert sequential_forecasting_protocol_ready is True

assert "production_package_ready" in globals()
assert production_package_ready is True

assert "predict_with_frozen_production_model" in globals()

assert len(production_feature_columns) == 70


# ============================================================
# B. PROJECT PATHS
# ============================================================

project_root = Path.cwd().resolve()

while (
    project_root.name != "premier-league-probability-engine"
    and project_root.parent != project_root
):
    project_root = project_root.parent

assert project_root.name == "premier-league-probability-engine"


schedule_path = (
    project_root
    / "data"
    / "raw"
    / "premier_league_fixtures_2026_27_official.csv"
)

current_results_path = (
    project_root
    / "data"
    / "raw"
    / "premier_league_results_2026_27.csv"
)

historical_results_path = (
    project_root
    / "data"
    / "raw"
    / "premier_league_goal_targets_2015_16_to_2025_26.csv"
)

forecast_directory = (
    project_root
    / "outputs"
    / "forecasts"
    / "2026_27"
)

forecast_directory.mkdir(
    parents=True,
    exist_ok=True,
)

current_queue_output_path = (
    forecast_directory
    / "current_forecast_queue.csv"
)

current_feature_output_path = (
    forecast_directory
    / "current_live_features.csv"
)

current_forecast_output_path = (
    forecast_directory
    / "current_live_forecasts.csv"
)


assert schedule_path.is_file()
assert current_results_path.is_file()
assert historical_results_path.is_file()


# ============================================================
# C. LOAD CANONICAL SCHEDULE
# ============================================================

live_schedule = pd.read_csv(
    schedule_path,
    low_memory=False,
).copy()


required_schedule_columns = [
    "FixtureNumber",
    "Season",
    "Kickoff",
    "OfficialHomeTeam",
    "OfficialAwayTeam",
    "HomeTeam",
    "AwayTeam",
]


missing_schedule_columns = [
    column
    for column in required_schedule_columns
    if column not in live_schedule.columns
]

assert not missing_schedule_columns, (
    "Canonical fixture schedule is missing columns: "
    f"{missing_schedule_columns}"
)


live_schedule[
    "FixtureNumber"
] = pd.to_numeric(
    live_schedule[
        "FixtureNumber"
    ],
    errors="raise",
).astype(int)


live_schedule[
    "Kickoff"
] = pd.to_datetime(
    live_schedule[
        "Kickoff"
    ],
    errors="raise",
)


live_schedule = (
    live_schedule
    .sort_values(
        by=[
            "Kickoff",
            "FixtureNumber",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


assert len(live_schedule) == 380

assert live_schedule[
    "FixtureNumber"
].is_unique


season_teams = sorted(
    set(
        live_schedule[
            "HomeTeam"
        ]
    )
    |
    set(
        live_schedule[
            "AwayTeam"
        ]
    )
)

assert len(season_teams) == 20


# ============================================================
# D. LOAD AND VALIDATE COMPLETED 2026-27 RESULTS
# ============================================================

live_results = pd.read_csv(
    current_results_path,
    low_memory=False,
).copy()


required_result_columns = [
    "FixtureNumber",
    "ActualKickoff",
    "HomeTeam",
    "AwayTeam",
    "FTHG",
    "FTAG",
    "FTR",
]


missing_result_columns = [
    column
    for column in required_result_columns
    if column not in live_results.columns
]

assert not missing_result_columns, (
    "Current result file is missing columns: "
    f"{missing_result_columns}"
)


live_results = live_results[
    required_result_columns
].copy()


if not live_results.empty:

    live_results[
        "FixtureNumber"
    ] = pd.to_numeric(
        live_results[
            "FixtureNumber"
        ],
        errors="raise",
    ).astype(int)


    live_results[
        "ActualKickoff"
    ] = pd.to_datetime(
        live_results[
            "ActualKickoff"
        ],
        errors="raise",
    )


    live_results[
        "FTHG"
    ] = pd.to_numeric(
        live_results[
            "FTHG"
        ],
        errors="raise",
    ).astype(int)


    live_results[
        "FTAG"
    ] = pd.to_numeric(
        live_results[
            "FTAG"
        ],
        errors="raise",
    ).astype(int)


    live_results[
        "FTR"
    ] = (
        live_results[
            "FTR"
        ]
        .astype(str)
        .str.strip()
        .str.upper()
    )


    assert live_results[
        "FTR"
    ].isin(
        [
            "H",
            "D",
            "A",
        ]
    ).all()


    assert (
        live_results[
            [
                "FTHG",
                "FTAG",
            ]
        ] >= 0
    ).all().all()


    derived_results = np.select(
        [
            (
                live_results[
                    "FTHG"
                ]
                >
                live_results[
                    "FTAG"
                ]
            ),
            (
                live_results[
                    "FTHG"
                ]
                ==
                live_results[
                    "FTAG"
                ]
            ),
        ],
        [
            "H",
            "D",
        ],
        default="A",
    )


    assert np.array_equal(
        derived_results,
        live_results[
            "FTR"
        ].to_numpy(),
    ), (
        "At least one FTR label disagrees with "
        "the recorded score."
    )


    assert not live_results.duplicated(
        subset=[
            "FixtureNumber"
        ]
    ).any()


    result_identity_check = (
        live_results[
            [
                "FixtureNumber",
                "HomeTeam",
                "AwayTeam",
            ]
        ]
        .merge(
            live_schedule[
                [
                    "FixtureNumber",
                    "HomeTeam",
                    "AwayTeam",
                ]
            ],
            on=[
                "FixtureNumber",
                "HomeTeam",
                "AwayTeam",
            ],
            how="left",
            indicator=True,
            validate="one_to_one",
        )
    )


    assert (
        result_identity_check[
            "_merge"
        ]
        == "both"
    ).all(), (
        "At least one result does not match "
        "the canonical 2026-27 fixture schedule."
    )


    live_results = (
        live_results
        .sort_values(
            by=[
                "ActualKickoff",
                "FixtureNumber",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


# ============================================================
# E. REPLAY HISTORICAL ELO THROUGH 2025-26
# ============================================================

INITIAL_ELO = 1500.0
ELO_K_FACTOR = 20.0
ELO_HOME_ADVANTAGE = 50.0
ELO_SCALE = 400.0


historical_results = pd.read_csv(
    historical_results_path,
    low_memory=False,
).copy()


historical_results[
    "Date"
] = pd.to_datetime(
    historical_results[
        "Date"
    ],
    errors="raise",
).dt.normalize()


historical_results[
    "_ReplayOrder"
] = np.arange(
    len(historical_results)
)


historical_results = (
    historical_results
    .sort_values(
        by=[
            "Date",
            "_ReplayOrder",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


assert len(historical_results) == 4180


def expected_home_elo_score(
    home_rating: float,
    away_rating: float,
) -> float:

    return 1.0 / (
        1.0
        + 10.0 ** (
            (
                away_rating
                - (
                    home_rating
                    + ELO_HOME_ADVANTAGE
                )
            )
            / ELO_SCALE
        )
    )


historical_end_elo = {}


for row in historical_results.itertuples(
    index=False
):

    home_rating = historical_end_elo.get(
        row.HomeTeam,
        INITIAL_ELO,
    )

    away_rating = historical_end_elo.get(
        row.AwayTeam,
        INITIAL_ELO,
    )


    expected_home = expected_home_elo_score(
        home_rating,
        away_rating,
    )


    if row.FTR == "H":
        actual_home = 1.0

    elif row.FTR == "D":
        actual_home = 0.5

    elif row.FTR == "A":
        actual_home = 0.0

    else:
        raise ValueError(
            f"Unexpected historical result: {row.FTR}"
        )


    change = (
        ELO_K_FACTOR
        * (
            actual_home
            - expected_home
        )
    )


    historical_end_elo[
        row.HomeTeam
    ] = (
        home_rating
        + change
    )

    historical_end_elo[
        row.AwayTeam
    ] = (
        away_rating
        - change
    )


# ============================================================
# F. BUILD TEAM-LEVEL CURRENT-SEASON HISTORY
# ============================================================

team_history_records = []


for row in live_results.itertuples(
    index=False
):

    if row.FTR == "H":

        home_points = 3
        away_points = 0

        home_win = 1
        away_win = 0

    elif row.FTR == "D":

        home_points = 1
        away_points = 1

        home_win = 0
        away_win = 0

    else:

        home_points = 0
        away_points = 3

        home_win = 0
        away_win = 1


    team_history_records.extend(
        [
            {
                "FixtureNumber": (
                    row.FixtureNumber
                ),
                "ActualKickoff": (
                    row.ActualKickoff
                ),
                "Team": (
                    row.HomeTeam
                ),
                "Opponent": (
                    row.AwayTeam
                ),
                "Venue": "Home",
                "GoalsFor": (
                    row.FTHG
                ),
                "GoalsAgainst": (
                    row.FTAG
                ),
                "Points": (
                    home_points
                ),
                "Win": (
                    home_win
                ),
            },
            {
                "FixtureNumber": (
                    row.FixtureNumber
                ),
                "ActualKickoff": (
                    row.ActualKickoff
                ),
                "Team": (
                    row.AwayTeam
                ),
                "Opponent": (
                    row.HomeTeam
                ),
                "Venue": "Away",
                "GoalsFor": (
                    row.FTAG
                ),
                "GoalsAgainst": (
                    row.FTHG
                ),
                "Points": (
                    away_points
                ),
                "Win": (
                    away_win
                ),
            },
        ]
    )


current_team_history = pd.DataFrame(
    team_history_records,
    columns=[
        "FixtureNumber",
        "ActualKickoff",
        "Team",
        "Opponent",
        "Venue",
        "GoalsFor",
        "GoalsAgainst",
        "Points",
        "Win",
    ],
)


if not current_team_history.empty:

    current_team_history = (
        current_team_history
        .sort_values(
            by=[
                "Team",
                "ActualKickoff",
                "FixtureNumber",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


# ============================================================
# G. IDENTIFY CURRENT FORECASTING FRONTIER
# ============================================================

completed_fixture_numbers = set(
    live_results[
        "FixtureNumber"
    ].tolist()
)


pending_schedule = (
    live_schedule.loc[
        ~live_schedule[
            "FixtureNumber"
        ].isin(
            completed_fixture_numbers
        )
    ]
    .copy()
    .sort_values(
        by=[
            "Kickoff",
            "FixtureNumber",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


next_fixture_by_team = {}


for row in pending_schedule.itertuples():

    if row.HomeTeam not in next_fixture_by_team:

        next_fixture_by_team[
            row.HomeTeam
        ] = row.FixtureNumber


    if row.AwayTeam not in next_fixture_by_team:

        next_fixture_by_team[
            row.AwayTeam
        ] = row.FixtureNumber


def next_for_both_clubs(
    row,
) -> bool:

    return (
        row.FixtureNumber
        == next_fixture_by_team.get(
            row.HomeTeam
        )
        == next_fixture_by_team.get(
            row.AwayTeam
        )
    )


if pending_schedule.empty:

    live_forecast_queue = (
        pending_schedule.copy()
    )

else:

    pending_schedule[
        "NextForBothClubs"
    ] = pending_schedule.apply(
        next_for_both_clubs,
        axis=1,
    )


    live_forecast_queue = (
        pending_schedule.loc[
            pending_schedule[
                "NextForBothClubs"
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )


# ============================================================
# H. FEATURE-ENGINEERING HELPERS
# ============================================================

def safe_difference(
    home_value,
    away_value,
):

    if (
        pd.isna(home_value)
        or pd.isna(away_value)
    ):
        return np.nan

    return float(
        home_value
        - away_value
    )


def team_history_before(
    team: str,
    kickoff: pd.Timestamp,
) -> pd.DataFrame:

    if current_team_history.empty:

        return current_team_history.copy()


    return (
        current_team_history.loc[
            (
                current_team_history[
                    "Team"
                ]
                == team
            )
            &
            (
                current_team_history[
                    "ActualKickoff"
                ]
                < kickoff
            )
        ]
        .sort_values(
            by=[
                "ActualKickoff",
                "FixtureNumber",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


def rolling_form(
    history: pd.DataFrame,
):

    # Exact historical contract:
    # five completed prior matches are required.

    if len(history) < 5:

        return {
            "Points": np.nan,
            "GoalsFor": np.nan,
            "GoalsAgainst": np.nan,
            "GoalDifference": np.nan,
            "WinRate": np.nan,
        }


    recent = history.tail(5)


    goals_for = float(
        recent[
            "GoalsFor"
        ].sum()
    )

    goals_against = float(
        recent[
            "GoalsAgainst"
        ].sum()
    )


    return {
        "Points": float(
            recent[
                "Points"
            ].sum()
        ),
        "GoalsFor": goals_for,
        "GoalsAgainst": goals_against,
        "GoalDifference": (
            goals_for
            - goals_against
        ),
        "WinRate": float(
            recent[
                "Win"
            ].mean()
        ),
    }


def venue_form(
    history: pd.DataFrame,
    venue: str,
):

    venue_history = (
        history.loc[
            history[
                "Venue"
            ]
            == venue
        ]
        .copy()
    )

    return rolling_form(
        venue_history
    )


def rest_days_before(
    history: pd.DataFrame,
    kickoff: pd.Timestamp,
):

    if history.empty:
        return np.nan


    previous_kickoff = (
        history[
            "ActualKickoff"
        ].iloc[-1]
    )


    return float(
        (
            kickoff.normalize()
            - previous_kickoff.normalize()
        ).days
    )


def congestion_count(
    history: pd.DataFrame,
    kickoff: pd.Timestamp,
    days: int,
):

    if history.empty:
        return 0.0


    lower_bound = (
        kickoff
        - pd.Timedelta(
            days=days
        )
    )


    return float(
        (
            (
                history[
                    "ActualKickoff"
                ]
                >= lower_bound
            )
            &
            (
                history[
                    "ActualKickoff"
                ]
                < kickoff
            )
        ).sum()
    )


# ============================================================
# I. ELO STATE BEFORE A TARGET FIXTURE
# ============================================================

def elo_state_before(
    kickoff: pd.Timestamp,
):

    ratings = dict(
        historical_end_elo
    )


    earlier_results = (
        live_results.loc[
            live_results[
                "ActualKickoff"
            ]
            < kickoff
        ]
        .sort_values(
            by=[
                "ActualKickoff",
                "FixtureNumber",
            ],
            kind="mergesort",
        )
    )


    for row in earlier_results.itertuples(
        index=False
    ):

        home_rating = ratings.get(
            row.HomeTeam,
            INITIAL_ELO,
        )

        away_rating = ratings.get(
            row.AwayTeam,
            INITIAL_ELO,
        )


        expected_home = (
            expected_home_elo_score(
                home_rating,
                away_rating,
            )
        )


        if row.FTR == "H":
            actual_home = 1.0

        elif row.FTR == "D":
            actual_home = 0.5

        else:
            actual_home = 0.0


        change = (
            ELO_K_FACTOR
            * (
                actual_home
                - expected_home
            )
        )


        ratings[
            row.HomeTeam
        ] = (
            home_rating
            + change
        )

        ratings[
            row.AwayTeam
        ] = (
            away_rating
            - change
        )


    return ratings


# ============================================================
# J. LEAGUE TABLE BEFORE A TARGET FIXTURE
# ============================================================

def league_table_before(
    kickoff: pd.Timestamp,
):

    table = pd.DataFrame(
        {
            "Team": season_teams,
            "Points": 0,
            "GoalsFor": 0,
            "GoalsAgainst": 0,
            "GoalDifference": 0,
            "MatchesPlayed": 0,
        }
    ).set_index(
        "Team"
    )


    earlier_results = (
        live_results.loc[
            live_results[
                "ActualKickoff"
            ]
            < kickoff
        ]
        .sort_values(
            by=[
                "ActualKickoff",
                "FixtureNumber",
            ],
            kind="mergesort",
        )
    )


    for row in earlier_results.itertuples(
        index=False
    ):

        home_team = row.HomeTeam
        away_team = row.AwayTeam


        table.loc[
            home_team,
            "GoalsFor",
        ] += row.FTHG

        table.loc[
            home_team,
            "GoalsAgainst",
        ] += row.FTAG


        table.loc[
            away_team,
            "GoalsFor",
        ] += row.FTAG

        table.loc[
            away_team,
            "GoalsAgainst",
        ] += row.FTHG


        table.loc[
            home_team,
            "MatchesPlayed",
        ] += 1

        table.loc[
            away_team,
            "MatchesPlayed",
        ] += 1


        if row.FTR == "H":

            table.loc[
                home_team,
                "Points",
            ] += 3

        elif row.FTR == "A":

            table.loc[
                away_team,
                "Points",
            ] += 3

        else:

            table.loc[
                home_team,
                "Points",
            ] += 1

            table.loc[
                away_team,
                "Points",
            ] += 1


    table[
        "GoalDifference"
    ] = (
        table[
            "GoalsFor"
        ]
        -
        table[
            "GoalsAgainst"
        ]
    )


    ranked = (
        table
        .reset_index()
        .sort_values(
            by=[
                "Points",
                "GoalDifference",
                "GoalsFor",
                "Team",
            ],
            ascending=[
                False,
                False,
                False,
                True,
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )


    # Match the historical convention:
    # position is undefined before any completed fixture.
    if earlier_results.empty:

        ranked[
            "LeaguePosition"
        ] = np.nan

    else:

        ranked[
            "LeaguePosition"
        ] = np.arange(
            1,
            len(ranked) + 1,
        )


    return ranked.set_index(
        "Team"
    )


# ============================================================
# K. POSITION-CATEGORY HELPER
# ============================================================

def position_categories(
    position,
):

    if pd.isna(position):

        return {
            "Top4": np.nan,
            "Top6": np.nan,
            "TopHalf": np.nan,
            "Bottom3": np.nan,
        }


    position = int(
        position
    )


    return {
        "Top4": float(
            position <= 4
        ),
        "Top6": float(
            position <= 6
        ),
        "TopHalf": float(
            position <= 10
        ),
        "Bottom3": float(
            position >= 18
        ),
    }


# ============================================================
# L. CONSTRUCT ONE COMPLETE 70-FEATURE ROW
# ============================================================

def construct_live_fixture_features(
    fixture,
):

    kickoff = pd.Timestamp(
        fixture.Kickoff
    )

    home_team = (
        fixture.HomeTeam
    )

    away_team = (
        fixture.AwayTeam
    )


    # --------------------------------------------------------
    # Team histories before target kickoff
    # --------------------------------------------------------

    home_history = team_history_before(
        home_team,
        kickoff,
    )

    away_history = team_history_before(
        away_team,
        kickoff,
    )


    # --------------------------------------------------------
    # Elo
    # --------------------------------------------------------

    ratings = elo_state_before(
        kickoff
    )


    home_elo = float(
        ratings.get(
            home_team,
            INITIAL_ELO,
        )
    )

    away_elo = float(
        ratings.get(
            away_team,
            INITIAL_ELO,
        )
    )


    # --------------------------------------------------------
    # General five-match form
    # --------------------------------------------------------

    home_form = rolling_form(
        home_history
    )

    away_form = rolling_form(
        away_history
    )


    # --------------------------------------------------------
    # Venue-specific five-match form
    # --------------------------------------------------------

    home_venue_form = venue_form(
        home_history,
        "Home",
    )

    away_venue_form = venue_form(
        away_history,
        "Away",
    )


    # --------------------------------------------------------
    # Rest / congestion
    # --------------------------------------------------------

    home_rest = rest_days_before(
        home_history,
        kickoff,
    )

    away_rest = rest_days_before(
        away_history,
        kickoff,
    )


    home_short_rest = (
        np.nan
        if pd.isna(
            home_rest
        )
        else float(
            home_rest <= 3
        )
    )

    away_short_rest = (
        np.nan
        if pd.isna(
            away_rest
        )
        else float(
            away_rest <= 3
        )
    )


    home_last_7 = congestion_count(
        home_history,
        kickoff,
        7,
    )

    away_last_7 = congestion_count(
        away_history,
        kickoff,
        7,
    )

    home_last_14 = congestion_count(
        home_history,
        kickoff,
        14,
    )

    away_last_14 = congestion_count(
        away_history,
        kickoff,
        14,
    )


    # --------------------------------------------------------
    # Team-specific match number / season progress
    # --------------------------------------------------------

    home_match_week = float(
        len(
            home_history
        )
        + 1
    )

    away_match_week = float(
        len(
            away_history
        )
        + 1
    )


    home_progress = (
        home_match_week
        - 1.0
    ) / 37.0

    away_progress = (
        away_match_week
        - 1.0
    ) / 37.0


    average_progress = (
        home_progress
        + away_progress
    ) / 2.0


    # --------------------------------------------------------
    # Pre-match league table
    # --------------------------------------------------------

    league_table = league_table_before(
        kickoff
    )


    home_points = float(
        league_table.loc[
            home_team,
            "Points",
        ]
    )

    away_points = float(
        league_table.loc[
            away_team,
            "Points",
        ]
    )


    home_goal_difference = float(
        league_table.loc[
            home_team,
            "GoalDifference",
        ]
    )

    away_goal_difference = float(
        league_table.loc[
            away_team,
            "GoalDifference",
        ]
    )


    home_position = (
        league_table.loc[
            home_team,
            "LeaguePosition",
        ]
    )

    away_position = (
        league_table.loc[
            away_team,
            "LeaguePosition",
        ]
    )


    home_categories = (
        position_categories(
            home_position
        )
    )

    away_categories = (
        position_categories(
            away_position
        )
    )


    # --------------------------------------------------------
    # Complete frozen feature dictionary
    # --------------------------------------------------------

    features = {
        # Elo
        "HomeEloBefore": (
            home_elo
        ),
        "AwayEloBefore": (
            away_elo
        ),

        # General form
        "HomeRollingPoints5": (
            home_form[
                "Points"
            ]
        ),
        "AwayRollingPoints5": (
            away_form[
                "Points"
            ]
        ),
        "HomeRollingGoalsFor5": (
            home_form[
                "GoalsFor"
            ]
        ),
        "AwayRollingGoalsFor5": (
            away_form[
                "GoalsFor"
            ]
        ),
        "HomeRollingGoalsAgainst5": (
            home_form[
                "GoalsAgainst"
            ]
        ),
        "AwayRollingGoalsAgainst5": (
            away_form[
                "GoalsAgainst"
            ]
        ),
        "HomeRollingGoalDifference5": (
            home_form[
                "GoalDifference"
            ]
        ),
        "AwayRollingGoalDifference5": (
            away_form[
                "GoalDifference"
            ]
        ),
        "HomeRollingWinRate5": (
            home_form[
                "WinRate"
            ]
        ),
        "AwayRollingWinRate5": (
            away_form[
                "WinRate"
            ]
        ),

        # Venue form
        "HomeVenueRollingPoints5": (
            home_venue_form[
                "Points"
            ]
        ),
        "AwayVenueRollingPoints5": (
            away_venue_form[
                "Points"
            ]
        ),
        "HomeVenueRollingGoalsFor5": (
            home_venue_form[
                "GoalsFor"
            ]
        ),
        "AwayVenueRollingGoalsFor5": (
            away_venue_form[
                "GoalsFor"
            ]
        ),
        "HomeVenueRollingGoalsAgainst5": (
            home_venue_form[
                "GoalsAgainst"
            ]
        ),
        "AwayVenueRollingGoalsAgainst5": (
            away_venue_form[
                "GoalsAgainst"
            ]
        ),
        "HomeVenueRollingGoalDifference5": (
            home_venue_form[
                "GoalDifference"
            ]
        ),
        "AwayVenueRollingGoalDifference5": (
            away_venue_form[
                "GoalDifference"
            ]
        ),
        "HomeVenueRollingWinRate5": (
            home_venue_form[
                "WinRate"
            ]
        ),
        "AwayVenueRollingWinRate5": (
            away_venue_form[
                "WinRate"
            ]
        ),

        # Rest
        "HomeRestDays": (
            home_rest
        ),
        "AwayRestDays": (
            away_rest
        ),
        "HomeShortRest3": (
            home_short_rest
        ),
        "AwayShortRest3": (
            away_short_rest
        ),

        # Congestion
        "HomeMatchesLast7Days": (
            home_last_7
        ),
        "AwayMatchesLast7Days": (
            away_last_7
        ),
        "HomeMatchesLast14Days": (
            home_last_14
        ),
        "AwayMatchesLast14Days": (
            away_last_14
        ),

        # Elo difference
        "EloDifference": (
            home_elo
            - away_elo
        ),

        # General-form differences
        "PointsFormDifference5": (
            safe_difference(
                home_form[
                    "Points"
                ],
                away_form[
                    "Points"
                ],
            )
        ),
        "GoalsForFormDifference5": (
            safe_difference(
                home_form[
                    "GoalsFor"
                ],
                away_form[
                    "GoalsFor"
                ],
            )
        ),
        "GoalsAgainstFormDifference5": (
            safe_difference(
                home_form[
                    "GoalsAgainst"
                ],
                away_form[
                    "GoalsAgainst"
                ],
            )
        ),
        "GoalDifferenceFormDifference5": (
            safe_difference(
                home_form[
                    "GoalDifference"
                ],
                away_form[
                    "GoalDifference"
                ],
            )
        ),
        "WinRateFormDifference5": (
            safe_difference(
                home_form[
                    "WinRate"
                ],
                away_form[
                    "WinRate"
                ],
            )
        ),

        # Venue-form differences
        "VenuePointsFormDifference5": (
            safe_difference(
                home_venue_form[
                    "Points"
                ],
                away_venue_form[
                    "Points"
                ],
            )
        ),
        "VenueGoalsForFormDifference5": (
            safe_difference(
                home_venue_form[
                    "GoalsFor"
                ],
                away_venue_form[
                    "GoalsFor"
                ],
            )
        ),
        "VenueGoalsAgainstFormDifference5": (
            safe_difference(
                home_venue_form[
                    "GoalsAgainst"
                ],
                away_venue_form[
                    "GoalsAgainst"
                ],
            )
        ),
        "VenueGoalDifferenceFormDifference5": (
            safe_difference(
                home_venue_form[
                    "GoalDifference"
                ],
                away_venue_form[
                    "GoalDifference"
                ],
            )
        ),
        "VenueWinRateFormDifference5": (
            safe_difference(
                home_venue_form[
                    "WinRate"
                ],
                away_venue_form[
                    "WinRate"
                ],
            )
        ),

        # Schedule differences
        "RestDaysDifference": (
            safe_difference(
                home_rest,
                away_rest,
            )
        ),
        "ShortRestDifference3": (
            safe_difference(
                home_short_rest,
                away_short_rest,
            )
        ),
        "MatchesLast7DaysDifference": (
            home_last_7
            - away_last_7
        ),
        "MatchesLast14DaysDifference": (
            home_last_14
            - away_last_14
        ),

        # Match number / season progress
        "HomeMatchWeek": (
            home_match_week
        ),
        "AwayMatchWeek": (
            away_match_week
        ),
        "HomeSeasonProgress": (
            home_progress
        ),
        "AwaySeasonProgress": (
            away_progress
        ),
        "AverageSeasonProgress": (
            average_progress
        ),
        "SeasonProgressDifference": (
            home_progress
            - away_progress
        ),
        "MatchesPlayedDifference": (
            home_match_week
            - away_match_week
        ),
        "SecondHalfSeason": float(
            average_progress
            >= 0.5
        ),

        # League table
        "HomePointsBefore": (
            home_points
        ),
        "AwayPointsBefore": (
            away_points
        ),
        "HomeLeaguePosition": (
            float(
                home_position
            )
            if pd.notna(
                home_position
            )
            else np.nan
        ),
        "AwayLeaguePosition": (
            float(
                away_position
            )
            if pd.notna(
                away_position
            )
            else np.nan
        ),
        "HomeGoalDifferenceBefore": (
            home_goal_difference
        ),
        "AwayGoalDifferenceBefore": (
            away_goal_difference
        ),
        "PointsDifference": (
            home_points
            - away_points
        ),

        # Positive = home team has better position.
        "PositionDifference": (
            safe_difference(
                away_position,
                home_position,
            )
        ),

        "GoalDifferenceDifference": (
            home_goal_difference
            - away_goal_difference
        ),

        # League-position categories
        "HomeTop4Before": (
            home_categories[
                "Top4"
            ]
        ),
        "HomeTop6Before": (
            home_categories[
                "Top6"
            ]
        ),
        "HomeTopHalfBefore": (
            home_categories[
                "TopHalf"
            ]
        ),
        "HomeBottom3Before": (
            home_categories[
                "Bottom3"
            ]
        ),
        "AwayTop4Before": (
            away_categories[
                "Top4"
            ]
        ),
        "AwayTop6Before": (
            away_categories[
                "Top6"
            ]
        ),
        "AwayTopHalfBefore": (
            away_categories[
                "TopHalf"
            ]
        ),
        "AwayBottom3Before": (
            away_categories[
                "Bottom3"
            ]
        ),
    }


    # --------------------------------------------------------
    # Exact frozen feature contract
    # --------------------------------------------------------

    missing_features = (
        set(
            production_feature_columns
        )
        - set(
            features
        )
    )

    unexpected_features = (
        set(
            features
        )
        - set(
            production_feature_columns
        )
    )


    assert not missing_features, (
        "Live constructor is missing frozen features: "
        f"{sorted(missing_features)}"
    )

    assert not unexpected_features, (
        "Live constructor created unexpected features: "
        f"{sorted(unexpected_features)}"
    )


    return {
        feature: features[
            feature
        ]
        for feature
        in production_feature_columns
    }


# ============================================================
# M. RECONSTRUCT CURRENT QUEUE FEATURES
# ============================================================

live_feature_records = []


for fixture in live_forecast_queue.itertuples(
    index=False
):

    feature_record = (
        construct_live_fixture_features(
            fixture
        )
    )


    feature_record = {
        "FixtureNumber": (
            fixture.FixtureNumber
        ),
        "Season": (
            fixture.Season
        ),
        "Kickoff": (
            fixture.Kickoff
        ),
        "OfficialHomeTeam": (
            fixture.OfficialHomeTeam
        ),
        "OfficialAwayTeam": (
            fixture.OfficialAwayTeam
        ),
        "HomeTeam": (
            fixture.HomeTeam
        ),
        "AwayTeam": (
            fixture.AwayTeam
        ),

        "ResultsAvailableBeforeFixture": int(
            (
                live_results[
                    "ActualKickoff"
                ]
                < fixture.Kickoff
            ).sum()
        )
        if not live_results.empty
        else 0,

        **feature_record,
    }


    live_feature_records.append(
        feature_record
    )


live_feature_table = pd.DataFrame(
    live_feature_records
)


if live_feature_table.empty:

    live_feature_table = pd.DataFrame(
        columns=(
            [
                "FixtureNumber",
                "Season",
                "Kickoff",
                "OfficialHomeTeam",
                "OfficialAwayTeam",
                "HomeTeam",
                "AwayTeam",
                "ResultsAvailableBeforeFixture",
            ]
            + production_feature_columns
        )
    )


# ============================================================
# N. FEATURE VALIDATION
# ============================================================

assert len(
    live_feature_table
) == len(
    live_forecast_queue
)


assert all(
    feature
    in live_feature_table.columns
    for feature
    in production_feature_columns
)


if not live_feature_table.empty:

    live_numeric_features = (
        live_feature_table[
            production_feature_columns
        ]
        .to_numpy(
            dtype=float
        )
    )


    assert not np.isinf(
        live_numeric_features
    ).any(), (
        "Live feature matrix contains infinite values."
    )


    assert (
        live_feature_table[
            "HomeMatchWeek"
        ]
        .between(
            1,
            38,
        )
        .all()
    )


    assert (
        live_feature_table[
            "AwayMatchWeek"
        ]
        .between(
            1,
            38,
        )
        .all()
    )


    assert (
        live_feature_table[
            "HomeSeasonProgress"
        ]
        .between(
            0,
            1,
        )
        .all()
    )


    assert (
        live_feature_table[
            "AwaySeasonProgress"
        ]
        .between(
            0,
            1,
        )
        .all()
    )


    assert np.allclose(
        live_feature_table[
            "EloDifference"
        ],
        (
            live_feature_table[
                "HomeEloBefore"
            ]
            -
            live_feature_table[
                "AwayEloBefore"
            ]
        ),
    )


    assert np.allclose(
        live_feature_table[
            "PointsDifference"
        ],
        (
            live_feature_table[
                "HomePointsBefore"
            ]
            -
            live_feature_table[
                "AwayPointsBefore"
            ]
        ),
    )


    assert np.allclose(
        live_feature_table[
            "GoalDifferenceDifference"
        ],
        (
            live_feature_table[
                "HomeGoalDifferenceBefore"
            ]
            -
            live_feature_table[
                "AwayGoalDifferenceBefore"
            ]
        ),
    )


# ============================================================
# O. APPLY THE FROZEN PRODUCTION MODEL
# ============================================================

if live_feature_table.empty:

    live_predictions = pd.DataFrame(
        columns=[
            "HomeExpectedGoals",
            "AwayExpectedGoals",
            "Probability_H",
            "Probability_D",
            "Probability_A",
            "ModalHomeGoals",
            "ModalAwayGoals",
            "ModalScoreProbability",
        ]
    )

else:

    live_predictions = (
        predict_with_frozen_production_model(
            live_feature_table
        )
        .reset_index(drop=True)
    )


# ============================================================
# P. BUILD CURRENT LIVE FORECAST TABLE
# ============================================================

forecast_generated_utc = datetime.now(
    timezone.utc
)


if live_feature_table.empty:

    live_forecasts = pd.DataFrame()

else:

    live_forecasts = pd.concat(
        [
            live_feature_table[
                [
                    "FixtureNumber",
                    "Season",
                    "Kickoff",
                    "OfficialHomeTeam",
                    "OfficialAwayTeam",
                    "HomeTeam",
                    "AwayTeam",
                    "ResultsAvailableBeforeFixture",
                ]
            ]
            .reset_index(drop=True),

            live_predictions,
        ],
        axis=1,
    )


    probability_matrix = (
        live_forecasts[
            [
                "Probability_H",
                "Probability_D",
                "Probability_A",
            ]
        ]
        .to_numpy()
    )


    outcome_codes = np.array(
        [
            "H",
            "D",
            "A",
        ]
    )


    favourite_indices = np.argmax(
        probability_matrix,
        axis=1,
    )


    live_forecasts[
        "MostLikelyOutcome"
    ] = outcome_codes[
        favourite_indices
    ]


    live_forecasts[
        "MostLikelyOutcomeProbability"
    ] = np.max(
        probability_matrix,
        axis=1,
    )


    def readable_live_favourite(
        row,
    ):

        if (
            row[
                "MostLikelyOutcome"
            ]
            == "H"
        ):

            return row[
                "OfficialHomeTeam"
            ]

        if (
            row[
                "MostLikelyOutcome"
            ]
            == "A"
        ):

            return row[
                "OfficialAwayTeam"
            ]

        return "Draw"


    live_forecasts[
        "ModelFavourite"
    ] = (
        live_forecasts.apply(
            readable_live_favourite,
            axis=1,
        )
    )


    live_forecasts[
        "ForecastGeneratedUTC"
    ] = (
        forecast_generated_utc
        .isoformat()
    )


# ============================================================
# Q. PRE-SEASON REGRESSION TEST
#
# With zero 2026-27 results, the live runner must reproduce
# the opening-round feature matrix and forecasts already
# validated earlier in this notebook.
# ============================================================

opening_feature_reproduction_error = np.nan
opening_prediction_reproduction_error = np.nan


if len(live_results) == 0:

    assert "opening_round_feature_table" in globals()
    assert "opening_round_forecasts" in globals()


    expected_opening_features = (
        opening_round_feature_table
        .sort_values(
            "FixtureNumber"
        )
        .reset_index(drop=True)
    )


    runner_opening_features = (
        live_feature_table
        .sort_values(
            "FixtureNumber"
        )
        .reset_index(drop=True)
    )


    assert (
        expected_opening_features[
            "FixtureNumber"
        ].tolist()
        ==
        runner_opening_features[
            "FixtureNumber"
        ].tolist()
    )


    expected_feature_array = (
        expected_opening_features[
            production_feature_columns
        ]
        .to_numpy(
            dtype=float
        )
    )


    runner_feature_array = (
        runner_opening_features[
            production_feature_columns
        ]
        .to_numpy(
            dtype=float
        )
    )


    feature_difference = np.abs(
        expected_feature_array
        -
        runner_feature_array
    )


    finite_feature_difference = (
        feature_difference[
            np.isfinite(
                feature_difference
            )
        ]
    )


    opening_feature_reproduction_error = (
        float(
            finite_feature_difference.max()
        )
        if finite_feature_difference.size
        else 0.0
    )


    assert np.allclose(
        expected_feature_array,
        runner_feature_array,
        atol=1e-10,
        rtol=0.0,
        equal_nan=True,
    ), (
        "Live runner does not reproduce the validated "
        "opening-round feature state."
    )


    expected_predictions = (
        opening_round_forecasts
        .sort_values(
            "FixtureNumber"
        )
        .reset_index(drop=True)
    )


    runner_predictions = (
        live_forecasts
        .sort_values(
            "FixtureNumber"
        )
        .reset_index(drop=True)
    )


    prediction_columns = [
        "HomeExpectedGoals",
        "AwayExpectedGoals",
        "Probability_H",
        "Probability_D",
        "Probability_A",
    ]


    prediction_difference = np.abs(
        expected_predictions[
            prediction_columns
        ].to_numpy()
        -
        runner_predictions[
            prediction_columns
        ].to_numpy()
    )


    opening_prediction_reproduction_error = float(
        prediction_difference.max()
    )


    assert (
        opening_prediction_reproduction_error
        < 1e-8
    ), (
        "Live runner does not reproduce the validated "
        "opening-round forecasts closely enough: "
        f"{opening_prediction_reproduction_error:.3e}"
    )


# ============================================================
# R. FINAL PROBABILITY VALIDATION
# ============================================================

if not live_forecasts.empty:

    live_probability_values = (
        live_forecasts[
            [
                "Probability_H",
                "Probability_D",
                "Probability_A",
            ]
        ]
        .to_numpy()
    )


    maximum_probability_sum_error = float(
        np.max(
            np.abs(
                live_probability_values.sum(
                    axis=1
                )
                - 1.0
            )
        )
    )


    assert (
        maximum_probability_sum_error
        < 1e-10
    )

    assert (
        live_probability_values >= 0
    ).all()

    assert (
        live_probability_values <= 1
    ).all()

else:

    maximum_probability_sum_error = 0.0


# ============================================================
# S. EXPORT CURRENT STATE + TIMESTAMPED SNAPSHOT
# ============================================================

live_forecast_queue.to_csv(
    current_queue_output_path,
    index=False,
)


live_feature_table.to_csv(
    current_feature_output_path,
    index=False,
)


live_forecasts.to_csv(
    current_forecast_output_path,
    index=False,
)


snapshot_timestamp = (
    forecast_generated_utc
    .strftime(
        "%Y%m%dT%H%M%SZ"
    )
)


snapshot_output_path = (
    forecast_directory
    / (
        "live_forecast_snapshot_"
        f"{snapshot_timestamp}.csv"
    )
)


live_forecasts.to_csv(
    snapshot_output_path,
    index=False,
)


# ============================================================
# T. VALIDATION REPORT
# ============================================================

live_runner_validation = pd.DataFrame(
    [
        {
            "Validation": (
                "Canonical fixtures"
            ),
            "Value": len(
                live_schedule
            ),
            "Expected": 380,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Completed fixtures"
            ),
            "Value": len(
                live_results
            ),
            "Expected": "0 to 380",
            "Status": "PASS",
        },
        {
            "Validation": (
                "Current queue features"
            ),
            "Value": len(
                live_feature_table
            ),
            "Expected": len(
                live_forecast_queue
            ),
            "Status": "PASS",
        },
        {
            "Validation": (
                "Frozen predictor count"
            ),
            "Value": len(
                production_feature_columns
            ),
            "Expected": 70,
            "Status": "PASS",
        },
        {
            "Validation": (
                "Probability sum max error"
            ),
            "Value": (
                maximum_probability_sum_error
            ),
            "Expected": "< 1e-10",
            "Status": "PASS",
        },
        {
            "Validation": (
                "Production model refitted"
            ),
            "Value": False,
            "Expected": False,
            "Status": "PASS",
        },
    ]
)


if len(live_results) == 0:

    live_runner_validation = pd.concat(
        [
            live_runner_validation,

            pd.DataFrame(
                [
                    {
                        "Validation": (
                            "Opening feature reproduction error"
                        ),
                        "Value": (
                            opening_feature_reproduction_error
                        ),
                        "Expected": "< 1e-10",
                        "Status": "PASS",
                    },
                    {
                        "Validation": (
                            "Opening prediction reproduction error"
                        ),
                        "Value": (
                            opening_prediction_reproduction_error
                        ),
                        "Expected": "< 1e-8",
                        "Status": "PASS",
                    },
                ]
            ),
        ],
        ignore_index=True,
    )


live_runner_profile = pd.DataFrame(
    [
        {
            "Metric": (
                "Completed 2026-27 fixtures"
            ),
            "Value": len(
                live_results
            ),
        },
        {
            "Metric": (
                "Remaining fixtures"
            ),
            "Value": (
                380
                - len(
                    live_results
                )
            ),
        },
        {
            "Metric": (
                "Current queue fixtures"
            ),
            "Value": len(
                live_forecast_queue
            ),
        },
        {
            "Metric": (
                "Feature rows reconstructed"
            ),
            "Value": len(
                live_feature_table
            ),
        },
        {
            "Metric": (
                "Forecasts generated"
            ),
            "Value": len(
                live_forecasts
            ),
        },
        {
            "Metric": (
                "Frozen predictors"
            ),
            "Value": 70,
        },
        {
            "Metric": (
                "In-season model refit"
            ),
            "Value": False,
        },
    ]
)


# ------------------------------------------------------------
# Compact display
# ------------------------------------------------------------

if not live_forecasts.empty:

    live_forecast_display = (
        live_forecasts[
            [
                "Kickoff",
                "OfficialHomeTeam",
                "OfficialAwayTeam",
                "HomeExpectedGoals",
                "AwayExpectedGoals",
                "Probability_H",
                "Probability_D",
                "Probability_A",
                "ModelFavourite",
                "ResultsAvailableBeforeFixture",
            ]
        ]
        .copy()
    )


    live_forecast_display[
        "HomeExpectedGoals"
    ] = (
        live_forecast_display[
            "HomeExpectedGoals"
        ]
        .round(3)
    )


    live_forecast_display[
        "AwayExpectedGoals"
    ] = (
        live_forecast_display[
            "AwayExpectedGoals"
        ]
        .round(3)
    )


    for column in [
        "Probability_H",
        "Probability_D",
        "Probability_A",
    ]:

        live_forecast_display[
            column
        ] = (
            live_forecast_display[
                column
            ]
            .round(4)
        )

else:

    live_forecast_display = pd.DataFrame()


live_season_runner_ready = True


# ============================================================
# U. OUTPUT
# ============================================================

print(
    "Live-season runner profile:"
)
display(
    live_runner_profile
)

print(
    "\nLive-season runner validation:"
)
display(
    live_runner_validation
)

print(
    "\nCurrent live forecasts:"
)
display(
    live_forecast_display
)

print(
    "\nCurrent forecast file:",
    current_forecast_output_path
    .relative_to(
        project_root
    )
    .as_posix(),
)

print(
    "Timestamped snapshot:",
    snapshot_output_path
    .relative_to(
        project_root
    )
    .as_posix(),
)

print(
    "\nLive-season forecast runner status: VALID"
)

print(
    "Reusable live-season runner ready:",
    live_season_runner_ready,
)

Live-season runner profile:


,Metric,Value
0,Completed 2026-27 fixtures,0
1,Remaining fixtures,380
2,Current queue fixtures,10
3,Feature rows reconstructed,10
4,Forecasts generated,10
5,Frozen predictors,70
6,In-season model refit,False



Live-season runner validation:


,Validation,Value,Expected,Status
0,Canonical fixtures,380,380,PASS
1,Completed fixtures,0,0 to 380,PASS
2,Current queue features,10,10,PASS
3,Frozen predictor count,70,70,PASS
4,Probability sum max error,0.0,< 1e-10,PASS
5,Production model refitted,False,False,PASS
6,Opening feature reproduction error,0.0,< 1e-10,PASS
7,Opening prediction reproduction error,0.0,< 1e-8,PASS



Current live forecasts:


,Kickoff,OfficialHomeTeam,OfficialAwayTeam,HomeExpectedGoals,AwayExpectedGoals,Probability_H,Probability_D,Probability_A,ModelFavourite,ResultsAvailableBeforeFixture
0,2026-08-21 20:00:00,Arsenal,Coventry City,2.470,0.951,0.7061,0.1682,0.1257,Arsenal,0
1,2026-08-22 12:30:00,Hull City,Manchester United,1.010,2.045,0.1794,0.2079,0.6127,Manchester United,0
2,2026-08-22 15:00:00,Everton,Crystal Palace,1.489,1.413,0.3932,0.2475,0.3593,Everton,0
3,2026-08-22 15:00:00,Ipswich Town,Sunderland,1.171,1.619,0.2760,0.2461,0.4779,Sunderland,0
4,2026-08-22 15:00:00,Nottingham Forest,Leeds United,1.545,1.353,0.4196,0.2466,0.3338,Nottingham Forest,0
5,2026-08-22 17:30:00,Brentford,Tottenham Hotspur,1.691,1.236,0.4806,0.2395,0.2798,Brentford,0
6,2026-08-23 14:00:00,Brighton & Hove Albion,Aston Villa,1.418,1.596,0.3404,0.2413,0.4183,Aston Villa,0
7,2026-08-23 14:00:00,Manchester City,AFC Bournemouth,2.019,1.223,0.5578,0.2143,0.2278,Manchester City,0
8,2026-08-23 16:30:00,Newcastle United,Liverpool,1.342,1.686,0.3061,0.2379,0.4560,Liverpool,0
9,2026-08-24 20:00:00,Fulham,Chelsea,1.441,1.500,0.3640,0.2456,0.3903,Chelsea,0



Current forecast file: outputs/forecasts/2026_27/current_live_forecasts.csv
Timestamped snapshot: outputs/forecasts/2026_27/live_forecast_snapshot_20260809T183159Z.csv

Live-season forecast runner status: VALID
Reusable live-season runner ready: True


### Results and Interpretation

The live-season forecasting runner was successfully constructed and validated.

At the current pre-season state, no 2026–27 Premier League fixtures have been completed, leaving all 380 fixtures outstanding. The runner correctly identified the ten opening-round fixtures as the current forecasting frontier, reconstructed one complete feature row for each fixture and generated ten forecasts using the frozen production model.

The production specification remained unchanged throughout the process. All 70 frozen predictors were reconstructed and the production model was not refitted, preserving the modelling decisions established during model selection and production refitting.

All live-runner validation checks passed. In particular:

- the canonical fixture schedule contained exactly 380 fixtures;
- the completed-results ledger contained zero fixtures, as expected before the season begins;
- ten forecastable fixtures produced ten corresponding feature rows;
- the frozen predictor count remained exactly 70;
- predicted home, draw and away probabilities summed to one within numerical precision;
- no in-season model refitting occurred.

The strongest validation is the pre-season regression test. The independently reconstructed live feature matrix reproduced the previously validated opening-round feature matrix with a maximum error of **0.0**, while the resulting production forecasts also reproduced the existing opening-round predictions with a maximum error of **0.0**.

This demonstrates that the live-season runner is consistent with the feature-engineering and production pipelines already validated earlier in the project. The opening forecasts are therefore not a separate manually constructed output: they can be regenerated from the same sequential process that will be used after real 2026–27 results become available.

The runner is now operational for the season. As fixtures are completed, new results can be added to the current-season results file and the section rerun. The system will then reconstruct the latest pre-match Elo, rolling form, venue form, rest, congestion, season-progress and league-table state before identifying and forecasting the next eligible fixtures.

The statistical model itself remains frozen. Only information that becomes available through completed matches is allowed to change, maintaining the chronological and leakage-control principles used throughout the project.

## 12. Final Conclusions and Operational Notes

This notebook completes the transition from historical model development to genuine 2026–27 Premier League forecasting.

The production Independent Poisson model selected through the earlier chronological evaluation was loaded from its frozen artefacts and applied without further model selection, calibration or refitting. The official 2026–27 fixture schedule was validated, the opening pre-match state was reconstructed using the established 70-feature schema, and the first genuinely unseen forecasts of the project were generated.

### Opening-Round Forecasts

Ten opening-round fixtures were forecast before any 2026–27 league results were available.

The strongest model forecasts were:

- Arsenal to beat Coventry City, with approximately **70.6%** home-win probability;
- Manchester United to beat Hull City away, with approximately **61.3%** away-win probability.

Using the descriptive confidence framework established in this notebook:

- **2** fixtures were classified as `Strong`;
- **4** as `Moderate`;
- **4** as `Low`.

These classifications depend on both the probability assigned to the most likely outcome and its separation from the second-most likely outcome. They are descriptive reporting categories rather than statistically estimated confidence intervals.

### Market Comparison

The production forecasts were compared with an early 1X2 market snapshot after removing the quoted-market overround through proportional normalisation.

The model and market agreed on the favourite in **9 of the 10 fixtures**, although several underlying probability distributions differed materially.

The only opening fixture where the model and market selected different favourites was Brighton & Hove Albion versus Aston Villa, where the production model favoured Aston Villa while the market favoured Brighton.

Seven individual outcomes exceeded the descriptive threshold of a five percentage-point positive model–market disagreement. Six also received at least 20% probability from the production model and therefore formed the primary disagreement watchlist:

- Sunderland away at Ipswich;
- Aston Villa away at Brighton;
- Bournemouth away at Manchester City;
- Crystal Palace away at Everton;
- Brentford at home to Tottenham;
- Fulham at home to Chelsea.

Coventry away at Arsenal was separated as `Longshot-sensitive` because its model probability remained relatively low despite a substantial difference from the market.

These observations represent differences in probabilistic opinion rather than demonstrated betting opportunities. The earlier historical analysis found no persistent evidence that the model could reliably outperform closing market prices, and the opening-round prices used here are an early market snapshot rather than closing odds.

### Live-Season Forecasting

A sequential live-season forecasting runner has also been implemented.

The runner maintains the production model as a fixed statistical object while allowing the information available to it to evolve as real matches are completed.

For each new forecasting cycle, the system reconstructs:

- pre-match Elo ratings;
- five-match rolling form;
- venue-specific rolling form;
- rest and short-rest indicators;
- seven- and fourteen-day congestion measures;
- team-specific match number and season progress;
- pre-match league points, goal difference and position;
- league-position category indicators;
- all frozen home–away difference features.

Only completed fixtures occurring before the target match may affect these variables.

At the pre-season starting point, the live runner identified the same ten opening fixtures, reconstructed all ten 70-feature rows and regenerated the existing opening-round production forecasts.

The maximum feature reproduction error was **0.0** and the maximum prediction reproduction error was **0.0**. This confirms that the live-season implementation is consistent with the independently validated opening-round pipeline.

### Operational Workflow During 2026–27

Once the season begins, the production workflow is:

1. append newly completed Premier League results to `data/raw/premier_league_results_2026_27.csv`;
2. refresh the canonical fixture schedule when postponements, kickoff changes or rearrangements occur;
3. rerun the live-season update section;
4. allow the pipeline to reconstruct the current 70-feature pre-match state;
5. generate forecasts for the new forecasting frontier using the frozen production model;
6. optionally obtain a fresh bookmaker-price snapshot and repeat the market-comparison analysis;
7. retain timestamped forecast outputs so predictions can later be evaluated against the results that actually occurred.

The fitted imputer, scaler and Poisson estimators should **not** be retrained during this process. Changes in forecasts should arise from new football information rather than changes to the statistical model.

### Final Project Position

The project now contains a complete end-to-end football probability forecasting workflow:

**historical data → leakage-controlled feature engineering → competing statistical models → chronological backtesting → calibration analysis → bookmaker benchmarking → model selection → production refit → reproducible model packaging → genuine unseen forecasts → live sequential updating.**

The resulting system produces expected goals and full-time `(H, D, A)` probability distributions rather than deterministic match predictions.

The Independent Poisson model was selected because it offered the strongest overall balance of historical predictive performance, stability, interpretability and production suitability among the models evaluated. Its historical advantage over the alternative ensemble was small and should not be interpreted as decisive evidence of statistical superiority.

Similarly, the project does not claim to have discovered a persistent profitable betting strategy. Historical closing-market probabilities remained a stronger benchmark overall.

The principal achievement is therefore the construction of a reproducible, chronologically valid and operational probability forecasting engine that can now be tested prospectively throughout the 2026–27 Premier League season.